Copyright (C) 2026 Giulio Barni, Eric Madge  
This file is part of `HydróDex`.

`HydróDex` is free software: you can redistribute it and/or modify it under the
terms of the GNU General Public License as published by the Free Software
Foundation, either version 3 of the License, or (at your option) any later
version.

`HydróDex` is distributed in the hope that it will be useful, but WITHOUT ANY
WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A
PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with
`HydróDex`. If not, see <https://www.gnu.org/licenses/>. 

# `HydróDex` paper figures

This notebook reproduces the figures of two companion papers:

- **Long paper**: Giulio Barni and Eric Madge,
  *"`HydróDex`: gotta match 'em all!"*,
  [arXiv:2611.XXXXX [hep-ph]](https://arxiv.org/abs/2611.XXXXX)
- **Letter**: Giulio Barni and Eric Madge,
  *"Microscopic Realizability of Non-unique Bubble Hydrodynamics"*,
  [arXiv:2611.YYYYY [hep-ph]](https://arxiv.org/abs/2611.YYYYY)

Please remember to cite the papers if you use this notebook or the `HydróDex` code.

Run the notebook from top to bottom, from the folder that contains it: later cells
use results (scans, fits, helper functions) of earlier ones. Figures 1–5 of the long
paper are fast to reproduce. The other figures rely on dense scans over the
hydrodynamic parameter space (and, for Figures 13 and 14, on the computation of
gravitational-wave spectra), which take a few hours in total. Their results are saved
in the folder `data` and reused in later runs (see
[Saving and reusing scans](#Saving-and-reusing-scans)). The figures are saved as PDF
files in the folder `figures`.

| Paper | Figure | Section of the paper | Output file (`figures/`) |
|:--|:--|:--|:--|
| long | 1 | 2.4 | `solutions.pdf` |
| long | 2 | 3.2 | `matching.pdf` |
| long | 3 | 3.2 | `v_plus_v_minus.pdf` |
| long | 4 | 4.1 | `xi_w_alpha_T+.pdf` |
| long | 5 | 4.1 | `xi_w_alpha_TN.pdf` |
| long | 6 | 4.2 | `representative_profiles.pdf`, `representative_profiles_entropy_enthalpy.pdf` |
| letter | 1 | — | `representative_overlay_phase_plane.pdf` |
| long | 7 | 4.3 | `solutions_evolution.pdf` |
| long | 8 | 5.1 | `alphaN_alpha_plus_folding_with_faded_cm_overlays_2x2.pdf` |
| long | 9 | 5.2 | `phase_plane_all_branches_2panels.pdf` |
| long | 10 | 5.3 | `phase_plane_model_classes_panel.pdf` |
| letter | 2 | — | `model_classes_phase_plane_noninjective_map.pdf` |
| long | 11 | 5.4 | `LTE_splitting.pdf` |
| long | 12 | 6 | `efficiency_factor.pdf` |
| long | 13 | 7.2 | `unequal_css_peak_tracking_panel.pdf` |
| long | 14 | 7.2 | `combined_mixed_hybrid_diagnostics.pdf` |
| long | 15 | App. A.1 | `exotic_profiles_panel.pdf` |
| long | 16 | App. B.1 | `strong_monotonicity_test.pdf` |
| long | 17 | App. B.6 | `zero_crossing_bound_maximum.pdf` |

The two figures of the letter are built from the same scans as Figures 6 and 10 of
the long paper and are therefore placed right after them.

In [ ]:
import pickle
import re
import warnings
from contextlib import contextmanager
from pathlib import Path

import numpy as np
from scipy.integrate import solve_ivp
from scipy.interpolate import CubicSpline
from scipy.optimize import brentq, least_squares, minimize_scalar

import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import ListedColormap, LogNorm, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse, FancyArrowPatch, Patch, PathPatch, Wedge
from matplotlib.path import Path as MplPath
import matplotlib.patheffects as pe
from mpl_toolkits.axes_grid1.inset_locator import mark_inset

# shapely is used for the overlap regions of Figures 9 and 10 and of Figure 2 of the
# letter (pip install shapely)
import shapely
from shapely.geometry import GeometryCollection, MultiPolygon, Polygon, box
from shapely.ops import unary_union

import hydrodex
from hydrodex import profile_calculator as pc
from hydrodex import sound_shell_model as ssm

ST = pc.HydroSolutionType

## Settings

### HydróDex settings

Since we are scanning over initial conditions of the fluid profiles, including
initial fluid velocities close to zero, we decrease the profile integration step
size to $10^{-5}$. Furthermore, we silence the warnings of the consistency checks
of the profile integration and of the non-injectivity check: the former are triggered
by deflagration or hybrid profiles whose shock front lies close to
$(\xi,v)=(c_\pm, 0)$, i.e. by very weak shocks, the latter by the sound speeds with
$c_+>c_-$ studied on purpose in Figures 8–11. Both are expected in the scans below.

The projection $\alpha_+\mapsto\alpha_N$ of a given branch (`pc.compute_alpha_N`) is
evaluated with the step size `ALPHA_N_STEP` $=10^{-4}$, the `HydróDex` default;
the figures that need finer profiles pass their own step size explicitly.

In [ ]:
pc.set_profile_step_size(1e-5)
hydrodex.set_consistency_check(hydrodex.CheckMode.SILENT)
hydrodex.set_noninjectivity_check(hydrodex.CheckMode.SILENT)

ALPHA_N_STEP = 1e-4

### Matplotlib settings

The figures do not change the global matplotlib settings. Each figure cell is
wrapped in `with figure_style(...)`, which applies the style of that figure only
inside the `with` block:

- `"paper"`: the style `PAPER_RC`, used for Figures 1–5. Text is rendered with LaTeX
  (requires a working LaTeX installation with the amsmath and amssymb packages),
  so that fonts and symbols match the paper.
- `"profile"`: matplotlib defaults with larger LaTeX fonts, grids and white
  backgrounds (`PROFILE_RC`), used for all the other figures, possibly with
  figure-specific font sizes on top (e.g. `LARGE_FONT_RC`). These figures are scaled
  when included in the papers.

The figures were produced with matplotlib 3.10.9; other versions can lead to small
differences in the layout (e.g. the line spacing of multi-line labels).

`quiet(...)` suppresses only the warnings with the given messages, e.g. the
`tight_layout` warnings of figures with a broken axis, whose layout is correct.

In [ ]:
fontsize = 9  # same as font size in figure captions (for class option 10pt)
labelsize = fontsize  # same as font size

# Paper style (Figures 1-5). The figure width of 6.05 inches corresponds to the text
# width of the paper, so that figures can be included without rescaling. Individual
# figures override the height and margins.
PAPER_RC = {
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{amsmath,amssymb}",
    "lines.linewidth": 1.0,
    "font.size": fontsize,
    "font.family": "serif",
    "mathtext.default": "regular",
    "axes.linewidth": 0.8,
    "axes.labelsize": fontsize,
    "xtick.top": True,
    "xtick.direction": "in",
    "xtick.labelsize": fontsize,
    "ytick.right": True,
    "ytick.direction": "in",
    "ytick.labelsize": fontsize,
    "xtick.major.size": 3.5,
    "xtick.major.width": 0.8,
    "xtick.major.pad": 4,
    "xtick.minor.size": 2.0,
    "xtick.minor.width": 0.8,
    "xtick.minor.pad": 4,
    "ytick.major.size": 3.5,
    "ytick.major.width": 0.8,
    "ytick.major.pad": 4,
    "ytick.minor.size": 2.0,
    "ytick.minor.width": 0.8,
    "ytick.minor.pad": 4,
    "figure.figsize": (6.052269601100412, 4.0769828309954015),
    "figure.titlesize": 10,
    "figure.dpi": 500,
    "figure.subplot.left": 0.09913636363636365,
    "figure.subplot.right": 0.975215909090909,
    "figure.subplot.bottom": 0.09811176955639507,
    "figure.subplot.top": 0.9018882304436049,
    "figure.subplot.wspace": 0.0,
    "figure.subplot.hspace": 0.0,
    "savefig.format": "pdf",
}

# Style of Figures 6-17 (on top of the matplotlib defaults)
PROFILE_RC = {
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "Times New Roman", "DejaVu Serif"],
    "font.size": 14,
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 14,
    "figure.titlesize": 16,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "none",
    "grid.alpha": 0.3,
    "grid.color": "gray",
    "axes.grid": True,
    "axes.edgecolor": "black",
    "axes.linewidth": 1.0,
}

# Larger fonts of Figures 8 and 10 and of Figure 2 of the letter
LARGE_FONT_RC = {
    "axes.labelsize": 24,
    "axes.titlesize": 21,
    "xtick.labelsize": 19,
    "ytick.labelsize": 19,
    "legend.fontsize": 16,
}


@contextmanager
def figure_style(base="paper", extra=None):
    """Apply a figure style inside a `with` block only.

    base: "paper" (PAPER_RC on top of the current settings) or "profile"
    (matplotlib defaults + PROFILE_RC). extra: optional dict of rcParams applied on
    top. The global rcParams are restored when the block is left.
    """
    with mpl.rc_context():
        if base == "paper":
            mpl.rcParams.update(PAPER_RC)
        else:
            mpl.rcdefaults()
            mpl.rcParams.update(PROFILE_RC)
        if extra:
            mpl.rcParams.update(extra)
        yield


@contextmanager
def quiet(*messages):
    """Suppress only the warnings whose message starts with one of `messages`."""
    with warnings.catch_warnings():
        for message in messages:
            warnings.filterwarnings("ignore", message=re.escape(message))
        yield


TIGHT_LAYOUT_WARNING = "This figure includes Axes that are not compatible with tight_layout"

### Main colors

Colors and labels for the different stationary fluid-profile solution types.

We use shades of blue for direct profiles, shades of red for inverse profiles,
as well as purple and green for mixed hybrids. The colors are taken from the
5-class single-hue sequential color maps of
[colorbrewer2.org](https://colorbrewer2.org/). The order of `color_dict` is the
order in which the branches are drawn and listed in the legends.

The figures of the letter merge the two sub-branches of the direct hybrids and of the
inverse detonations into one region each (`merged_color_dict`); some figures use
the shorter labels `short_label_dict`.

In [ ]:
color_dict = {
    ST.Detonation : '#08519c',
    ST.HybridPlus : '#3182bd',
    ST.Hybrid : '#6baed6',
    ST.Deflagration : '#bdd7e7',
    ST.InverseDetonationPlus : '#a50f15',
    ST.InverseDetonation : '#de2d26',
    ST.InverseHybrid : '#fb6a4a',
    ST.InverseDeflagration : '#fcae91',
    ST.MixedHybridPlus : '#756bb1',
    ST.MixedHybridMinus : '#31a354',
}

label_dict = {
    ST.Detonation : "direct detonation",
    ST.HybridPlus : "direct hybrid `$\\scriptstyle (+)$'",
    ST.Hybrid : "direct hybrid `$\\scriptstyle (-)$'",
    ST.Deflagration : "direct deflagration",
    ST.InverseDetonationPlus : "inverse detonation `$\\scriptstyle (+)$'",
    ST.InverseDetonation : "inverse detonation `$\\scriptstyle (-)$'",
    ST.InverseHybrid : "inverse hybrid",
    ST.InverseDeflagration : "inverse deflagration",
    ST.MixedHybridPlus : "mixed hybrid, $v_+=c_+$",
    ST.MixedHybridMinus : "mixed hybrid, $v_-=c_-$",
}

short_label_dict = {
    ST.Detonation: "direct detonation",
    ST.HybridPlus: "direct hybrid +",
    ST.Hybrid: "direct hybrid -",
    ST.Deflagration: "direct deflagration",
    ST.InverseDetonationPlus: "inverse detonation +",
    ST.InverseDetonation: "inverse detonation -",
    ST.InverseHybrid: "inverse hybrid",
    ST.InverseDeflagration: "inverse deflagration",
    ST.MixedHybridPlus: r"mixed, $v_+=c_+$",
    ST.MixedHybridMinus: r"mixed, $v_-=c_-$",
}

# Letter: direct hybrids in the colour of DH+, deflagrations in the former colour of
# DH, both inverse detonations in the colour of ID+
merged_color_dict = {
    **color_dict,
    ST.Hybrid: '#3182bd',
    ST.Deflagration: '#6baed6',
    ST.InverseDetonation: '#a50f15',
}

### Plot settings

We generate plots for $\xi$ between `ximin=0.0001` and `ximax=0.9999` with
`N_points=101` linear steps.

In [ ]:
N_points = 101
ximin = 1e-4
ximax = 1.-ximin

### Output

The figures are saved as PDF files in the folder `figures`.

In [ ]:
save_figures = True
save_prefix = "figures/"
Path(save_prefix).mkdir(exist_ok=True)

### Saving and reusing scans

The results of the expensive scans (Figures 7–14 and 16 and the letter figures) are
computed with `cached_scan(name, compute)`. With `SAVE_SCANS = True` the result of
`compute()` is saved as `data/<name>.pkl`; with `REUSE_SCANS = True` a saved result
is loaded instead of being recomputed, so that the figures can be regenerated in a
few minutes. A saved scan is reused as it is: after changing the parameters of a
scan, delete the corresponding file in `data` (or set `REUSE_SCANS = False` to
recompute all scans).

In [ ]:
SAVE_SCANS = True
REUSE_SCANS = False
DATA_DIR = Path("data")


def cached_scan(name, compute):
    """Return compute(), loaded from (and saved to) DATA_DIR/<name>.pkl."""
    path = DATA_DIR / f"{name}.pkl"
    if REUSE_SCANS and path.is_file():
        with open(path, "rb") as f:
            return pickle.load(f)
    result = compute()
    if SAVE_SCANS:
        DATA_DIR.mkdir(exist_ok=True)
        with open(path, "wb") as f:
            pickle.dump(result, f)
    return result

## Figure 1

Different types of solutions for the fluid motion in front (green region) and behind (pink region)
the bubble wall: compression waves (dark green or dark pink) end at a shock front (grey dotted line),
whereas rarefaction waves go to $v=0$ at $\xi=c_s$. The grey region does not support consistent
solutions. For simplicity, the figure assumes that the speed of sound $c_s$ is constant.

The fluid profiles $v(\xi)$ are self-similar solutions of the hydrodynamic
equations, which we obtain by numerically integrating $\xi(v)$ with
`pc.integrate_xi_of_v`. The helper function `plot_trajectories` draws a set of
such trajectories covering the $(\xi, v)$ plane, and its `select` argument
allows to only draw the segments lying in a given region. We first draw all
trajectories as thin dotted lines, and then overlay the physically relevant
segments (compression and rarefaction waves ahead of and behind the wall) in
color.

In [ ]:
def plot_trajectories(
    ax,
    cs,
    select=lambda xi, v: np.ones_like(xi, dtype=bool),
    step_size=1e-3,
    **plot_kwargs
):
    """Plot a family of solutions v(xi) of the fluid equation of motion.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes to draw on
    cs : float
        Speed of sound (assumed constant)
    select : callable, optional
        Function mapping the arrays xi and v of a trajectory to a boolean mask;
        only the points where the mask is True are plotted (default: all)
    step_size : float, optional
        Integration step size in v (default: 1e-3)
    **plot_kwargs
        Passed on to ax.plot
    """
    # Stay slightly away from |v| = 1 and v = 0, where the equation is singular.
    vmin = 1e-4
    vmax = 1.0-vmin
    # Note: `integrate_xi_of_v` appends to the input arrays, continuing from
    # their last element. Integrating in one direction, reversing the result,
    # and integrating in the opposite direction thus yields a trajectory
    # extending to both sides of the initial point. The shock check is switched
    # off (stop_at_shock=False) to obtain the full trajectories.
    for v0 in np.arange(.1, 1., .1):
        # positive v: trajectory through the point xi = v = v0 on the
        # co-moving line, integrated up towards v = 1 and down towards v = 0
        _, xi, v = pc.integrate_xi_of_v(
            np.array([v0,]),
            np.array([v0,]),
            vmax,
            step_size,
            False,
            cs
        )
        _, xi, v = pc.integrate_xi_of_v(
            xi[::-1].copy(),
            v[::-1].copy(),
            vmin,
            -step_size,
            False,
            cs
        )
        idx = select(xi,v)
        ax.plot(xi[idx], v[idx], **plot_kwargs)
        # positive v: trajectory starting from (xi, v) = (1, v0), integrated
        # down towards v = 0
        _, xi, v = pc.integrate_xi_of_v(
            np.array([vmax,]),
            np.array([v0,]),
            vmin,
            -step_size,
            False,cs
        )
        idx = select(xi,v)
        ax.plot(xi[idx], v[idx], **plot_kwargs)
        # negative v: trajectory through (xi, v) = (1, -v0), integrated up
        # towards v = 0 and down towards v = -1
        _, xi, v = pc.integrate_xi_of_v(
            np.array([vmax,]),
            np.array([-v0,]),
            -vmin,
            step_size,
            False,
            cs
        )
        _, xi, v = pc.integrate_xi_of_v(
            xi[::-1].copy(),
            v[::-1].copy(),
            -vmax,
            -step_size,
            False,
            cs
        )
        idx = select(xi,v)
        ax.plot(xi[idx], v[idx], **plot_kwargs)
    # additional trajectories close to v = -1, where they are spaced more
    # densely
    for v0 in [.95, .99, .999]:
        _, xi, v = pc.integrate_xi_of_v(
            np.array([vmax,]),
            np.array([-v0,]),
            -vmin,
            step_size,
            False,
            cs
        )
        _, xi, v = pc.integrate_xi_of_v(
            xi[::-1].copy(),
            v[::-1].copy(),
            -vmax,
            -step_size,
            False,
            cs
        )
        idx = select(xi,v)
        ax.plot(xi[idx], v[idx], **plot_kwargs) 

In [ ]:
with figure_style("paper"):
    fig, ax = plt.subplots(1, 1, figsize=(6.052269601100412, 3.002269601100412))
    fig.subplots_adjust(
        left=0.09913636363636365,
        right=0.5043181818181818,
        bottom=0.13323253842805766,
        top=0.9500377980894784
    )

    # 7-class PiYG diverging color map from colorbrewer2.org: pink for the region
    # behind the wall, green for the region ahead of it
    colors = [
        '#c51b7d', '#e9a3c9', '#fde0ef',
        '#f7f7f7',
        '#e6f5d0', '#a1d76a', '#4d9221'
    ]

    # speed of sound, taken to be constant (bag model value 1/sqrt(3))
    cs = hydrodex.cs_bag

    ax.set_xlabel("$\\xi$")
    ax.set_ylabel("$v$")
    ax.set_xlim(0.,1.)
    ax.set_xticks([0.,cs**2,cs,1.], labels=["$0$","$c_s^2$","$c_s$","$1$"])
    ax.set_ylim(-1.,1.)
    ax.set_yticks([-1.,-cs,0.,cs,1.], labels=["$-1$","$-c_s$","$0$","$c_s$","$1$"])

    xi = np.linspace(ximin, ximax, N_points)

    # Shaded regions in the (xi, v) plane, delimited by the co-moving line v = xi,
    # the sonic curve mu(xi,v) = cs, and the shock curve xi mu(xi,v) = cs^2:
    # - above the co-moving line (gray)
    ax.fill_between(xi, xi, 1., fc='0.5', ec=None, alpha=.33, zorder=0.5)
    # - below the co-moving line and both curves: solutions ahead of the wall (green)
    handle_front = ax.fill_between(
        xi,
        np.maximum(pc.mu(xi, cs), pc.mu(xi, cs**2/xi)),
        xi,
        fc=colors[-3], ec=None, alpha=1., zorder=0.5,
        label='allowed solution space preceding the wall'
    )
    # - between the sonic and shock curves: no consistent solutions (gray)
    handle_forbidden = ax.fill_between(
        xi,
        np.minimum(pc.mu(xi, cs), pc.mu(xi, cs**2/xi)),
        np.maximum(pc.mu(xi, cs), pc.mu(xi, cs**2/xi)),
        fc='0.5', ec=None, alpha=.33, zorder=0.5,
        label='no consistent solution'
    )
    # - beyond both curves: solutions behind the wall (pink)
    handle_back = ax.fill_between(
        xi,
        -1.,
        np.minimum(pc.mu(xi, cs), pc.mu(xi, cs**2/xi)),
        fc=colors[2], ec=None, alpha=1., zorder=0.5,
        label='allowed solution space trailing the wall'
    )

    # characteristic curves
    col_cs = '0.2'
    col_sh = '0.3'
    handle_sonic = ax.plot(
        xi, pc.mu(xi, cs),
        c=col_cs, lw=1.5, zorder=1.5, ls=(0,(4,1.5)),
        label=r"sonic curve, $\mu^2(\xi,v) = c_s^2$"
    )[0]
    ax.plot(
        xi, pc.mu(xi, -cs),
        c=col_cs, lw=1.5, zorder=1.5, ls=(0,(4,1.5))
    )
    handle_shock = ax.plot(
        xi, pc.mu(xi, cs**2/xi),
        c=col_sh, lw=1.5, zorder=1.5, ls=(0,(1,1.5)),
        label=r"shock front, $\xi \mu(\xi,v) = c_s^2$"
    )[0]
    handle_comov = ax.plot(
        xi, xi,
        c='k', lw=1.5, zorder=1.5,
        label=r"co-moving front, $\xi = v$"
    )[0]
    ax.axhline(0., c='k', lw=1., zorder=1.5)
    handle_cs = ax.axvline(
        cs, c='0.1', lw=1., ls=(0,(5,5)), zorder=1.5, label=r"$\xi=c_s$"
    )

    # all solution trajectories (thin dotted lines)
    plot_trajectories(ax, cs, c='k', lw=0.5, ls=(0,(1,6)), zorder=1.)
    # physical segments, from top to bottom: compression and rarefaction waves
    # ahead of the wall (v >= 0 and v <= 0 below the sonic and shock curves),
    # rarefaction and compression waves behind the wall (v >= 0 and v <= 0 beyond
    # both curves)
    plot_trajectories(ax, cs, select=lambda xi,v: np.logical_and(
        np.logical_and(v>=0., xi>=v),
        np.logical_and(pc.mu(xi,v)<=cs,xi*pc.mu(xi,v)<=cs**2)
    ), step_size=1e-4, c=colors[-1], lw=1., zorder=1.1)
    plot_trajectories(ax, cs, select=lambda xi,v: np.logical_and(
        np.logical_and(v<=0., xi>=v),
        np.logical_and(pc.mu(xi,v)<=cs,xi*pc.mu(xi,v)<=cs**2)
    ), step_size=1e-4, c=colors[-2], lw=1., zorder=1.1)
    plot_trajectories(ax, cs, select=lambda xi,v: np.logical_and(
        v>=0., np.logical_and(xi*pc.mu(xi,v)>=cs**2,pc.mu(xi,v)>=cs)
    ), step_size=1e-4, c=colors[1], lw=1., zorder=1.1)
    plot_trajectories(ax, cs, select=lambda xi,v: np.logical_and(
        v<=0., np.logical_and(xi*pc.mu(xi,v)>=cs**2,pc.mu(xi,v)>=cs)
    ), step_size=1e-4, c=colors[0], lw=1., zorder=1.1)

    # legend outside the axes, to the right of the plot
    ax.legend(
        loc = "upper left",
        handles = [
            handle_front,
            handle_back,
            handle_forbidden,
            mpl.lines.Line2D([],[], c='k', lw=0.5, ls=(0,(1,6)),
                             label="solution trajectories"),
            mpl.lines.Line2D([],[], c=colors[-1], lw=1.,
                             label="compression waves ahead of the wall"),
            mpl.lines.Line2D([],[], c=colors[-2], lw=1.,
                             label="rarefaction waves ahead of the wall"),
            mpl.lines.Line2D([],[], c=colors[0], lw=1.,
                             label="compression waves behind the wall"),
            mpl.lines.Line2D([],[], c=colors[1], lw=1.,
                             label="rarefaction waves behind the wall"),
            handle_shock,
            handle_sonic,
            handle_comov,
            handle_cs
        ],
        frameon = False,
        bbox_to_anchor=(1,1.033),
        labelspacing=.75
    )

    if save_figures: plt.savefig(save_prefix+"solutions")

    plt.show()
    plt.close()

## Figure 2

Fluid velocities $v_+$ and $v_-$ ahead of and behind the wall in the wall rest frame for different
values of $\alpha_+$ (see contour labels). The '(+)' and '(−)' branches are shown in orange and
purple, respectively, with the branch cuts occurring along the dash-dotted contour tracing the
extremal values of $v_+(v_-,\alpha_+)$ and the dotted $v_-=c_-$ line. The solid and dashed black
lines indicate the $v_+=v_-$ and $v_+v_-=c_-^2$ contours across which the sign of $\alpha_+$
changes.

The junction conditions across the wall relate the velocities $v_+$ and $v_-$
for a given $\alpha_+$, with two solution branches '(+)' and '(-)'. The left
panel shows the branches of $v_+(v_-,\alpha_+)$, the right panel the branches of
the inverse function $v_-(v_+,\alpha_+)$. Note that the contours for
$\alpha_+ > 0$ are computed with `pc.get_v_plus_from_v_minus`, whereas those for
$\alpha_+ < 0$ are computed with `pc.get_v_minus_from_v_plus`, and the color is
switched at the respective branch cut ($v_\text{extr}$ in the left panel,
$v_-=c_-$ in the right panel). The contour labels are placed and rotated by
hand.

In [ ]:
with figure_style("paper"):
    fig, axs = plt.subplots(1,2, figsize=(6.052269601100412, 2.701134800550206))
    fig.subplots_adjust(
        left = 0.08261363636363638,
        right = 0.7934659090909091,
        bottom = 0.14808590815923822,
        top = 0.8519140918407617,
        wspace = 0.26300081396400476
    )

    v = np.linspace(ximin, ximax, N_points)
    cm = hydrodex.cs_bag # speed of sound c_- behind the wall

    # axes and reference lines common to both panels
    for ax in axs: 
        ax.set_ylabel("$v_+$")
        ax.set_xlabel("$v_-$")
        ax.set_xlim(0.,1.)
        ax.set_ylim(0.,1.)
        ax.set_xticks([0.,cm**2,cm,1.], labels=["$0$","$c_-^2$","$c_-$","$1$"])
        ax.set_yticks([0.,cm**2,cm,1.], labels=["$0$","$c_-^2$","$c_-$","$1$"])

        # alpha_+ changes sign across v_+ = v_- and v_+ v_- = c_-^2
        ax.plot(v,v, c='k', zorder=1.6)
        ax.plot(v, cm**2/v, c='k', ls='--', zorder=1.6)

        # branch cut of v_-(v_+, alpha_+) for alpha_+ > 0
        ax.axvline(cm, c='k', ls=':', zorder=1.6)

        # branch cut of v_+(v_-, alpha_+) for alpha_+ < 0, along the extrema
        vm = np.linspace(cm**2, ximax, N_points)
        vp = pc.get_v_plus_extremum_from_v_minus(vm, cm)
        ax.plot(vm, vp, c='k', ls='-.', zorder=1.6)

    lw_cnt = 1.5 # line width of the alpha_+ contours
    col_plus = '#d95f02'
    col_minus = '#7570b3'

    # Contour labels: the label is placed at (xpos, ypos) on the contour and
    # rotated along it, with the slope estimated by a finite difference.

    # left panel: v_+(v_-, alpha_+)
    ax = axs[0]
    # alpha_+ > 0, '(-)' branch
    for alpha in [0.01,0.05,0.1,0.2]:
        v_ = pc.get_v_plus_from_v_minus(v, alpha, pc.RootBranch.Minus, cm)
        ax.plot(v, v_, c=col_minus, lw=lw_cnt, zorder=1.5)
        xpos = cm+0.06
        ypos = pc.get_v_plus_from_v_minus(xpos, alpha, pc.RootBranch.Minus, cm)
        angle = 180./np.pi*np.arctan((
            pc.get_v_plus_from_v_minus(xpos+0.001, alpha, pc.RootBranch.Minus, cm)
            - ypos
        )/0.001)
        ax.text(xpos, ypos-0.025, r'\tiny %g'%alpha, va='center', ha='center',
                rotation=angle)
    # alpha_+ > 0, '(+)' branch
    for alpha in [0.01,0.1,0.3,1.]:
        v_ = pc.get_v_plus_from_v_minus(v, alpha, pc.RootBranch.Plus, cm)
        ax.plot(v, v_, c=col_plus, lw=lw_cnt, zorder=1.5)
        xpos = cm+0.06
        ypos = pc.get_v_plus_from_v_minus(xpos, alpha, pc.RootBranch.Plus, cm)
        angle = 180./np.pi*np.arctan((
            pc.get_v_plus_from_v_minus(xpos+0.001, alpha, pc.RootBranch.Plus, cm)
            - ypos
        )/0.001)
        ax.text(xpos, ypos+0.035, r'\tiny %g'%alpha, va='center', ha='center',
                rotation=angle)
    # alpha_+ < 0: computed as v_-(v_+) (on its '(-)' branch) and split at the
    # extremum v_+ = vc into the '(-)' (below) and '(+)' (above) branch of v_+(v_-)
    for alpha in [-0.01,-0.1,-1.,-10.]:
        v_ = pc.get_v_minus_from_v_plus(v, alpha, pc.RootBranch.Minus, cm)
        vc = pc.get_v_plus_extremum_from_alpha(alpha, cm)
        ax.plot(np.concatenate([
            v_[v<vc],
            [pc.get_v_minus_from_v_plus(vc, alpha, pc.RootBranch.Minus, cm),]
        ]), np.concatenate([v[v<vc],[vc,]]), c=col_minus, lw=lw_cnt, zorder=1.5)
        ax.plot(np.concatenate([[
            pc.get_v_minus_from_v_plus(vc, alpha, pc.RootBranch.Minus, cm),],
            v_[v>vc]
        ]), np.concatenate([[vc,],v[v>vc]]), c=col_plus, lw=lw_cnt, zorder=1.5)
        ypos = {-0.01:0.45,-0.1:0.5,-1.:0.75,-10.:0.9}[alpha]
        xpos = pc.get_v_minus_from_v_plus(ypos, alpha, pc.RootBranch.Minus, cm)
        angle = 180./np.pi*np.arctan(0.001/(
            pc.get_v_minus_from_v_plus(ypos+0.001, alpha, pc.RootBranch.Minus, cm)
            - xpos
        ))
        ax.text(xpos-0.04, ypos, r'\tiny %g'%alpha, va='center', ha='center',
                rotation=angle)
    # alpha_+ < 0: same for the '(+)' branch of v_-(v_+)
    for alpha in [-0.01,-0.05,-0.1,-0.2]:
        v_ = pc.get_v_minus_from_v_plus(v, alpha, pc.RootBranch.Plus, cm)
        vc = pc.get_v_plus_extremum_from_alpha(alpha, cm)
        ax.plot(np.concatenate([
            v_[v<vc],
            [pc.get_v_minus_from_v_plus(vc, alpha, pc.RootBranch.Plus, cm),]
        ]), np.concatenate([v[v<vc],[vc,]]), c=col_minus, lw=lw_cnt, zorder=1.5)
        ax.plot(np.concatenate([
            [pc.get_v_minus_from_v_plus(vc, alpha, pc.RootBranch.Plus, cm),],
            v_[v>vc]
        ]), np.concatenate([[vc,],v[v>vc]]), c=col_plus, lw=lw_cnt, zorder=1.5)
        ypos = {-0.01:0.5, -0.05:0.5, -0.1:0.63, -0.2:0.7}[alpha]
        xpos = pc.get_v_minus_from_v_plus(ypos, alpha, pc.RootBranch.Plus, cm)
        angle = 180./np.pi*np.arctan(0.001/(
            pc.get_v_minus_from_v_plus(ypos+0.001, alpha, pc.RootBranch.Plus, cm)
            - xpos
        ))
        offset={-0.01:0.05, -0.05:0.05, -0.1:0.035, -0.2:-0.02}[alpha]
        ax.text(xpos+offset, ypos, r'\tiny %g'%alpha, va='center', ha='center', 
                rotation=angle)
    ax.set_title(r'$v_+^{(\pm)}(v_-,\alpha_+)$', pad=10)

    # right panel: v_-(v_+, alpha_+)
    ax = axs[1]
    # alpha_+ > 0: computed as v_+(v_-) and split at v_- = c_- into the '(-)'
    # (v_- < c_-) and '(+)' (v_- > c_-) branch of v_-(v_+)
    for alpha in [0.01,0.05,0.1,0.2]:
        v_ = pc.get_v_plus_from_v_minus(v, alpha, pc.RootBranch.Minus, cm)
        vc = pc.get_v_plus_from_v_minus(cm, alpha, pc.RootBranch.Minus, cm)
        ax.plot(np.concatenate([v[v<cm],[cm,]]), np.concatenate([v_[v<cm],[vc,]]),
                c=col_minus, lw=lw_cnt, zorder=1.5)
        ax.plot(np.concatenate([[cm,],v[v>cm]]), np.concatenate([[vc,],v_[v>cm]]),
                c=col_plus, lw=lw_cnt, zorder=1.5)
        xpos = cm+0.06
        ypos = pc.get_v_plus_from_v_minus(xpos, alpha, pc.RootBranch.Minus, cm)
        angle = 180./np.pi*np.arctan((
            pc.get_v_plus_from_v_minus(xpos+0.001, alpha, pc.RootBranch.Minus, cm)
            - ypos
        )/0.001)
        ax.text(xpos, ypos-0.025, r'\tiny %g'%alpha, va='center', ha='center',
                rotation=angle)
    for alpha in [0.01,0.1,0.3,1.]:
        v_ = pc.get_v_plus_from_v_minus(v, alpha, pc.RootBranch.Plus, cm)
        vc = pc.get_v_plus_from_v_minus(cm, alpha, pc.RootBranch.Plus, cm)
        ax.plot(np.concatenate([v[v<cm],[cm,]]), np.concatenate([v_[v<cm],[vc,]]),
                c=col_minus, lw=lw_cnt, zorder=1.5)
        ax.plot(np.concatenate([[cm,],v[v>cm]]), np.concatenate([[vc,],v_[v>cm]]),
                c=col_plus, lw=lw_cnt, zorder=1.5)
        xpos = cm+0.06
        ypos = pc.get_v_plus_from_v_minus(xpos, alpha, pc.RootBranch.Plus, cm)
        angle = 180./np.pi*np.arctan((
            pc.get_v_plus_from_v_minus(xpos+0.001, alpha, pc.RootBranch.Plus, cm)
            - ypos
        )/0.001)
        ax.text(xpos, ypos+0.035, r'\tiny %g'%alpha, va='center', ha='center',
                rotation=angle)
    # alpha_+ < 0: '(-)' and '(+)' branch of v_-(v_+) are computed directly
    for alpha in [-0.01,-0.1,-1.,-10.]:
        v_ = pc.get_v_minus_from_v_plus(v, alpha, pc.RootBranch.Minus, cm)
        ax.plot(v_, v, c=col_minus, lw=lw_cnt, zorder=1.5)
        ypos = {-0.01:0.45,-0.1:0.5,-1.:0.75,-10.:0.9}[alpha]
        xpos = pc.get_v_minus_from_v_plus(ypos, alpha, pc.RootBranch.Minus, cm)
        angle = 180./np.pi*np.arctan(0.001/(
            pc.get_v_minus_from_v_plus(ypos+0.001, alpha, pc.RootBranch.Minus, cm)
            - xpos
        ))
        ax.text(xpos-0.04, ypos, r'\tiny %g'%alpha, va='center', ha='center',
                rotation=angle)
    for alpha in [-0.01,-0.05,-0.1,-0.2]:
        v_ = pc.get_v_minus_from_v_plus(v, alpha, pc.RootBranch.Plus, cm)
        ax.plot(v_, v, c=col_plus, lw=lw_cnt, zorder=1.5)
        ypos = {-0.01:0.5, -0.05:0.5, -0.1:0.63, -0.2:0.7}[alpha]
        xpos = pc.get_v_minus_from_v_plus(ypos, alpha, pc.RootBranch.Plus, cm)
        angle = 180./np.pi*np.arctan(0.001/(
            pc.get_v_minus_from_v_plus(ypos+0.001, alpha, pc.RootBranch.Plus, cm)
            - xpos
        ))
        offset={-0.01:0.05, -0.05:0.05, -0.1:0.035, -0.2:-0.02}[alpha]
        ax.text(xpos+offset, ypos, r'\tiny %g'%alpha, va='center', ha='center',
                rotation=angle)

    ax.set_title(r'$v_-^{(\pm)}(v_+,\alpha_+)$', pad=10)

    ax.legend(
        loc='upper left',
        bbox_to_anchor=(1.05,1.),
        handles = [
            mpl.lines.Line2D([],[], c=col_plus, lw=lw_cnt, label=r"`$(+)$' branch"),
            mpl.lines.Line2D([],[], c=col_minus, lw=lw_cnt, label=r"`$(-)$' branch"),
            mpl.lines.Line2D([],[], c='k', ls='-', label=r"$v_+=v_-$"),
            mpl.lines.Line2D([],[], c='k', ls='--', label=r"$v_+ v_- = c_-^2$"),
            mpl.lines.Line2D([],[], c='k', ls=':', label=r"$v_- = c_-$"),
            mpl.lines.Line2D([],[], c='k', ls='-.', label=r"$v_\text{extr}$")
        ],
        frameon=False,
        handlelength=1.,
        labelspacing=.75
    )

    if save_figures: plt.savefig(save_prefix+"matching")

    plt.show()
    plt.close()

## Figure 3

Classification of solutions in terms of $v_-$ and $v_+$. Direct solutions are shown in shades of
blue and inverse solutions in shades of red (from dark to light: detonation, hybrid, deflagration).
The purple and green regions correspond to mixed direct–inverse hybrids with $v_+=c_+$ or
$v_-=c_-$, respectively. For all hybrid solutions (direct, inverse, and mixed) either $v_+=c_+$ or
$v_-=c_-$, and the corresponding axis is plotted in terms of the wall velocity $\xi_w$ rather than
$v_-$ or $v_+$. The dashed light-grey lines correspond to $v_\pm=c_-$ and $v_+v_-=c_-^2$, whereas
the dark-grey ones indicate $v_\pm=c_+$ and $v_+v_-=c_+^2$.

The function `make_plot_vp_vm` draws one panel for given speeds of sound $c_+$
and $c_-$ ahead of and behind the wall. The region of each solution type is
obtained from the range of wall velocities `pc.xi_min` to `pc.xi_max` for which
it exists, together with the constraints on $v_\pm$ (e.g. $v_+ = \xi_w$ for
detonations). For hybrid regions, the horizontal ($v_-=c_-$) or vertical
($v_+=c_+$) coordinate is $\xi_w$ instead. Overlapping solution types are
hatched. The four panels cover the different orderings of $c_+$, $c_-$, $c_+^2$,
and $c_-^2$. The thin black lines are $\alpha_+$ contours as in figure 2.

In [ ]:
def make_plot_vp_vm(ax, cp, cm):
    """Draw the regions of the different solution types in the (v_-, v_+) plane.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes to draw on
    cp : float
        Speed of sound c_+ ahead of the wall
    cm : float
        Speed of sound c_- behind the wall
    """
    ax.set_title(f"$c_+ = {cp},\\ c_-={cm}$")
    
    # axis range
    ax.set_xlim(0.,1.)
    ax.set_ylim(0.,1.)
    ax.set_xticks([0.,cm**2,cp**2,cm,cp,1.],
                  labels=["$0$","$c_-^2$","$c_+^2$","$c_-$","$c_+$","$1$"])
    ax.set_yticks([0.,cm**2,cp**2,cm,cp,1.],
                  labels=["$0$","$c_-^2$","$c_+^2$","$c_-$","$c_+$","$1$"])
    
    # reference lines: v_+ = v_- (black), as well as v_pm = c_pm and
    # v_+ v_- = c_pm^2 (gray)
    v = np.linspace(ximin, ximax, N_points)
    ax.plot(v,v, c='k', zorder=1.5)
    colp = '.3'
    colm = '.5'
    
    ax.axhline(cp, c=colp, ls=':', zorder=1.5)
    ax.axvline(cp, c=colp, ls=':', zorder=1.5)
    ax.axhline(cm, c=colm, ls='--', zorder=1.5)
    ax.axvline(cm, c=colm, ls='--', zorder=1.5)
    ax.plot(v, cp**2/v, c=colp, ls=':', zorder=1.5)
    ax.plot(v, cm**2/v, c=colm, ls='--', zorder=1.5)

    # alpha_+ contours, as in figure 2
    v = np.linspace(ximin, ximax, N_points)
    for alpha in [0.01,0.05,0.1,0.2]:
        v_ = pc.get_v_plus_from_v_minus(v, alpha, pc.RootBranch.Minus, cm)
        ax.plot(v, v_, c='k', lw=.5, zorder=1.5)
    for alpha in [0.01,0.1,0.3,1.]:
        v_ = pc.get_v_plus_from_v_minus(v, alpha, pc.RootBranch.Plus, cm)
        ax.plot(v, v_, c='k', lw=.5, zorder=1.5)
    for alpha in [-0.01,-0.1,-1.,-10.]:
        v_ = pc.get_v_minus_from_v_plus(v, alpha, pc.RootBranch.Minus, cm)
        ax.plot(v_, v, c='k', lw=.5, zorder=1.5)
    for alpha in [-0.01,-0.05,-0.1,-0.2]:
        v_ = pc.get_v_minus_from_v_plus(v, alpha, pc.RootBranch.Plus, cm)
        ax.plot(v_, v, c='k', lw=.5, zorder=1.5)
    
    # extrema of v_+(v_-, alpha_+) for alpha_+ < 0
    vm = np.linspace(cm**2, ximax, N_points)
    vp = pc.get_v_plus_extremum_from_v_minus(vm, cm)
    ax.plot(vm, vp, c='k', ls='--', lw=.5, zorder=1.5)
    
    # Solution regions. For each type, xi runs over the range of wall velocities
    # for which the solution exists (clipped to [ximin, ximax]). Hybrids are
    # overlaid with a hatch in the same color as they overlap with other
    # solutions (although this overlap is not physical as one of the axes
    # corresponds to xi_w.

    # direct detonation
    sol = pc.HydroSolutionType.Detonation
    xi = np.linspace(
        max(ximin, pc.xi_min(sol, cp, cm)),
        min(ximax, pc.xi_max(sol, cp, cm)),
        N_points
    )
    ax.fill_betweenx(xi, cm, xi, color=color_dict[sol])

    # direct deflagration
    sol = pc.HydroSolutionType.Deflagration
    xi = np.linspace(
        max(ximin, pc.xi_min(sol, cp, cm)),
        min(ximax, pc.xi_max(sol, cp, cm)),
        N_points
    )
    ax.fill_between(xi, 0., np.minimum(xi,np.minimum(cp,cp**2/xi)),
                    color=color_dict[sol])

    # direct hybrid, '+' branch (v+>c-)
    sol = pc.HydroSolutionType.HybridPlus
    if pc.xi_min(sol, cp, cm) < pc.xi_max(sol, cp, cm):
        xi = np.linspace(
            max(ximin, pc.xi_min(sol, cp, cm)),
            min(ximax, pc.xi_max(sol, cp, cm)),
            N_points
        )
        ax.fill_between(xi, cm, np.minimum(cp, np.minimum(xi, cp**2/xi)),
                        color=color_dict[sol])
        ax.fill_between(xi, cm, np.minimum(cp, np.minimum(xi, cp**2/xi)),
                        edgecolor=color_dict[sol], facecolor=(0,0,0,0),
                        hatch='//', lw=0., hatch_linewidth=4., zorder=1.1)

    # direct hybrid, '-' branch (v+<c-)
    sol = pc.HydroSolutionType.Hybrid
    xi = np.linspace(
        max(ximin, pc.xi_min(sol, cp, cm)),
        min(ximax, pc.xi_max(sol, cp, cm)),
        N_points
    )
    ax.fill_between(xi, 0., np.minimum(cp, np.minimum(cm, cp**2/xi)),
                    color=color_dict[sol])
    ax.fill_between(xi, 0., np.minimum(cp, np.minimum(cm, cp**2/xi)),
                    edgecolor=color_dict[sol], facecolor=(0,0,0,0), hatch='//',
                    lw=0., hatch_linewidth=4., zorder=1.1)
    
    # inverse detonation, '-' branch
    sol = pc.HydroSolutionType.InverseDetonation
    xi = np.linspace(
        max(ximin, pc.xi_min(sol, cp, cm)),
        min(ximax, pc.xi_max(sol, cp, cm)),
        N_points
    )
    ax.fill_between(
        xi, xi, np.minimum(cp, pc.get_v_plus_extremum_from_v_minus(xi,cm)),
        color=color_dict[sol]
    )
    
    # inverse detonation, '+' branch
    sol = pc.HydroSolutionType.InverseDetonationPlus
    if pc.xi_min(sol, cp, cm) < pc.xi_max(sol, cp, cm):
        xi = np.linspace(
            max(ximin, pc.xi_min(sol, cp, cm)),
            min(ximax, pc.xi_max(sol, cp, cm)),
            N_points
        )
        ax.fill_between(
            xi, np.minimum(cp, pc.get_v_plus_extremum_from_v_minus(xi,cm)), cp,
            color=color_dict[sol]
        )

    # inverse deflagration
    sol = pc.HydroSolutionType.InverseDeflagration
    xi = np.linspace(
        max(ximin, pc.xi_min(sol, cp, cm)),
        min(ximax, pc.xi_max(sol, cp, cm)),
        N_points
    )
    ax.fill_betweenx(xi, np.maximum(np.maximum(cm,xi),cm**2/xi), 1.0,
                     color=color_dict[sol])
    
    # inverse hybrid
    sol = pc.HydroSolutionType.InverseHybrid
    if pc.xi_min(sol, cp, cm) < pc.xi_max(sol, cp, cm):
        xi = np.linspace(
            max(ximin, pc.xi_min(sol, cp, cm)),
            min(ximax, pc.xi_max(sol, cp, cm)),
            N_points
        )
        ax.fill_betweenx(xi, np.maximum(np.maximum(cm,xi),cm**2/xi), 1.0,
                         color=color_dict[sol])
    
    # mixed hybrid, v+=c+
    sol = pc.HydroSolutionType.MixedHybridPlus
    if pc.xi_min(sol, cp, cm) < pc.xi_max(sol, cp, cm):
        xi = np.linspace(
            max(ximin, pc.xi_min(sol, cp, cm)),
            min(ximax, pc.xi_max(sol, cp, cm)),
            N_points
        )
        ax.fill_betweenx(xi, cm, xi, color=color_dict[sol])
    
    # mixed hybrid, v-=c- (hatch only, since it overlaps with other regions)
    sol = pc.HydroSolutionType.MixedHybridMinus
    if pc.xi_min(sol, cp, cm) < pc.xi_max(sol, cp, cm):
        xi = np.linspace(
            max(ximin, pc.xi_min(sol, cp, cm)),
            min(ximax, pc.xi_max(sol, cp, cm)),
            N_points
        )
        ax.fill_between(
            xi, xi, cp,
            edgecolor=color_dict[sol], facecolor=(0,0,0,0), hatch='\\\\',
            lw=0., hatch_linewidth=4., zorder=1.1
        )
    

In [ ]:
with figure_style("paper"):
    fig, axs = plt.subplots(2,2, figsize=(6.052269601100412, 7.102269601100412))
    fig.subplots_adjust(
        left = 0.09913636363636365,
        right = 0.9669545454545454,
        bottom = 0.1971200867653752,
        top = 0.964799984506183,
        wspace = 0.12115657028580952,
        hspace = 0.20192761714301588
    )

    # c_- > c_+ > c_-^2
    make_plot_vp_vm(axs[0,0], 0.5, .6)
    # c_+ > c_- > c_+^2
    make_plot_vp_vm(axs[0,1], 0.6, 0.5)
    # c_-^2 > c_+
    make_plot_vp_vm(axs[1,0], 0.4, 0.75)
    # c_+^2 > c_-
    make_plot_vp_vm(axs[1,1], 0.75, 0.4)

    for ax in axs[:,0]: 
        ax.set_ylabel("$v_+$")
    for ax in axs[1]: 
        ax.set_xlabel("$v_-$")

    # common legend below the panels (legend entries reordered so that all direct
    # and all inverse solutions are in the same column).
    fig.legend(
        handles=[
            mpl.patches.Patch(color=color_dict[k], label=label_dict[k])
            for k in [list(color_dict.keys())[i] for i in [0,1,2,3,9,4,5,6,7,8]]
        ] + [
            mpl.lines.Line2D([],[], c='k', label="$v_+=v_-$"),
            mpl.lines.Line2D([],[], c='.3', ls='--',
                             label="$v_\\pm=c_+$ or $v_+ v_- = c_+^2$"),
            mpl.lines.Line2D([],[], c='.5', ls='--',
                             label="$v_\\pm=c_-$ or $v_+ v_- = c_-^2$"),
            mpl.lines.Line2D([],[], c='k', lw=0.5,
                             label=r"$\alpha_{+} = \text{const}$"),
            mpl.lines.Line2D([],[], c='k', lw=0.5, ls='--', label="max/min"),
        ],
        loc='lower center',
        bbox_to_anchor=[0.5,0.],
        ncols=3, frameon=False, handlelength=1.5,
    )

    if save_figures: plt.savefig(save_prefix+"v_plus_v_minus")

    plt.show()
    plt.close()

## Figure 4

Classification of possible solutions in terms of $\xi_w$ and $\alpha_+$. Direct solutions are shown
in shades of blue (from dark to light: detonation, hybrid with $v_+$ on the '(+)' branch, hybrid
with $v_+$ on the '(−)' branch, deflagration), whereas inverse solutions are red (from dark to light:
detonation with $v_+$ on the '(+)' branch, detonation with $v_+$ on the '(−)' branch, hybrid,
deflagration). The purple and green regions correspond to mixed hybrids with $v_+=c_+$ and
$v_-=c_-$, respectively. The dashed and dotted lines indicate $\xi_w=c_-$ and $\xi_w=c_+$. Hatched
regions allow for more than one possible solution.

For each solution type, `pc.alpha_min` and `pc.alpha_max` (with
`return_alpha_N=False`) give the range of $\alpha_+$ for which a solution exists
at a given wall velocity $\xi_w$. These are analytic and therefore fast to
evaluate. Overlapping regions are made visible by drawing each region twice:
once filled, with increasing `zorder`, and once hatched, with decreasing
`zorder`, such that the hatches of the lower regions are drawn on top of the
fills of the higher ones.

In [ ]:
def make_plot_xiw_alphap(ax, cp, cm):
    """Draw the regions of the different solution types in the (xi_w, alpha_+)
    plane.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes to draw on
    cp : float
        Speed of sound c_+ ahead of the wall
    cm : float
        Speed of sound c_- behind the wall
    """
    ax.set_title(f"$c_+ = {cp},\\ c_-={cm}$")
    
    ax.set_xlim(0.,1.)
    ax.set_ylim(-1.,1.)
    ax.set_xticks([0.,cp,cm,1.], labels=["$0$","$c_+$","$c_-$","$1$"])

    ax.axhline(0., c='k')
    ax.axvline(cm, c='k', ls='--')
    ax.axvline(cp, c='k', ls=':')

    def add(sol, hatch="//", hatch_linewidth=4.):
        """Fill and outline the region alpha_min < alpha_+ < alpha_max of
        solution type `sol`. The counter `add.z` sets the stacking order."""
        z = add.z
        xi = np.linspace(
            max(ximin, pc.xi_min(sol, cp, cm)),
            min(ximax, pc.xi_max(sol, cp, cm)),
            N_points
        )
        if xi[0] > xi[-1]: return
        amin = pc.alpha_min(sol, xi, cp, cm, False)
        amax = pc.alpha_max(sol, xi, cp, cm, False)
        ax.fill_between(xi, amin, amax, ec=None, fc=color_dict[sol], 
                        zorder=1.+z*1e-3)
        ax.fill_between(xi, amin, amax, fc=(0,0,0,0), ec=color_dict[sol], 
                        hatch=hatch, hatch_linewidth=hatch_linewidth,
                        zorder=1.1-z*1e-3)
        # boundary of the region, including vertical edges at the endpoints
        # of the xi_w range (unless they lie at the boundary of the plot)
        ax.plot(xi, amax, c='k')
        ax.plot(xi, amin, c='k')
        if xi[0] > ximin: ax.plot([xi[0],xi[0]],[amin[0],amax[0]], c='k')
        if xi[-1] < ximax: ax.plot([xi[-1],xi[-1]],[amin[-1],amax[-1]], c='k')
        add.z += 1
    add.z=0
    
    # The hatch direction is flipped for regions that overlap with another one
    # of the same direction.
    add(pc.HydroSolutionType.Detonation)
    add(pc.HydroSolutionType.Deflagration)
    add(pc.HydroSolutionType.Hybrid, hatch='\\\\')
    add(pc.HydroSolutionType.HybridPlus)
    add(pc.HydroSolutionType.InverseDetonation)
    add(pc.HydroSolutionType.InverseDetonationPlus, hatch='\\\\')
    add(pc.HydroSolutionType.InverseDeflagration)
    add(pc.HydroSolutionType.InverseHybrid)
    add(pc.HydroSolutionType.MixedHybridPlus)
    add(pc.HydroSolutionType.MixedHybridMinus)

In [ ]:
with figure_style("paper"):
    fig, axs = plt.subplots(2,2, figsize=(6.052269601100412, 7.102269601100412),
                            sharex=False, sharey=True)
    fig.subplots_adjust(
        left = 0.09913636363636365,
        right = 0.9669545454545454,
        bottom = 0.1971200867653752,
        top = 0.964799984506183,
        wspace = 0.12115657028580952,
        hspace = 0.20192761714301588
    )

    # c_- > c_+ > c_-^2
    make_plot_xiw_alphap(axs[0,0], 0.45, 0.6)
    # c_+ > c_- > c_+^2
    make_plot_xiw_alphap(axs[0,1], 0.6, 0.45)
    # c_-^2 > c_+
    make_plot_xiw_alphap(axs[1,0], 0.4, 0.75)
    # c_+^2 > c_-
    make_plot_xiw_alphap(axs[1,1], 0.75, 0.4)

    for ax in axs[:,0]: 
        ax.set_ylabel("$\\alpha_+$")
    for ax in axs[1]: 
        ax.set_xlabel("$\\xi_w$")

    fig.legend(
        handles=[
            mpl.patches.Patch(color=color_dict[l], label=label_dict[l])
            for l in color_dict.keys()
        ] + [
            mpl.lines.Line2D([],[], c='k', ls=':', label="$c_+$"),
            mpl.lines.Line2D([],[], c='k', ls='--', label="$c_-$"),
        ],
        loc='lower center',
        bbox_to_anchor=[0.5,0.],
        ncols=3, frameon=False, handlelength=1.5
    )

    if save_figures: plt.savefig(save_prefix+"xi_w_alpha_T+")

    plt.show()
    plt.close()

## Figure 5

Classification of possible solutions in terms of $\xi_w$ and $\alpha_N$. Direct solutions are shown
in shades of blue [from dark to light: detonation, hybrid with $v_+$ on the '(+)' branch, hybrid
with $v_+$ on the '(−)' branch, deflagration], whereas inverse solutions are red [from dark to light:
detonation with $v_+$ on the '(+)' branch, detonation with $v_+$ on the '(−)' branch, hybrid,
deflagration]. The purple and green regions correspond to mixed hybrids with $v_+=c_+$ and
$v_-=c_-$, respectively. The dashed and dotted lines indicate $\xi_w=c_-$ and $\xi_w=c_+$.

As for Figure 4, `make_plot_xiw_alphaN` draws one panel. Here the bounds are
given in terms of the physical strength $\alpha_N$, obtained from
`pc.alpha_min` and `pc.alpha_max` with `return_alpha_N=True`. The function is
also used for the phase-plane panel of Figure 6 (without title, with the regions
starting at $\xi_\text{min}+10^{-4}$, `xi_floor=0`).

In [ ]:
# drawing order of the regions (later regions are drawn on top)
REGION_ORDER = [
    ST.Detonation,
    ST.Deflagration,
    ST.Hybrid,
    ST.HybridPlus,
    ST.InverseDetonation,
    ST.InverseDetonationPlus,
    ST.InverseDeflagration,
    ST.InverseHybrid,
    ST.MixedHybridPlus,
    ST.MixedHybridMinus,
]


def make_plot_xiw_alphaN(ax, cp, cm, title=True, xi_floor=3e-4):
    """Draw the regions of the different solution types in the (xi_w, alpha_N)
    plane.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes to draw on
    cp : float
        Speed of sound c_+ ahead of the wall
    cm : float
        Speed of sound c_- behind the wall
    title : bool, optional
        Show the sound speeds as title (default: True)
    xi_floor : float, optional
        Smallest wall velocity of the regions (default: 3e-4)
    """
    if title:
        ax.set_title(f"$c_+ = {cp},\\ c_-={cm}$")
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(-1.0, 1.0)
    ax.set_xticks([0.0, cp, cm, 1.0], labels=["$0$", "$c_+$", "$c_-$", "$1$"])
    ax.axhline(0.0, c="k")
    ax.axvline(cm, c="k", ls="--")
    ax.axvline(cp, c="k", ls=":")

    for sol in REGION_ORDER:
        xi = np.linspace(
            np.maximum(xi_floor, pc.xi_min(sol, cp, cm) + ximin),
            pc.xi_max(sol, cp, cm) - ximin,
            N_points,
        )
        if xi[0] > xi[-1]:
            continue
        amin = np.vectorize(pc.alpha_min)(sol, xi, cp, cm, True)
        amax = np.vectorize(pc.alpha_max)(sol, xi, cp, cm, True)
        ax.fill_between(xi, amin, amax, ec=None, fc=color_dict[sol], rasterized=True)
        # boundary of the region, including vertical edges at the endpoints
        # of the xi_w range (unless they lie at the boundary of the plot)
        ax.plot(xi, amax, c="k")
        ax.plot(xi, amin, c="k")
        if xi[0] > ximin:
            ax.plot([xi[0], xi[0]], [amin[0], amax[0]], c="k")
        if xi[-1] < ximax:
            ax.plot([xi[-1], xi[-1]], [amin[-1], amax[-1]], c="k")

In [ ]:
with figure_style("paper"):
    fig, axs = plt.subplots(
        2, 2, figsize=(6.052269601100412, 7.102269601100412), sharex=False, sharey=True
    )
    fig.subplots_adjust(
        left=0.09913636363636365,
        right=0.9669545454545454,
        bottom=0.1971200867653752,
        top=0.964799984506183,
        wspace=0.12115657028580952,
        hspace=0.20192761714301588,
    )

    # c_- > c_+ > c_-^2
    make_plot_xiw_alphaN(axs[0, 0], 0.45, 0.6)
    # c_+ > c_- > c_+^2
    make_plot_xiw_alphaN(axs[0, 1], 0.6, 0.45)
    # c_-^2 > c_+
    make_plot_xiw_alphaN(axs[1, 0], 0.4, 0.75)
    # c_+^2 > c_-
    make_plot_xiw_alphaN(axs[1, 1], 0.75, 0.4)

    for ax in axs[:, 0]:
        ax.set_ylabel("$\\alpha_N$")
    for ax in axs[1]:
        ax.set_xlabel("$\\xi_w$")

    fig.legend(
        handles=[mpl.patches.Patch(color=color_dict[l], label=label_dict[l]) for l in color_dict.keys()]
        + [
            mpl.lines.Line2D([], [], c="k", ls=":", label="$c_+$"),
            mpl.lines.Line2D([], [], c="k", ls="--", label="$c_-$"),
        ],
        loc="lower center",
        bbox_to_anchor=[0.5, 0.0],
        ncols=3,
        frameon=False,
        handlelength=1.5,
    )

    if save_figures:
        plt.savefig(save_prefix + "xi_w_alpha_TN")

    plt.show()
    plt.close()

## Figure 6

Representative hydrodynamic profiles for $c_+=0.75$ and $c_-=0.40$. The top table shows the
velocity profiles $v(\xi)$ for the different direct, inverse, and mixed branches. The bottom table
shows the corresponding entropy and enthalpy ratios, $s/s_N$ (dashed) and $w/w_N$ (solid). Vertical
lines mark $\xi_w$ (red dashed), $c_+$ (dark-grey short-dashed), $c_-$ (light-grey short-dashed),
and, when present, $\xi_\mathrm{sh}$ (black dotted). The value of $\Psi_N$ has been chosen so that
all displayed solutions are entropic, namely, such that the entropy-current flux increases across
the wall.

We pick by hand one point $(\xi_w,\alpha_N)$ inside the existence region of each
solution type for $(c_+,c_-)=(0.75,0.40)$ and compute the corresponding profile with
`hydrodex.FluidProfile`. The same profiles are used for both parts of Figure 6. The
panels are arranged in a $3\times4$ grid: direct solutions in the first row, inverse
solutions in the second, mixed hybrids in the third, followed by the legend and the
phase plane of Figure 5 with the chosen points (top part) or the values of $\Psi_N$
(bottom part).

In [ ]:
C_PLUS_REP, C_MINUS_REP = 0.75, 0.40

# (xi_w, alpha_N) and panel title, in the order of the panels
REPRESENTATIVE_POINTS = {
    ST.Deflagration: (0.35, +0.50, r"direct deflagration"),
    ST.Hybrid: (0.7, +0.8, r"direct hybrid, $v_+ < c_-$"),
    ST.HybridPlus: (0.68, +0.45, r"direct hybrid, $v_+ > c_-$"),
    ST.Detonation: (0.95, +0.7, r"direct detonation"),
    ST.InverseDetonation: (0.16, -0.45, r"inverse detonation, $-$ branch"),
    ST.InverseDetonationPlus: (0.38, -0.10, r"inverse detonation, $+$ branch"),
    ST.InverseHybrid: (0.48, -0.2, r"inverse hybrid"),
    ST.InverseDeflagration: (0.90, -0.45, r"inverse deflagration"),
    ST.MixedHybridMinus: (0.45, -0.05, r"mixed hybrid, $v_- = c_-$"),
    ST.MixedHybridPlus: (0.65, +0.05, r"mixed hybrid, $v_+ = c_+$"),
}

representative_profiles = {
    sol: hydrodex.FluidProfile(xi_w, alpha_N, C_PLUS_REP, C_MINUS_REP)
    for sol, (xi_w, alpha_N, _) in REPRESENTATIVE_POINTS.items()
}


def shock_position(profile):
    """Position xi_sh of the shock front of a FluidProfile (NaN without shock)."""
    if not profile.has_shock():
        return np.nan
    xi_sh, _ = pc.get_shock_front(
        profile.get_solution_type(), profile.get_xi_coordinate(), profile.get_fluid_velocity()
    )
    return xi_sh


def parameter_box_text(alpha_N, xi_w, xi_sh):
    """Aligned list of alpha_N, xi_w and (if finite) xi_sh for the panel boxes."""
    rows = [rf"\alpha_N &=& {alpha_N:+.3g}", rf"\xi_w &=& {xi_w:.3g}"]
    if np.isfinite(xi_sh):
        rows.append(rf"\xi_{{\mathrm{{sh}}}} &=& {xi_sh:.3g}")
    body = r" \\ ".join(rows)
    return rf"$\begin{{array}}{{rcl}}{body}\end{{array}}$"


def draw_representative_panels(axes, curves, ylims, anchor, legend_handles, legend_fontsize):
    """Panels of Figure 6 for the representative profiles.

    curves(ax, sol, profile) draws the curves of one panel; ylims[row] is the
    y-range of each row and anchor(panel_idx) the (x, y, ha, va) of the parameter box.
    The 11th panel shows the legend; the 12th panel is left to the caller.
    """
    for panel_idx, (ax, sol) in enumerate(zip(axes, REPRESENTATIVE_POINTS)):
        row, col = divmod(panel_idx, 4)
        profile = representative_profiles[sol]
        xi_w = profile.get_wall_velocity()
        alpha_N, title = REPRESENTATIVE_POINTS[sol][1:]
        xi_sh = shock_position(profile)
        ax.tick_params(axis="both", which="major", labelsize=18)
        if col != 0:
            ax.tick_params(axis="y", which="both", left=False, labelleft=False)
        ax.set_xlim(0.0, 1.0)
        ax.set_ylim(*ylims[row])
        ax.grid(True, alpha=0.18)
        ax.axvline(C_PLUS_REP, color="0.55", ls="--", lw=1.2, zorder=1)
        ax.axvline(C_MINUS_REP, color="0.75", ls="--", lw=1.2, zorder=1)
        curves(ax, sol, profile)
        ax.axvline(xi_w, color="red", ls="--", lw=1.8, zorder=2)
        if np.isfinite(xi_sh):
            ax.axvline(xi_sh, color="black", ls=":", lw=1.6, zorder=2)
        ax.set_title(title, fontsize=16)
        x, y, ha, va = anchor(panel_idx)
        ax.text(
            x, y, parameter_box_text(alpha_N, xi_w, xi_sh),
            transform=ax.transAxes, ha=ha, va=va, fontsize=18,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.45", alpha=0.92),
        )
    ax = axes[10]
    ax.tick_params(axis="both", which="major", labelsize=18)
    ax.tick_params(axis="y", which="both", left=False, labelleft=False)
    ax.axis("off")
    ax.legend(
        handles=legend_handles, loc="center", ncol=1, frameon=False,
        fontsize=legend_fontsize, labelspacing=0.65, handlelength=3.0,
        handletextpad=1.0, borderaxespad=0.0,
    )
    axes[11].tick_params(axis="both", which="major", labelsize=18)
    axes[11].tick_params(axis="y", which="both", left=False, labelleft=False)


REFERENCE_LINE_HANDLES = [
    Line2D([0], [0], color="red", ls="--", lw=1.8, label=r"$\xi_w$"),
    Line2D([0], [0], color="0.55", ls="--", lw=1.2, label=r"$c_+$"),
    Line2D([0], [0], color="0.75", ls="--", lw=1.2, label=r"$c_-$"),
    Line2D([0], [0], color="black", ls=":", lw=1.6, label=r"$\xi_{\mathrm{sh}}$"),
]

Velocity profiles $v(\xi)$, marking $\xi_w$, $c_\pm$ and, where present, the shock
front; the last panel shows the chosen points in the phase plane of Figure 5
(`representative_profiles.pdf`).

In [ ]:
with figure_style("profile"):
    fig, axes = plt.subplots(3, 4, figsize=(16.5, 10.0), sharex=True, sharey=False)
    axes = axes.ravel()

    def velocity_curve(ax, sol, profile):
        ax.plot(profile.get_xi_coordinate(), profile.get_fluid_velocity(), color=color_dict[sol], lw=2.2, zorder=3)

    def anchor(panel_idx):
        row, col = divmod(panel_idx, 4)
        if row == 1:
            return (0.97, 0.03, "right", "bottom") if col < 3 else (0.03, 0.03, "left", "bottom")
        return 0.03, 0.97, "left", "top"

    draw_representative_panels(
        axes,
        velocity_curve,
        ylims=[(-0.05, 0.50), (-0.50, 0.05), (-0.2, 0.15)],
        anchor=anchor,
        legend_handles=[Line2D([0], [0], color="black", lw=2.2, label=r"$v(\xi)$")]
        + REFERENCE_LINE_HANDLES,
        legend_fontsize=26,
    )

    # last panel: phase plane of Figure 5 with the chosen points
    ax = axes[11]
    make_plot_xiw_alphaN(ax, C_PLUS_REP, C_MINUS_REP, title=False, xi_floor=0.0)
    for sol, profile in representative_profiles.items():
        ax.plot(
            profile.get_wall_velocity(), REPRESENTATIVE_POINTS[sol][1], marker="o", ms=5.0,
            color=color_dict[sol], mec="black", mew=0.5, zorder=5,
        )
    ax.tick_params(axis="both", which="major", labelsize=18)
    ax.tick_params(axis="y", which="both", left=True, labelleft=True)
    ax.set_xlabel(r"$\xi_w$", fontsize=18)
    ax.set_ylabel(r"$\alpha_N$", fontsize=18)

    for j in [8, 9, 11]:
        axes[j].set_xlabel(r"$\xi$", fontsize=18)
    for j in [0, 4, 8]:
        axes[j].set_ylabel(r"$v(\xi)$", fontsize=18)
    fig.suptitle(
        rf"Hydrodynamic profiles for $c_+={C_PLUS_REP:.2f}$, $c_-={C_MINUS_REP:.2f}$",
        fontsize=20,
        y=0.985,
    )
    fig.subplots_adjust(left=0.07, right=0.99, bottom=0.07, top=0.93, wspace=0.1, hspace=0.16)

    if save_figures:
        fig.savefig(save_prefix + "representative_profiles.pdf", bbox_inches="tight", facecolor="white")

    plt.show()

Entropy and enthalpy ratios $s/s_N$ and $w/w_N$ for the same points and layout. The
entropy density requires the ratio $\Psi_N=w_-(T_N)/w_+(T_N)$, which is not fixed by
$(\xi_w,\alpha_N)$; we use $\Psi_N=1/1.1$ for $\alpha_N>0$ and $\Psi_N=1/0.1$ for
$\alpha_N<0$, as stated in the last panel of the figure
(`representative_profiles_entropy_enthalpy.pdf`). The y-range of each row covers all
the curves of the row, with a margin of 5%.

In [ ]:
PSI_N_DIRECT, PSI_N_INVERSE = 1.0 / 1.1, 1.0 / 0.1


def enthalpy_entropy_ratios(sol):
    """w/w_N and s/s_N of a representative profile."""
    profile = representative_profiles[sol]
    alpha_N = REPRESENTATIVE_POINTS[sol][1]
    w = np.asarray(profile.get_enthalpy_density())
    s = np.asarray(profile.get_entropy_density(PSI_N_DIRECT if alpha_N > 0 else PSI_N_INVERSE, 1.0))
    return w / profile.get_mean_enthalpy_density(), s / s[-1]


with figure_style("profile"):
    fig, axes = plt.subplots(3, 4, figsize=(16.5, 10.0), sharex=True, sharey=False)
    axes = axes.ravel()
    ratios = {sol: enthalpy_entropy_ratios(sol) for sol in REPRESENTATIVE_POINTS}

    ylims = []
    for row_sols in np.split(np.array(list(REPRESENTATIVE_POINTS), dtype=object), [4, 8]):
        y = np.concatenate([np.concatenate(ratios[sol]) for sol in row_sols])
        pad = 0.05 * (y.max() - y.min()) + 1e-6
        ylims.append((y.min() - pad, y.max() + pad))

    def ratio_curves(ax, sol, profile):
        w_ratio, s_ratio = ratios[sol]
        ax.plot(profile.get_xi_coordinate(), s_ratio, color=color_dict[sol], lw=2.0, ls="--", zorder=3)
        ax.plot(profile.get_xi_coordinate(), w_ratio, color=color_dict[sol], lw=2.2, ls="-", zorder=3)

    def anchor(panel_idx):
        return (0.97, 0.97, "right", "top") if panel_idx // 4 == 1 else (0.03, 0.97, "left", "top")

    draw_representative_panels(
        axes,
        ratio_curves,
        ylims=ylims,
        anchor=anchor,
        legend_handles=[
            Line2D([0], [0], color="black", ls="--", lw=2.0, label=r"$s/s_N$"),
            Line2D([0], [0], color="black", ls="-", lw=2.2, label=r"$w/w_N$"),
        ]
        + REFERENCE_LINE_HANDLES,
        legend_fontsize=22,
    )

    # last panel: values of Psi_N (several lines, since mathtext has no array)
    axes[11].axis("off")
    for i, text in enumerate([
        r"$\Psi_N = \displaystyle\frac{(1+c_-^2)a_-}{(1+c_+^2)a_+}$",
        r"$ $",
        rf"$\alpha_N>0 \Rightarrow \Psi_N = {PSI_N_DIRECT:.2g}$",
        rf"$\alpha_N<0 \Rightarrow \Psi_N = {PSI_N_INVERSE:.2g}$",
    ]):
        axes[11].text(
            0.5, 0.70 - i * 0.15, text, transform=axes[11].transAxes,
            ha="center", va="center", fontsize=26,
        )

    for j in [8, 9]:
        axes[j].set_xlabel(r"$\xi$", fontsize=18)
    for j in [0, 4, 8]:
        axes[j].set_ylabel(r"$s/s_N$, $w/w_N$", fontsize=18)
    fig.suptitle(
        rf"Entropy and enthalpy ratios for $c_+={C_PLUS_REP:.2f}$, $c_-={C_MINUS_REP:.2f}$",
        fontsize=20,
        y=0.985,
    )
    fig.subplots_adjust(left=0.07, right=0.99, bottom=0.07, top=0.93, wspace=0.1, hspace=0.16)

    if save_figures:
        fig.savefig(
            save_prefix + "representative_profiles_entropy_enthalpy.pdf",
            bbox_inches="tight",
            facecolor="white",
        )

    plt.show()

### Letter, Figure 1

**Hydrodynamic profiles beyond the bag model.** Left: representative fluid velocity profiles
$v(\xi)$ in the bubble-center frame for the different stationary branches: direct detonations,
direct hybrids, direct deflagrations, inverse detonations, inverse hybrids, inverse deflagrations,
mixed hybrids with $v_-=c_-$, and mixed hybrids with $v_+=c_+$. Right: the corresponding domains
in the $(\xi_w,\alpha_N)$ plane for $c_+=0.75$ and $c_-=0.40$, using the same color coding.
Circles mark the parameters of the profiles displayed in the left panel.

Eight representative profiles for $(c_+,c_-)=(0.75,0.40)$ (the points of Figure 6, with
a different inverse-hybrid point) and the phase-plane regions, merging the two
sub-branches of the direct hybrids and of the inverse detonations into one region each
(`representative_overlay_phase_plane.pdf`). The profiles of the mixed hybrids are drawn
with thicker lines.

In [ ]:
OVERLAY_POINTS = {
    ST.Deflagration: (0.35, +0.50),
    ST.Hybrid: (0.70, +0.80),
    ST.Detonation: (0.95, +0.70),
    ST.InverseDetonation: (0.16, -0.45),
    ST.InverseHybrid: (0.30, -0.38),
    ST.InverseDeflagration: (0.90, -0.45),
    ST.MixedHybridMinus: (0.45, -0.05),
    ST.MixedHybridPlus: (0.65, +0.05),
}
MIXED_HYBRIDS = (ST.MixedHybridMinus, ST.MixedHybridPlus)
LW_PROFILE, LW_PROFILE_MIXED = 1.15, 4

# regions of the phase plane: merged branches and the colour of the region
MERGED_REGIONS = [
    ((ST.Detonation,), ST.Detonation),
    ((ST.Deflagration,), ST.Deflagration),
    ((ST.Hybrid, ST.HybridPlus), ST.Hybrid),
    ((ST.InverseDetonation, ST.InverseDetonationPlus), ST.InverseDetonation),
    ((ST.InverseDeflagration,), ST.InverseDeflagration),
    ((ST.InverseHybrid,), ST.InverseHybrid),
    ((ST.MixedHybridPlus,), ST.MixedHybridPlus),
    ((ST.MixedHybridMinus,), ST.MixedHybridMinus),
]


def add_merged_region(ax, sols, color, cp, cm):
    """Fill the union of the adjacent regions `sols` of Figure 5 with one colour and
    outline only its outer boundary."""
    ranges = [
        (max(3e-4, pc.xi_min(s, cp, cm) + ximin), pc.xi_max(s, cp, cm) - ximin) for s in sols
    ]
    ranges = [(a, b) for a, b in ranges if a < b]
    if not ranges:
        return
    lo = min(a for a, _ in ranges)
    hi = max(b for _, b in ranges)
    xi = np.unique(
        np.concatenate([np.linspace(lo, hi, N_points)] + [np.linspace(a, b, N_points) for a, b in ranges])
    )
    amin = np.full_like(xi, np.inf)
    amax = np.full_like(xi, -np.inf)
    for s in sols:
        a = max(3e-4, pc.xi_min(s, cp, cm) + ximin)
        b = pc.xi_max(s, cp, cm) - ximin
        m = (xi >= a) & (xi <= b)
        if m.any():
            amin[m] = np.minimum(amin[m], np.vectorize(pc.alpha_min)(s, xi[m], cp, cm, True))
            amax[m] = np.maximum(amax[m], np.vectorize(pc.alpha_max)(s, xi[m], cp, cm, True))
    ok = np.isfinite(amin) & np.isfinite(amax)
    xi, amin, amax = xi[ok], amin[ok], amax[ok]
    ax.fill_between(xi, amin, amax, ec=None, fc=color, rasterized=True)
    ax.plot(xi, amax, c="k")
    ax.plot(xi, amin, c="k")
    if xi[0] > ximin:
        ax.plot([xi[0], xi[0]], [amin[0], amax[0]], c="k")
    if xi[-1] < ximax:
        ax.plot([xi[-1], xi[-1]], [amin[-1], amax[-1]], c="k")


with figure_style("profile"):
    PHI = 0.5 * (1.0 + np.sqrt(5.0))
    fig, (ax_v, ax_pp) = plt.subplots(
        1, 2, figsize=(14.0, 14.0 / PHI**2), gridspec_kw={"width_ratios": [1.0, 1.0], "wspace": 0.22}
    )

    # left: representative v(xi)
    ax_v.axvline(C_PLUS_REP, color="0.55", ls="--", lw=1.2, zorder=1)
    ax_v.axvline(C_MINUS_REP, color="0.75", ls="--", lw=1.2, zorder=1)
    for sol, (xi_w, alpha_N) in OVERLAY_POINTS.items():
        profile = hydrodex.FluidProfile(xi_w, alpha_N, C_PLUS_REP, C_MINUS_REP)
        mixed = sol in MIXED_HYBRIDS
        ax_v.plot(
            profile.get_xi_coordinate(), profile.get_fluid_velocity(), color=merged_color_dict[sol],
            lw=LW_PROFILE_MIXED if mixed else LW_PROFILE, zorder=4 if mixed else 3,
        )
    ax_v.set_xlim(0.0, 1.0)
    ax_v.set_ylim(-0.65, 0.65)
    ax_v.set_xlabel(r"$\xi$", fontsize=24)
    ax_v.set_ylabel(r"$v(\xi)$", fontsize=24)
    ax_v.grid(True, alpha=0.18)
    ax_v.tick_params(axis="both", which="major", labelsize=22)
    legend_labels = {
        ST.Detonation: "direct detonation",
        ST.Hybrid: "direct hybrid",
        ST.Deflagration: "direct deflagration",
        ST.InverseDetonation: "inverse detonation",
        ST.InverseHybrid: "inverse hybrid",
        ST.InverseDeflagration: "inverse deflagration",
        ST.MixedHybridMinus: "mixed, $v_- = c_-$",
        ST.MixedHybridPlus: "mixed, $v_+ = c_+$",
    }
    ax_v.legend(
        handles=[
            Line2D([0], [0], color=merged_color_dict[sol], label=label,
                   lw=LW_PROFILE_MIXED if sol in MIXED_HYBRIDS else LW_PROFILE)
            for sol, label in legend_labels.items()
        ],
        loc="upper left", frameon=False, fontsize=11, handlelength=2.2, ncol=2,
    )

    # right: merged phase plane with the points of the profiles
    ax_pp.set_xlim(0.0, 1.0)
    ax_pp.set_ylim(-1.0, 1.0)
    ax_pp.set_xticks(
        [0.0, C_MINUS_REP, C_PLUS_REP, 1.0], labels=[r"$0$", r"$c_-$", r"$c_+$", r"$1$"]
    )
    ax_pp.axhline(0.0, c="k")
    ax_pp.axvline(C_MINUS_REP, c="k", ls="--")
    ax_pp.axvline(C_PLUS_REP, c="k", ls=":")
    for sols, key in MERGED_REGIONS:
        add_merged_region(ax_pp, sols, merged_color_dict[key], C_PLUS_REP, C_MINUS_REP)
    for xi_w, alpha_N in OVERLAY_POINTS.values():
        ax_pp.plot(
            xi_w, alpha_N, marker="o", ms=10.0, mew=2.0, fillstyle="none",
            color="black", mec="black", zorder=5,
        )
    ax_pp.tick_params(axis="both", which="major", labelsize=22)
    ax_pp.set_xlabel(r"$\xi_w$", fontsize=24)
    ax_pp.set_ylabel(r"$\alpha_N$", fontsize=24)

    fig.subplots_adjust(left=0.07, right=0.95, bottom=0.12, top=0.90, wspace=0.22)

    if save_figures:
        fig.savefig(
            save_prefix + "representative_overlay_phase_plane.pdf",
            bbox_inches="tight", facecolor="white", dpi=300,
        )

    plt.show()

## Figure 7

Evolution of the wall-frame fluid velocities $v_-$ and $v_+$ (black and blue lines,
respectively), the shock front position $\xi_\mathrm{sh}$ (red), and the ratio $\alpha_+/\alpha_N$
(orange) as a function of $\xi_w$, for fixed $\alpha_N=0.02$ [top] and $\alpha_N=-0.02$ [bottom]
and for $(c_+,c_-)=(0.72,0.45)$. The colour shading of the background indicates the corresponding
hydrodynamic branch. The plot illustrates that the local strength $\alpha_+$ can differ
significantly from the physical strength $\alpha_N$, and can even have the opposite sign.

Scan of the wall velocity $\xi_w$ at fixed $\alpha_N=\pm0.02$ for
$(c_+,c_-)=(0.72,0.45)$. For each $\xi_w$ the profile is computed with
`FluidProfile`, and we record the wall-frame velocities $v_\pm$ (from
`pc.find_alpha_plus` and `pc.compute_fluid_velocities`), the shock position
$\xi_\text{sh}$ (from `pc.get_shock_front`) and the ratio $\alpha_+/\alpha_N$.
At $|\alpha_N|=0.02$ all shocks are weak: the fluid velocity behind the shock
vanishes and $\xi_\text{sh}\to c_+$ (direct) or $c_-$ (inverse). The profiles are
therefore integrated with `ProfileSolveMode.D_V_D_XI_FOR_SHOCKS`, i.e. as $v(\xi)$ up
to the shock front: in the default $\xi(v)$ integration, the last step towards
$v=0$ crosses the pole of $d\xi/dv$ and can displace $\xi_\text{sh}$, which gave
isolated spikes. For $\xi_w<c_-$ the integration step is reduced by a factor of ten
(`solutions_evolution.pdf`).

In [ ]:
C_PLUS_FIG7, C_MINUS_FIG7 = 0.72, 0.45
ALPHA_N_FIG7 = [0.02, -0.02]
XI_W_FIG7 = np.linspace(0.005, 0.999, 300)

# v(xi) up to the shock front, with a ten times smaller step for xi_w < c_-
SOLVE_MODE_FIG7 = pc.ProfileSolveMode.D_V_D_XI_FOR_SHOCKS
STEP_FIG7, STEP_FINE_FIG7 = 1e-5, 1e-6


def scan_fixed_alpha_N(alpha_N):
    """v_+, v_-, xi_sh, alpha_+/alpha_N and the solution type along XI_W_FIG7 at fixed
    alpha_N (xi_sh is NaN for profiles without shock front)."""
    scan = {key: np.full(XI_W_FIG7.size, np.nan) for key in ("v_plus", "v_minus", "xi_sh", "ratio")}
    scan["sol_type"] = []
    for k, xi_w in enumerate(XI_W_FIG7):
        step = STEP_FINE_FIG7 if xi_w < C_MINUS_FIG7 else STEP_FIG7
        profile = hydrodex.FluidProfile(
            xi_w, alpha_N, C_PLUS_FIG7, C_MINUS_FIG7, step, SOLVE_MODE_FIG7
        )
        sol = profile.get_solution_type()
        alpha_plus = pc.find_alpha_plus(
            sol, alpha_N, xi_w, C_PLUS_FIG7, C_MINUS_FIG7, step, SOLVE_MODE_FIG7
        )
        scan["v_plus"][k], scan["v_minus"][k] = pc.compute_fluid_velocities(
            xi_w, alpha_plus, sol, C_PLUS_FIG7, C_MINUS_FIG7
        )
        scan["ratio"][k] = alpha_plus / alpha_N
        scan["xi_sh"][k] = shock_position(profile)
        scan["sol_type"].append(sol)
    return scan


fig07_scans = cached_scan(
    "fig07_xi_scans", lambda: [scan_fixed_alpha_N(alpha_N) for alpha_N in ALPHA_N_FIG7]
)

Plot. Where $\alpha_+/\alpha_N$ leaves the range of the main panel, an extra panel above
or below (with a broken axis) shows its extreme values.

In [ ]:
Y_MAIN_MIN, Y_MAIN_MAX = -0.1, 1.1  # y-range of the main panels
BAND_ALPHA = 0.35
LEGEND_FRAME_KW = dict(
    frameon=True, fancybox=True, facecolor="white", edgecolor="0.35", framealpha=0.88
)


def ratio_break(ratio):
    """Side ("upper"/"lower") and y-limits of the extra panel for the extreme value of
    alpha_+/alpha_N outside the main y-range, or None."""
    r = ratio[np.isfinite(ratio)]
    above = r.max() - Y_MAIN_MAX
    below = Y_MAIN_MIN - r.min()
    if max(above, below) <= 0.0:
        return None
    extreme = r.max() if above >= below else r.min()
    return ("upper" if above >= below else "lower"), sorted([0.9 * extreme, 1.1 * extreme])


def plot_scan(ax, scan, legends):
    """Solution-type bands, branch changes and the curves of one scan."""
    sol = scan["sol_type"]
    starts = [0] + [k for k in range(1, len(sol)) if sol[k] != sol[k - 1]]
    ends = starts[1:] + [len(sol)]
    for i0, i1 in zip(starts, ends):
        ax.axvspan(XI_W_FIG7[i0], XI_W_FIG7[i1 - 1], facecolor=color_dict[sol[i0]],
                   edgecolor="none", alpha=BAND_ALPHA, zorder=0)
    for k in starts[1:]:
        ax.axvline(0.5 * (XI_W_FIG7[k - 1] + XI_W_FIG7[k]), color="0.35", lw=1.6,
                   ls=(0, (5, 4)), zorder=2, alpha=0.95)
    for y in (0.0, 1.0):
        ax.axhline(y, color="0.45", lw=1.0, ls=(0, (4, 3)), zorder=1.2, alpha=0.85)
    ax.plot(XI_W_FIG7, scan["v_minus"], color="black", lw=1.8, label=r"$v_-$", zorder=3)
    ax.plot(XI_W_FIG7, scan["v_plus"], color="tab:blue", lw=1.8, label=r"$v_+$", zorder=3)
    ax.plot(XI_W_FIG7, scan["xi_sh"], color="tab:red", lw=1.6,
            label=r"$\xi_{\mathrm{sh}}$", zorder=3)
    ax.plot(XI_W_FIG7, scan["ratio"], color="tab:orange", lw=1.6,
            label=r"$\alpha_+/\alpha_N$", zorder=3)
    ax.set_xlim(0.0, 1.0)
    ax.grid(False)
    if legends:
        ax.add_artist(ax.legend(loc="upper left", fontsize=11, **LEGEND_FRAME_KW))
        ax.legend(
            handles=[
                Patch(facecolor=color_dict[s], edgecolor="0.45", linewidth=0.4,
                      alpha=BAND_ALPHA, label=label_dict[s])
                for s in color_dict if s in sol
            ],
            loc="lower right", bbox_to_anchor=(1.0, 0.0), fontsize=10,
            title="solution type", title_fontsize=11, **LEGEND_FRAME_KW,
        )


def add_break_marks(ax_upper, ax_lower, angle_deg=30.0, length_px=16.0):
    """Hide the touching spines and draw one slash at each end of the break, with the
    same angle on screen in both panels."""
    ax_upper.spines["bottom"].set_visible(False)
    ax_lower.spines["top"].set_visible(False)
    ax_upper.tick_params(axis="x", which="both", bottom=False, labelbottom=False)
    ax_lower.tick_params(axis="x", which="both", top=False)
    dx = 0.5 * length_px * np.cos(np.deg2rad(angle_deg))
    dy = 0.5 * length_px * np.sin(np.deg2rad(angle_deg))
    for ax, y in ((ax_upper, 0.0), (ax_lower, 1.0)):
        for x in (0.0, 1.0):
            x_disp, y_disp = ax.transAxes.transform((x, y))
            (x0, y0), (x1, y1) = ax.transAxes.inverted().transform(
                [(x_disp - dx, y_disp - dy), (x_disp + dx, y_disp + dy)]
            )
            ax.plot([x0, x1], [y0, y1], transform=ax.transAxes, color="black",
                    lw=1.2, clip_on=False, solid_capstyle="butt", zorder=20)


with figure_style("profile"):
    fig = plt.figure(figsize=(12, 9.6))
    outer = fig.add_gridspec(nrows=len(ALPHA_N_FIG7), ncols=1, hspace=0.32)

    for row, (scan, alpha_N) in enumerate(zip(fig07_scans, ALPHA_N_FIG7)):
        br = ratio_break(scan["ratio"])
        if br is None:
            ax_main = ax_upper = ax_lower = fig.add_subplot(outer[row, 0])
        else:
            side, ylim_extreme = br
            if side == "upper":
                inner = outer[row, 0].subgridspec(2, 1, height_ratios=[1.0, 3.0], hspace=0.025)
                ax_upper = ax_extreme = fig.add_subplot(inner[0, 0])
                ax_lower = ax_main = fig.add_subplot(inner[1, 0], sharex=ax_extreme)
            else:
                inner = outer[row, 0].subgridspec(2, 1, height_ratios=[3.0, 1.0], hspace=0.025)
                ax_upper = ax_main = fig.add_subplot(inner[0, 0])
                ax_lower = ax_extreme = fig.add_subplot(inner[1, 0], sharex=ax_main)
            plot_scan(ax_extreme, scan, legends=False)
            ax_extreme.set_ylim(*ylim_extreme)
            add_break_marks(ax_upper, ax_lower)
        plot_scan(ax_main, scan, legends=True)
        ax_main.set_ylim(Y_MAIN_MIN, Y_MAIN_MAX)
        ax_main.set_ylabel("dimensionless")
        ax_upper.set_title(rf"$\alpha_N = {alpha_N:g}$")
        ax_lower.set_xlabel(r"$\xi_w$")

    with quiet(TIGHT_LAYOUT_WARNING):
        fig.tight_layout()

    if save_figures:
        fig.savefig(save_prefix + "solutions_evolution.pdf", bbox_inches="tight")

    plt.show()

## Figure 8

Projected maps $\alpha_+\mapsto\alpha_N$ for $c_+=0.9$ and $c_-=0.16$ with internal minima on the
inverse detonation (upper left) and mixed hybrid with $v_-=c_-$ branch (upper right), as well as
direct hybrids (lower panels). Arrows indicate increasing $\alpha_+$, black dots mark branch
endpoints, and the faded curves in the upper panels show larger values of $c_-$ at fixed
$c_-/\xi_w$.

Figures 8–11 use the projection $\alpha_+\mapsto\alpha_N$ of a given branch at fixed
wall velocity, through the enthalpy ratio $r_w=w_+/w_N$ [eq. (3.11) of the paper]:
`alpha_N_map` samples $\alpha_+$ over the local existence interval of the branch
(`pc.alpha_min`, `pc.alpha_max` with `return_alpha_N=False`), slightly shortened at
both ends, and projects each point with `pc.compute_alpha_N`. The points are evenly
spaced or, with `clustered=True`, accumulate geometrically towards both ends of the
interval.

For $c_+=0.9$ and $c_-=0.16$, Figure 8 shows the map for four wall velocities; in the
upper row, the faded curves show $c_-\in\{0.18,0.20,0.25\}$ at fixed $c_-/\xi_w$,
i.e. for $\xi_w\to\xi_w\,c_-/0.16$.

In [ ]:
def branch_exists(sol, xi_w, cp, cm):
    """True if solutions of type sol exist at the wall velocity xi_w."""
    return pc.xi_min(sol, cp, cm) < xi_w < pc.xi_max(sol, cp, cm)


def clustered_unit_grid(n):
    """n points in [0, 1] (n >= 9), linear plus geometric accumulation at both ends."""
    u_left = 0.5 * np.geomspace(1e-6, 1.0, max(10, n // 4))
    u = np.unique(np.concatenate([np.linspace(0.0, 1.0, n), u_left, 1.0 - u_left[::-1]]))
    return u[(u >= 0.0) & (u <= 1.0)]


def projected_alpha_N(sol, alpha_plus, xi_w, cp, cm):
    """pc.compute_alpha_N, NaN where the branch cannot be identified (close to the ends of
    its alpha_+ interval)."""
    try:
        return pc.compute_alpha_N(sol, alpha_plus, xi_w, cp, cm, step_size=ALPHA_N_STEP)
    except (RuntimeError, ValueError):
        return np.nan


def alpha_N_map(sol, xi_w, cp, cm, n, edge=1e-8, clustered=False):
    """Projection alpha_+ -> alpha_N of branch sol at fixed xi_w.

    alpha_+ runs over n points of the local interval [alpha_min, alpha_max],
    shortened at both ends by edge * max(1, |alpha_min|, |alpha_max|, width).
    Returns the arrays (alpha_+, alpha_N), empty if nothing is left of the interval.
    """
    a0 = pc.alpha_min(sol, xi_w, cp, cm, False)
    a1 = pc.alpha_max(sol, xi_w, cp, cm, False)
    lo, hi = min(a0, a1), max(a0, a1)
    cut = edge * max(1.0, abs(lo), abs(hi), hi - lo)
    lo, hi = lo + cut, hi - cut
    if not lo < hi:
        return np.array([]), np.array([])
    ap = lo + (hi - lo) * clustered_unit_grid(n) if clustered else np.linspace(lo, hi, n)
    aN = np.array([projected_alpha_N(sol, a, xi_w, cp, cm) for a in ap])
    return ap, aN

In [ ]:
C_PLUS_FIG8, C_MINUS_FIG8 = 0.90, 0.16
XI_W_FIG8 = [0.10, 0.25, 0.85, 0.95]  # panels: upper left, upper right, lower left, lower right
C_MINUS_FADED = [0.18, 0.20, 0.25]  # faded curves, upper row only
N_ALPHA_FIG8 = 1200


def scaled_xi_w(cm):
    """Wall velocities of the panels at fixed c_-/xi_w."""
    if cm == C_MINUS_FIG8:
        return XI_W_FIG8
    return [min(xi_w * cm / C_MINUS_FIG8, 0.999999) for xi_w in XI_W_FIG8]


def branch_maps(cm):
    """{xi_w: {sol: (alpha_+, alpha_N)}} for the branches existing at each xi_w."""
    maps = {}
    for xi_w in scaled_xi_w(cm):
        maps[xi_w] = {}
        for sol in color_dict:
            if branch_exists(sol, xi_w, C_PLUS_FIG8, cm):
                ap, aN = alpha_N_map(sol, xi_w, C_PLUS_FIG8, cm, N_ALPHA_FIG8)
                if np.count_nonzero(np.isfinite(aN)) >= 5:
                    maps[xi_w][sol] = (ap, aN)
    return maps


fig08_maps = cached_scan(
    "fig08_branch_maps", lambda: {cm: branch_maps(cm) for cm in [C_MINUS_FIG8] + C_MINUS_FADED}
)

Plot (`alphaN_alpha_plus_folding_with_faded_cm_overlays_2x2.pdf`). The curves for
$c_-=0.16$ are drawn with arrows (direction of increasing $\alpha_+$) and endpoint
markers; coinciding endpoints of different branches are drawn once.

In [ ]:
PANEL_YLIMS_FIG8 = [(-1.0, 1.0), (-1.0, 1.5), (-10.0, 35.0), (-10.0, 125.0)]
FADED_ALPHA = {0.18: 0.36, 0.20: 0.22, 0.25: 0.12}  # opacity of the faded curves
CURVE_ALPHA_BASE = 0.96

# positions of the arrows along each curve (fractions of the visible part)
DEFAULT_ARROW_FRACS = (0.30, 0.58, 0.82)
ARROW_FRACS = {
    ST.Detonation: DEFAULT_ARROW_FRACS,
    ST.HybridPlus: (0.55,),
    ST.Hybrid: (0.25, 0.75),
    ST.Deflagration: (0.35, 0.75),
    ST.InverseDetonationPlus: (0.15, 0.30, 0.58, 0.82),
    ST.InverseDetonation: (0.75,),
    ST.InverseHybrid: DEFAULT_ARROW_FRACS,
    ST.InverseDeflagration: (0.75,),
    ST.MixedHybridPlus: (0.75,),
    ST.MixedHybridMinus: (0.15, 0.30, 0.58, 0.82),
}


def add_direction_arrows(ax, x, y, color, xlim, ylim, fracs, min_step=4):
    """Arrows along the longest visible part of the curve (increasing alpha_+)."""
    visible = (x >= xlim[0]) & (x <= xlim[1]) & (y >= ylim[0]) & (y <= ylim[1])
    idx = np.flatnonzero(visible)
    if len(idx) == 0:
        return
    block = max(np.split(idx, np.flatnonzero(np.diff(idx) > 1) + 1), key=len)
    if len(block) < 5:
        return
    for frac in fracs:
        j = int(frac * (len(block) - 1))
        i0 = block[max(0, j - min_step)]
        i1 = block[min(len(block) - 1, j + min_step)]
        ax.annotate(
            "",
            xy=(x[i1], y[i1]),
            xytext=(x[i0], y[i0]),
            arrowprops=dict(
                arrowstyle="-|>", color=color, lw=1.55, alpha=0.95,
                mutation_scale=16, shrinkA=0, shrinkB=0,
            ),
            zorder=9,
        )


def cluster_points(points, x_tol, y_tol):
    """Merge points closer than (x_tol, y_tol) to the mean of an existing cluster."""
    clusters = []
    for x, y in points:
        for cl in clusters:
            if abs(x - np.mean([p[0] for p in cl])) <= x_tol and abs(y - np.mean([p[1] for p in cl])) <= y_tol:
                cl.append((x, y))
                break
        else:
            clusters.append([(x, y)])
    return [(np.mean([p[0] for p in cl]), np.mean([p[1] for p in cl])) for cl in clusters]


with figure_style("profile", LARGE_FONT_RC):
    fig, axes = plt.subplots(
        2, 2, figsize=(16.0, 9.0), sharex=False, sharey=False,
        gridspec_kw={"wspace": 0.20, "hspace": 0.35},
    )
    axes = axes.ravel()
    base = fig08_maps[C_MINUS_FIG8]

    for panel_idx, (ax, xi_ref, ylim) in enumerate(zip(axes, XI_W_FIG8, PANEL_YLIMS_FIG8)):
        # x range from the base family
        all_ap = np.concatenate([ap for ap, _ in base[xi_ref].values()])
        pad = 0.06 * max(1.0e-8, all_ap.max() - all_ap.min())
        xlim = (all_ap.min() - pad, all_ap.max() + pad)
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)

        # faded curves (upper row)
        if panel_idx < 2:
            for cm, alpha in FADED_ALPHA.items():
                for sol, (ap, aN) in fig08_maps[cm][scaled_xi_w(cm)[panel_idx]].items():
                    good = np.isfinite(aN)
                    ax.plot(ap[good], aN[good], lw=1.25, color=color_dict[sol], alpha=alpha,
                            solid_capstyle="round", zorder=1)

        # base curves, arrows and endpoints
        endpoints = []
        for sol, (ap, aN) in base[xi_ref].items():
            good = np.isfinite(aN)
            (line,) = ax.plot(
                ap[good], aN[good], lw=1.85, color=color_dict[sol], alpha=CURVE_ALPHA_BASE,
                label=short_label_dict[sol], solid_capstyle="round", zorder=6,
            )
            line.set_path_effects([pe.Stroke(linewidth=3.4, foreground="white", alpha=0.78), pe.Normal()])
            add_direction_arrows(ax, ap, aN, color_dict[sol], xlim, ylim, ARROW_FRACS[sol])
            endpoints += [(ap[i], aN[i]) for i in (0, -1) if np.isfinite(aN[i])]
        x_tol = 2.5e-3 * max(1.0, xlim[1] - xlim[0])
        y_tol = 2.5e-3 * max(1.0, ylim[1] - ylim[0])
        for x, y in cluster_points(endpoints, x_tol, y_tol):
            if xlim[0] <= x <= xlim[1] and ylim[0] <= y <= ylim[1]:
                ax.plot(x, y, marker="o", ms=5, color="k", mec="white", mew=0.55, zorder=12)

        ax.axhline(0.0, color="0.35", lw=1.0, alpha=0.8, zorder=0)
        ax.axvline(0.0, color="0.35", lw=1.0, alpha=0.8, zorder=0)
        ax.grid(True, alpha=0.25)
        ax.set_xlabel(r"$\alpha_+$")
        ax.set_title(rf"$c_+={C_PLUS_FIG8}$, $c_-={C_MINUS_FIG8}$, $\xi_w={xi_ref}$")

    axes[0].set_ylabel(r"$\alpha_N$")
    axes[2].set_ylabel(r"$\alpha_N$")

    fig.legend(
        handles=[
            Patch(facecolor=color, edgecolor="none", alpha=0.85, label=short_label_dict[sol])
            for sol, color in color_dict.items()
        ]
        + [
            Line2D([], [], color="0.25", lw=1.7, marker=r"$\rightarrow$", markersize=12,
                   label=r"arrows: increasing $\alpha_+$"),
            Line2D([], [], color="k", marker="o", linestyle="None", markersize=6,
                   label="branch endpoint"),
            Line2D([], [], color="0.35", lw=2.0, alpha=0.25,
                   label=r"faded overlays in the upper row: " "\n"
                   r"$c_- = 0.18, 0.20, 0.25$" r" at fixed $c_-/\xi_w$"),
        ],
        loc="lower center",
        bbox_to_anchor=(0.5, -0.12),
        ncols=4,
        frameon=False,
    )

    with quiet(TIGHT_LAYOUT_WARNING):
        plt.tight_layout(rect=[0.0, 0.15, 1.0, 1.0])

    if save_figures:
        fig.savefig(
            save_prefix + "alphaN_alpha_plus_folding_with_faded_cm_overlays_2x2.pdf",
            format="pdf", bbox_inches="tight", pad_inches=0.03,
        )

    plt.show()

## Figure 9

Projected branch intervals in the $(c_-,\alpha_N)$ plane at fixed $c_+=0.6$, for two slices at
fixed wall velocity. The vertical dot-dashed line marks $c_-=c_+$. The black dashed lines indicate
the fold onset for the inverse detonations on the '(+)' branch (left) and mixed hybrids with
$v_-=c_-$ and direct hybrids on the '(+)' branch (right). Moving to the left of these lines
corresponds to increasing the hierarchy $c_+/c_-$, where the non-injectivities worsen.
**Left:** Subsonic slice $\xi_w=0.85\,c_-$. The $\alpha_+$–$\alpha_N$ relation for inverse
detonations on the '(+)' branch develops an internal minimum and its projected endpoint shared with
the inverse-hybrid side moves upward as $c_-$ decreases. For a sufficiently strong hierarchy,
different solution types overlap (hatched regions) and the inverse-detonation parameter space can
extend to positive $\alpha_N$. **Right:** Intermediate slice
$\xi_w=0.9\min(c_-,c_+)+0.1\max(c_-,c_+)$. For $c_+>c_-$, this slice corresponds to supersonic
wall velocities with respect to $c_-$. The mixed branch with $v_-=c_-$ develops a fold, leading to
visible overlaps between mixed, inverse, and direct branches.

For each slice, the projected interval of every branch,
$[\min\alpha_N,\max\alpha_N]$ over the local $\alpha_+$ interval, is computed on a grid
in $c_-$ at $c_+=0.6$. The grid is split at the exact boundaries
$\xi_w=\xi_\text{min/max}(c_+,c_-)$ of the solution types along the two slices (and,
as for the published figure, along a third slice $\xi_w=c_\text{hi}+0.05(1-c_\text{hi})$
that is not shown), so that neighbouring regions start and end at the same value of
$c_-$. Where a branch
exists on one side of such a boundary only, its interval at the boundary is the
one-sided limit, evaluated $10^{-3}$ grid steps away from it.

In [ ]:
C_PLUS_FIG9 = 0.60
C_MINUS_MIN_FIG9, C_MINUS_MAX_FIG9 = 0.04, 0.98
N_C_MINUS_FIG9 = 120  # approximate number of c_- values (boundaries are added)
N_ALPHA_FIG9, N_ALPHA_FIG9_BOUNDARY = 550, 2200

# wall velocity of each slice as a function of c_-, title and subtitle of the panel
FIG9_SLICES = {
    "sub": (
        lambda cm: 0.85 * cm,
        "subsonic slice",
        r"$\xi_w=0.85\,c_{-}$",
    ),
    "mid": (
        lambda cm: min(C_PLUS_FIG9, cm) + 0.10 * (max(C_PLUS_FIG9, cm) - min(C_PLUS_FIG9, cm)),
        "intermediate slice",
        r"$\xi_w = 0.9\,\min(c_-, c_+) + 0.1\,\max(c_-, c_+)$",
    ),
}


def projected_interval(sol, xi_w, cp, cm, n):
    """[min alpha_N, max alpha_N] of branch sol at xi_w (NaN if it does not exist)."""
    if not branch_exists(sol, xi_w, cp, cm):
        return np.nan, np.nan
    _, aN = alpha_N_map(sol, xi_w, cp, cm, n, edge=2e-6, clustered=True)
    aN = aN[np.isfinite(aN)]
    if len(aN) < max(20, n // 10):
        return np.nan, np.nan
    return aN.min(), aN.max()


def slice_boundaries(xi_func):
    """[(c_-, sol)] where the slice xi_w = xi_func(c_-) crosses xi_min or xi_max of sol."""
    probe = np.linspace(C_MINUS_MIN_FIG9, C_MINUS_MAX_FIG9, 2500)
    roots = []
    for sol in color_dict:
        for bound in (pc.xi_min, pc.xi_max):
            residual = lambda cm: xi_func(cm) - bound(sol, C_PLUS_FIG9, cm)
            f = np.array([residual(cm) for cm in probe])
            for i in np.flatnonzero((np.abs(f[:-1]) < 1e-10) | (f[:-1] * f[1:] < 0.0)):
                root = probe[i] if abs(f[i]) < 1e-10 else brentq(
                    residual, probe[i], probe[i + 1], xtol=1e-12, rtol=1e-12
                )
                # skip duplicates and sign changes at discontinuities of the bound
                if abs(residual(root)) > 1e-7 or any(
                    s == sol and abs(c - root) < 5e-9 for c, s, b in roots if b is bound
                ):
                    continue
                roots.append((root, sol, bound))
    return sorted(((c, s) for c, s, _ in roots), key=lambda r: r[0])


def boundary_aligned_grid(knots):
    """Piecewise uniform c_- grid with the knots as segment ends; the step is at most
    (c_-,max - c_-,min) / (N_C_MINUS_FIG9 - 1)."""
    base_step = (C_MINUS_MAX_FIG9 - C_MINUS_MIN_FIG9) / (N_C_MINUS_FIG9 - 1)
    pieces = [np.linspace(a, b, max(1, int(np.ceil((b - a) / base_step))) + 1)[(j > 0):]
              for j, (a, b) in enumerate(zip(knots[:-1], knots[1:]))]
    grid = np.concatenate(pieces)
    for knot in knots:
        grid[np.argmin(np.abs(grid - knot))] = knot
    return np.unique(grid)


def xi_outer_slice(cm):
    """Outer slice xi_w = c_hi + 0.05 (1 - c_hi), c_hi = max(c_+, c_-): only for the grid."""
    return max(C_PLUS_FIG9, cm) + 0.05 * (1.0 - max(C_PLUS_FIG9, cm))


def scan_fig9():
    boundaries = {key: slice_boundaries(xi_func) for key, (xi_func, *_) in FIG9_SLICES.items()}
    outer = slice_boundaries(xi_outer_slice)
    knots = []
    for c in sorted([C_MINUS_MIN_FIG9, C_PLUS_FIG9, C_MINUS_MAX_FIG9] + [c for b in [*boundaries.values(), outer] for c, _ in b]):
        if not knots or c - knots[-1] > 5e-9:
            knots.append(c)
    grid = boundary_aligned_grid(np.array(knots))
    scan = {"c_minus": grid}
    for key, (xi_func, *_) in FIG9_SLICES.items():
        low = {sol: np.full(grid.size, np.nan) for sol in color_dict}
        high = {sol: np.full(grid.size, np.nan) for sol in color_dict}
        for i, cm in enumerate(grid):
            for sol in color_dict:
                low[sol][i], high[sol][i] = projected_interval(
                    sol, xi_func(cm), C_PLUS_FIG9, cm, N_ALPHA_FIG9
                )
        # one-sided limits at the boundaries
        for root, sol in boundaries[key]:
            i = np.argmin(np.abs(grid - root))
            if np.isfinite(low[sol][i]):
                continue
            for j in (i - 1, i + 1):
                if not (0 <= j < grid.size and np.isfinite(low[sol][j])):
                    continue
                cm = root + np.sign(j - i) * max(1e-10, 1e-3 * abs(grid[j] - root))
                if C_MINUS_MIN_FIG9 < cm < C_MINUS_MAX_FIG9:
                    low[sol][i], high[sol][i] = projected_interval(
                        sol, xi_func(cm), C_PLUS_FIG9, cm, N_ALPHA_FIG9_BOUNDARY
                    )
                    if np.isfinite(low[sol][i]):
                        break
        scan[key] = {"low": low, "high": high}
    return scan


fig09_scan = cached_scan("fig09_branch_intervals", scan_fig9)

Fold-onset lines in the $(c_+,c_-)$ plane, used in Figures 9, 10 and 13 and in
Figure 2 of the letter:

- $\text{DH}^+$: analytic, $c_{-,\text{crit}}^{\text{DH}^+}(c_+)=
  \big[3-c_+^2-\sqrt{(1-c_+^2)(9-c_+^2)}\big]/(2c_+)$;
- $\text{ID}^+$: degree-20 fit of the numerical fold-onset line,
  `pc.inverse_detonation_plus_fold_critical_cm`;
- $\text{MH}^-$: computed here. At $\xi_w=c_-$ the sonic endpoint $v_+=c_+$ satisfies
  $d\alpha_N/dv_+=0$, and the fold appears where also the curvature
  $K=d^2\alpha_N/dv_+^2$ vanishes. For each $c_+$ we integrate the inverse rarefaction
  ahead of the wall (`ln_rw_inverse_rarefaction`, with high precision since $K$ is
  obtained from a fit of $\alpha_N(c_+-\delta)-\alpha_N(c_+)=a_2\delta^2+a_3\delta^3+a_4\delta^4$),
  and solve $K=2a_2=0$ for $c_-$. The numerical line is fitted with the
  endpoint-constrained form
  $c_-^\text{crit}(c_+)=c_+\big[1-(1-c_+^2)\sum_{k=0}^{20}a_k\,c_+^{2k}\big]$,
  which imposes $c_-^\text{crit}(0)=0$ and $c_-^\text{crit}(1)=1$.

`ln_rw_inverse_rarefaction` is also used for Figure 17.

In [ ]:
def cminus_crit_DHplus(cp):
    """Analytic fold onset of the DH+ branch in the (c_+, c_-) plane."""
    return (3.0 - cp**2 - np.sqrt((1.0 - cp**2) * (9.0 - cp**2))) / (2.0 * cp)


S_END = -40.0  # the integration ends at v = -exp(S_END)


def ln_rw_inverse_rarefaction(xi_w, v_plus, cp):
    """ln r_w = ln(w_+/w_N) for the inverse rarefaction ahead of the wall, with the
    wall-frame velocity v_+ in front of the wall.

    The self-similar profile is integrated in s = ln(-v) from the wall to
    v = -exp(S_END), where the remaining contribution is added analytically.
    """
    def rhs(s, y):
        xi, v = y[0], -np.exp(s)
        g2, m = pc.gamma(v) ** 2, pc.mu(xi, v)
        return [xi * g2 * (1.0 - xi * v) * (m * m / cp**2 - 1.0) / 2.0, g2 * m * v]

    sol = solve_ivp(
        rhs, (np.log(-pc.mu(xi_w, v_plus)), S_END), [xi_w, 0.0],
        method="DOP853", rtol=1e-12, atol=1e-14,
    )
    return -pc.get_nu_from_cs(cp) * (sol.y[1, -1] + cp * np.exp(S_END))


def MHminus_sonic_curvature(cp, cm):
    """d^2 alpha_N / d v_+^2 at xi_w = c_-, v_+ = c_+ (from a fit in delta = c_+ - v_+)."""
    def alpha_N(vp):
        r_w = np.exp(ln_rw_inverse_rarefaction(cm, vp, cp))
        return pc.rescale_alpha_by_enthalpy(pc.compute_alpha_plus(vp, cm, cm), r_w, cp, cm)

    delta = (cp - cm) * np.geomspace(3.0e-4, 1.5e-2, 9)
    d_alpha_N = np.array([alpha_N(cp - d) for d in delta]) - alpha_N(cp)
    x = delta / delta.max()
    coefficients, *_ = np.linalg.lstsq(np.column_stack((x**2, x**3, x**4)), d_alpha_N, rcond=None)
    return 2.0 * coefficients[0] / delta.max() ** 2


def MHminus_critical_cminus(cp):
    """Root in c_- of the MH- sonic curvature; if several, the largest one."""
    K = lambda cm: MHminus_sonic_curvature(cp, cm)
    bracket = (max(1.0e-5, 0.005 * cp), min((1.0 - 1.0e-6) * cp, 0.995 * cp))
    if K(bracket[0]) * K(bracket[1]) >= 0.0:
        cm_probe = cp * np.linspace(0.03, 0.985, 32)
        K_probe = np.array([K(cm) for cm in cm_probe])
        i = np.flatnonzero(K_probe[:-1] * K_probe[1:] < 0.0)[-1]
        bracket = (cm_probe[i], cm_probe[i + 1])
    return brentq(K, *bracket, xtol=5.0e-7, rtol=2.0e-6, maxiter=60)


CP_FOLD_SCAN = np.linspace(0.005, 0.995, 80)
cminus_crit_MHminus_raw = cached_scan(
    "fold_onset_MHminus", lambda: np.array([MHminus_critical_cminus(cp) for cp in CP_FOLD_SCAN])
)


def fit_fold_onset(C, m, degree=20):
    """Coefficients a_k of c_- = C [1 - (1 - C^2) sum_k a_k C^(2k)] fitted to (C, m), with
    more weight at large C and penalties for 0 <= c_-/c_+ <= 1 and for a decreasing
    c_-(c_+) on 0 <= c_+ <= 1."""
    model = lambda C, p: C * (1.0 - (1.0 - C**2) * np.polynomial.polynomial.polyval(C**2, p))
    weights = 1.0 + 3.0 * (C / C.max()) ** 8
    C_check = np.linspace(0.0, 1.0, 600)

    def residuals(p):
        ratio = model(C_check, p) / np.where(C_check > 0, C_check, 1.0)
        ratio[0] = 1.0 - np.polynomial.polynomial.polyval(0.0, p)
        return np.concatenate([
            weights * (model(C, p) - m),
            100.0 * np.minimum(ratio, 0.0),
            100.0 * np.maximum(ratio - 1.0, 0.0),
            100.0 * np.minimum(np.diff(model(C_check, p)), 0.0),
        ])

    with quiet("The fit may be poorly conditioned"):
        p0 = np.polynomial.polynomial.polyfit(C**2, (1.0 - m / C) / (1.0 - C**2), deg=degree)
    p = least_squares(residuals, p0, max_nfev=200000, ftol=1e-14, xtol=1e-14, gtol=1e-14).x
    return lambda C: model(C, p)


cminus_crit_MHminus = fit_fold_onset(CP_FOLD_SCAN, cminus_crit_MHminus_raw)

The overlap regions of Figures 9 and 10 and of Figure 2 of the letter are drawn with
[shapely](https://shapely.readthedocs.io): each branch region is a polygon in
coordinates normalized to the axes, the plane is split into cells covered by the same
set of branches, and cells covered by more than one branch are hatched with stripes in
the colours of the branches (at $+45^\circ$ or $-45^\circ$ depending on the branch,
with alternating stripes of both orientations when both are present).

In [ ]:
def clean_geometry(geom):
    """Repaired geometry (buffer(0)), or None if it is empty."""
    if geom is None or geom.is_empty:
        return None
    geom = geom.buffer(0)
    return None if geom.is_empty else geom


def polygon_parts(geom, min_area):
    """Polygons of a geometry with area larger than min_area."""
    geom = clean_geometry(geom)
    if isinstance(geom, Polygon):
        return [geom] if geom.area > min_area else []
    if isinstance(geom, MultiPolygon):
        return [g for g in geom.geoms if g.area > min_area]
    if isinstance(geom, GeometryCollection):
        return [p for g in geom.geoms for p in polygon_parts(g, min_area)]
    return []


def band_polygon(x, low, high, min_area):
    """Region low(x) <= y <= high(x) over the contiguous finite parts of the arrays."""
    valid = np.isfinite(x) & np.isfinite(low) & np.isfinite(high) & (low <= high)
    idx = np.flatnonzero(valid)
    polygons = []
    for block in np.split(idx, np.flatnonzero(np.diff(idx) > 1) + 1):
        if len(block) < 2:
            continue
        order = np.argsort(x[block])
        xb, lb, hb = x[block][order], low[block][order], high[block][order]
        coords = list(zip(xb, lb)) + list(zip(xb[::-1], hb[::-1]))
        polygons += polygon_parts(clean_geometry(Polygon(coords)), min_area)
    return clean_geometry(unary_union(polygons)) if polygons else None


def to_unit_square(geom, xlim, ylim):
    """Map a geometry from data coordinates to the unit square of the axes."""
    return shapely.transform(
        geom,
        lambda xy: np.column_stack(
            [(xy[:, 0] - xlim[0]) / (xlim[1] - xlim[0]), (xy[:, 1] - ylim[0]) / (ylim[1] - ylim[0])]
        ),
    )


def add_geometry_patch(ax, geom, xlim, ylim, min_area, antialiased=None, **kwargs):
    """Draw a geometry given in unit-square coordinates as filled patches."""
    for poly in polygon_parts(geom, min_area):
        verts, codes = [], []
        for ring in [poly.exterior, *poly.interiors]:
            xy = np.asarray(ring.coords)
            verts.append(np.column_stack([
                xlim[0] + xy[:, 0] * (xlim[1] - xlim[0]), ylim[0] + xy[:, 1] * (ylim[1] - ylim[0])
            ]))
            ring_codes = np.full(len(xy), MplPath.LINETO, dtype=np.uint8)
            ring_codes[0], ring_codes[-1] = MplPath.MOVETO, MplPath.CLOSEPOLY
            codes.append(ring_codes)
        ax.add_patch(PathPatch(
            MplPath(np.concatenate(verts), np.concatenate(codes)),
            edgecolor="none", lw=0.0, antialiased=antialiased, **kwargs,
        ))


def split_cells(geoms, order, min_area):
    """Split the unit square into cells covered by the same set of geometries."""
    cells = [(box(0.0, 0.0, 1.0, 1.0), frozenset())]
    for key in order:
        if key not in geoms:
            continue
        new_cells = []
        for cell, active in cells:
            inside = clean_geometry(cell.intersection(geoms[key]))
            outside = clean_geometry(cell.difference(geoms[key]))
            if inside is not None and inside.area > min_area:
                new_cells.append((inside, active | {key}))
            if outside is not None and outside.area > min_area:
                new_cells.append((outside, active))
        cells = new_cells
    return cells


def stripe_polygon(orientation, phase_lo, phase_hi, L=3.0):
    """Stripe phase_lo < x -/+ y < phase_hi (orientation +1/-1) in unit-square coordinates."""
    x0, x1 = -L, 1.0 + L
    if orientation == +1:
        return Polygon([(x0, x0 - phase_lo), (x1, x1 - phase_lo), (x1, x1 - phase_hi), (x0, x0 - phase_hi)])
    return Polygon([(x0, phase_lo - x0), (x1, phase_lo - x1), (x1, phase_hi - x1), (x0, phase_hi - x0)])


def draw_overlap_cells(ax, cells, stripe_order, orientation, colors, xlim, ylim, width,
                       solid_alpha, stripe_alpha, min_area, underfill=False, antialiased=None):
    """Fill the cells: one colour where a single branch is present, stripes otherwise.

    stripe_order fixes the order of the stripe colours, orientation[key] = +1/-1 their
    direction. With underfill, the cell is first filled with the first colour.
    """
    def stripes(geom, keys, sign, full_cover, zorder):
        if underfill:
            geom = clean_geometry(geom)
        geom = clean_geometry(geom.intersection(box(0.0, 0.0, 1.0, 1.0)))
        if geom is None:
            return
        phase_min, phase_max = (-1.2, 1.2) if sign == +1 else (-0.2, 2.2)
        period = width if full_cover else 2.0 * width
        for n in range(int(np.floor(phase_min / period)) - 2, int(np.ceil(phase_max / period)) + 3):
            lo = n * period
            hi = (n + 1) * width if full_cover else lo + width
            clipped = clean_geometry(geom.intersection(stripe_polygon(sign, lo, hi)))
            if clipped is not None:
                add_geometry_patch(ax, clipped, xlim, ylim, min_area, antialiased,
                                   facecolor=colors[keys[n % len(keys)]], alpha=stripe_alpha, zorder=zorder)

    for geom, active in cells:
        keys = [key for key in stripe_order if key in active]
        if not keys:
            continue
        if len(keys) == 1:
            add_geometry_patch(ax, geom, xlim, ylim, min_area, antialiased,
                               facecolor=colors[keys[0]], alpha=solid_alpha, zorder=2)
            continue
        if underfill:
            geom = clean_geometry(geom)
            add_geometry_patch(ax, geom, xlim, ylim, min_area, antialiased,
                               facecolor=colors[keys[0]], alpha=solid_alpha, zorder=9)
        pos = [key for key in keys if orientation[key] == +1]
        neg = [key for key in keys if orientation[key] == -1]
        if not neg:
            stripes(geom, pos, +1, True, 10)
        elif not pos:
            stripes(geom, neg, -1, True, 10)
        elif len(pos) >= len(neg):
            stripes(geom, pos, +1, True, 10)
            stripes(geom, neg, -1, False, 11)
        else:
            stripes(geom, neg, -1, True, 10)
            stripes(geom, pos, +1, False, 11)

Plot (`phase_plane_all_branches_2panels.pdf`), with the fold-onset lines at $c_+=0.6$.
Overlaps are hatched as described above; the filled regions are rasterized, so that
the PDF has no hairlines between neighbouring patches.

In [ ]:
with figure_style("profile"):
    xlim = (fig09_scan["c_minus"].min(), fig09_scan["c_minus"].max())
    YLIM_FIG9 = {"sub": (-0.75, 0.50), "mid": (-0.25, 0.50)}
    MIN_AREA_FIG9 = 1.0e-12
    # direct branches and MH+ hatched at +45 degrees, inverse branches and MH- at -45 degrees
    orientation = {sol: -1 if sol in (ST.InverseDetonationPlus, ST.InverseDetonation, ST.InverseHybrid,
                                      ST.InverseDeflagration, ST.MixedHybridMinus) else +1
                   for sol in color_dict}
    fold_onsets = {
        "sub": [(pc.inverse_detonation_plus_fold_critical_cm(C_PLUS_FIG9), r"ID$^+$ fold onset")],
        "mid": [
            (cminus_crit_DHplus(C_PLUS_FIG9), r"DH$^+$ fold onset"),
            (cminus_crit_MHminus(C_PLUS_FIG9), r"MH$^-$ fold onset"),
        ],
    }

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 6.2), sharey=False)
    plt.subplots_adjust(wspace=0.12, bottom=0.34)

    for ax, (key, (_, title, subtitle)) in zip(axes, FIG9_SLICES.items()):
        ylim = YLIM_FIG9[key]
        x = fig09_scan["c_minus"]  # contains c_- = c_+
        low, high = fig09_scan[key]["low"], fig09_scan[key]["high"]

        geoms = {}
        for sol in color_dict:
            geom = band_polygon(x, low[sol], high[sol], MIN_AREA_FIG9)
            if geom is not None:
                geom = clean_geometry(to_unit_square(geom, xlim, ylim).intersection(box(0.0, 0.0, 1.0, 1.0)))
                if geom is not None:
                    geoms[sol] = geom

        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.axhline(0.0, color="k", lw=0.9, alpha=0.7, zorder=1)
        ax.grid(True, alpha=0.22)
        draw_overlap_cells(
            ax, split_cells(geoms, color_dict, MIN_AREA_FIG9), color_dict, orientation, color_dict,
            xlim, ylim, width=0.030, solid_alpha=0.90, stripe_alpha=0.96, min_area=MIN_AREA_FIG9,
            underfill=True, antialiased=False,
        )
        # rasterize the filled regions (no hairlines in the PDF)
        for patch in ax.patches:
            patch.set_antialiased(False)
            patch.set_rasterized(True)
        for collection in ax.collections:
            collection.set_rasterized(True)

        for sol, color in color_dict.items():
            ax.plot(x, low[sol], color=color, lw=1.6, alpha=0.95, zorder=30)
            ax.plot(x, high[sol], color=color, lw=1.6, alpha=0.95, zorder=30)
        ax.axvline(C_PLUS_FIG9, color="0.25", lw=1.1, ls="-.", zorder=50)
        for i, (cm_fold, label) in enumerate(fold_onsets[key]):
            ax.axvline(cm_fold, color="k", lw=1.25, ls="--", zorder=49)
            ax.text(
                cm_fold, 0.97 - 0.12 * i, label, transform=ax.get_xaxis_transform(),
                rotation=90, ha="right", va="top", fontsize=11,
            )
        ax.set_title(title + "\n" + subtitle, fontsize=16)
        ax.set_xlabel(r"$c_-$", fontsize=18)
        ax.tick_params(axis="both", labelsize=14)

    axes[0].set_ylabel(r"$\alpha_N$", fontsize=18)

    labels_fig9 = {
        ST.Detonation: r"direct detonation",
        ST.HybridPlus: r"direct hybrid $+$",
        ST.Hybrid: r"direct hybrid $-$",
        ST.Deflagration: r"direct deflagration",
        ST.InverseDetonationPlus: r"inverse detonation $+$",
        ST.InverseDetonation: r"inverse detonation $-$",
        ST.InverseHybrid: r"inverse hybrid",
        ST.InverseDeflagration: r"inverse deflagration",
        ST.MixedHybridPlus: r"mixed hybrid $+$",
        ST.MixedHybridMinus: r"mixed hybrid $-$",
    }
    fig.legend(
        handles=[Line2D([0], [0], color=color_dict[sol], lw=12, label=label)
                 for sol, label in labels_fig9.items()]
        + [
            Line2D([0], [0], color="0.25", lw=1.1, ls="-.", label=r"$c_-=c_+$"),
            Line2D([0], [0], color="k", lw=1.25, ls="--", label=r"fold onset"),
        ],
        loc="lower center",
        bbox_to_anchor=(0.5, -0.0),
        ncol=4,
        frameon=False,
        fontsize=18,
    )

    if save_figures:
        fig.savefig(
            save_prefix + "phase_plane_all_branches_2panels.pdf",
            format="pdf", bbox_inches="tight", pad_inches=0.03, facecolor="white", dpi=300,
        )

    plt.show()

## Figure 10

**Left**: sound-speed plane showing physically motivated parameter regions and the thresholds
controlling the loss of injectivity. The curves $c_{-,\mathrm{crit}}^{\mathrm{DH}^+}(c_+)$,
$c_{-,\mathrm{crit}}^{\mathrm{MH}^-}(c_+)$, and $c_{-,\mathrm{crit}}^{\mathrm{ID}^+}(c_+)$ mark the
onset of folds in the respective branches, cf. eq. (5.2). The threshold
$c_{-,\mathrm{ex}}^{\mathrm{ID}^+}(c_+)$ marks the subsequent exchange in the ordering of the
projected branch endpoints, cf. eq. (5.3), while $c_{-,0}^{\mathrm{IH}}(c_+)$ marks the crossing of
the upper inverse-hybrid boundary through $\alpha_N=0$, cf. eq. (5.4). The corresponding threshold
$c_{-,0}^{\mathrm{ID}^+}(c_+)$ for inverse detonations, eq. (5.5), lies slightly below. The dotted
blue line denotes $c_+=c_-$, and the dashed grey lines $c_\pm=1/\sqrt{3}$; the grey background
indicates the region where at least one sound speed exceeds the conformal value. Coloured blobs
schematically indicate physically motivated equations of state, while the black star marks the
hierarchical benchmark used in the right panel. **Right**: hydrodynamic classification in the
$(\xi_w,\alpha_N)$ plane for $c_+=0.9$ and $c_-=0.16$, with the same colour coding as figures 3
to 5. Hatched regions indicate overlaps in which distinct stationary profiles correspond to the
same $(\xi_w,\alpha_N)$.

Phase plane $(\xi_w,\alpha_N)$ for $c_+=0.9$ and $c_-=0.16$ (right panel; also used
for Figure 11 and for Figure 2 of the letter). For each branch and wall velocity, the
bounds of $\alpha_N$ are the extremes of the projection of the local $\alpha_+$
interval (`alpha_N_map`), so that folded branches are resolved.

In [ ]:
C_PLUS_PP, C_MINUS_PP = 0.9, 0.16
N_XI_PP = N_ALPHA_PP = 301

# drawing order of the branches in the phase planes (Figures 10 and 11, letter Figure 2)
PHASE_PLANE_ORDER = [
    ST.Deflagration,
    ST.Hybrid,
    ST.HybridPlus,
    ST.Detonation,
    ST.InverseDeflagration,
    ST.InverseHybrid,
    ST.InverseDetonation,
    ST.InverseDetonationPlus,
    ST.MixedHybridPlus,
    ST.MixedHybridMinus,
]


def projected_bounds(sol, cp, cm):
    """Lower and upper alpha_N bound of branch sol on a grid in xi_w."""
    xi = np.linspace(pc.xi_min(sol, cp, cm) + ximin, pc.xi_max(sol, cp, cm) - ximin, N_XI_PP)
    low = np.full(N_XI_PP, np.nan)
    high = np.full(N_XI_PP, np.nan)
    for i, xi_w in enumerate(xi):
        _, aN = alpha_N_map(sol, xi_w, cp, cm, N_ALPHA_PP)
        aN = aN[np.isfinite(aN)]
        if len(aN) >= 5:
            low[i], high[i] = aN.min(), aN.max()
    return {"xi": xi, "low": low, "high": high,
            "finite": np.isfinite(low) & np.isfinite(high) & (low <= high)}


phase_plane = cached_scan(
    "fig10_phase_plane",
    lambda: {
        sol: projected_bounds(sol, C_PLUS_PP, C_MINUS_PP)
        for sol in PHASE_PLANE_ORDER
        if pc.xi_min(sol, C_PLUS_PP, C_MINUS_PP) + ximin < pc.xi_max(sol, C_PLUS_PP, C_MINUS_PP) - ximin
    },
)

Scan of the $(c_+,c_-)$ plane (left panel). For the upper inverse-detonation branch
$\text{ID}^+$, the endpoints of the $\alpha_+$ interval at fixed $\xi_w$ are
$v_+=c_+$ (shared with the inverse hybrids) and $v_+=v_{+,\text{extr}}(\xi_w)$; their
projections are evaluated as one-sided limits from inside the interval, where the
branch is not degenerate. `exchange` marks where
$\alpha_N^\text{shared}>\alpha_N^\text{extr}$ for some $\xi_w$, i.e. where the
endpoints exchange their ordering, and `IDplus_positive` where
$\alpha_N^\text{shared}>0$ for some $\xi_w$. Inverse hybrids share this endpoint for
$\xi_w<c_-$; above the low-$c_-$ part of `IDplus_positive`, `IH_positive` is extended
column by column with the maximum of the upper bound of the inverse hybrids
(`pc.alpha_max`) over $\xi_w<(c_-+c_+)/2$, which excludes the trivial end
$\xi_w\to c_+$, until it becomes negative. Points with $c_+-c_-<2\times10^{-3}$, where
the $\text{ID}^+$ interval collapses, are skipped.

In [ ]:
N_C_FIG10 = 180
SOUND_SPEEDS_FIG10 = np.linspace(0.005, 0.995, N_C_FIG10)
N_XI_ENDPOINTS = 90
MIN_SOUND_SPEED_GAP = 2e-3


def xi_range(sol, cp, cm, n):
    """n wall velocities of branch sol, 0.2% of the range away from its ends."""
    lo, hi = pc.xi_min(sol, cp, cm), pc.xi_max(sol, cp, cm)
    return np.linspace(lo + 2e-3 * (hi - lo), hi - 2e-3 * (hi - lo), n)


def IDplus_endpoint_alpha_N(cp, cm, xi_w, ap_end, ap_other):
    """alpha_N at the endpoint ap_end of the ID+ alpha_+ interval, approached from inside."""
    for frac in (1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 5e-2):
        aN = projected_alpha_N(ST.InverseDetonationPlus, ap_end + frac * (ap_other - ap_end), xi_w, cp, cm)
        if np.isfinite(aN):
            return aN
    return np.nan


def IDplus_endpoint_flags(cp, cm):
    """(exchange, positive) of the ID+ endpoints at (c_+, c_-), see above."""
    exchange = positive = False
    for xi_w in xi_range(ST.InverseDetonationPlus, cp, cm, N_XI_ENDPOINTS):
        vp_extr = pc.get_v_plus_extremum_from_v_minus(xi_w, cm)
        if not cm < vp_extr < cp:
            continue
        ap_shared = pc.compute_alpha_plus(cp, xi_w, cm)
        ap_extr = pc.compute_alpha_plus(vp_extr, xi_w, cm)
        if ap_shared == ap_extr:
            continue
        aN_shared = IDplus_endpoint_alpha_N(cp, cm, xi_w, ap_shared, ap_extr)
        aN_extr = IDplus_endpoint_alpha_N(cp, cm, xi_w, ap_extr, ap_shared)
        if np.isfinite(aN_shared) and np.isfinite(aN_extr):
            exchange |= aN_shared > aN_extr
            positive |= aN_shared > 0.0
    return exchange, positive


def IH_max_alpha_N(cp, cm):
    """Maximum of the upper alpha_N bound of the inverse hybrids over xi_w < (c_- + c_+)/2."""
    sol = ST.InverseHybrid
    lo = pc.xi_min(sol, cp, cm)
    hi = pc.xi_max(sol, cp, cm)
    lo, hi = lo + 2e-3 * (hi - lo), min(hi - 2e-3 * (hi - lo), 0.5 * (cp + cm))
    if not lo < hi:
        return np.nan

    def alpha_N_max(xi_w):
        try:
            return pc.alpha_max(sol, xi_w, cp, cm)
        except (ValueError, RuntimeError):
            return np.nan

    xis = np.linspace(lo, hi, N_XI_ENDPOINTS)
    values = np.array([alpha_N_max(xi_w) for xi_w in xis])
    if not np.any(np.isfinite(values)):
        return np.nan
    k = int(np.nanargmax(values))
    res = minimize_scalar(
        lambda x: -alpha_N_max(x) if np.isfinite(alpha_N_max(x)) else np.inf,
        bounds=(xis[max(k - 1, 0)], xis[min(k + 1, N_XI_ENDPOINTS - 1)]),
        method="bounded",
        options={"xatol": 1e-7},
    )
    return max(values[k], -res.fun) if res.success and np.isfinite(res.fun) else values[k]


def scan_sound_speed_plane():
    c = SOUND_SPEEDS_FIG10
    exchange = np.zeros((N_C_FIG10, N_C_FIG10), dtype=bool)  # index [i (c_-), j (c_+)]
    IDplus_positive = np.zeros_like(exchange)
    for j, cp in enumerate(c):
        for i, cm in enumerate(c):
            if cm < cp and cp - cm >= MIN_SOUND_SPEED_GAP and pc.xi_min(ST.InverseDetonationPlus, cp, cm) < pc.xi_max(ST.InverseDetonationPlus, cp, cm):
                exchange[i, j], IDplus_positive[i, j] = IDplus_endpoint_flags(cp, cm)
    IH_positive = IDplus_positive.copy()
    for j, cp in enumerate(c):
        col = IDplus_positive[:, j]
        # first c_- above the low-c_- run of ID+-positive points
        i = np.flatnonzero(col)[0] if col.any() else 0
        while i < N_C_FIG10 and (col[i] if col.any() else not c[i] < cp):
            i += 1
        for i in range(i, N_C_FIG10):
            if not (c[i] < cp and cp - c[i] >= MIN_SOUND_SPEED_GAP and IH_max_alpha_N(cp, c[i]) > 0.0):
                break
            IH_positive[i, j] = True
    return {"exchange": exchange, "IDplus_positive": IDplus_positive, "IH_positive": IH_positive}


sound_speed_plane = cached_scan("fig10_sound_speed_plane", scan_sound_speed_plane)

The upper boundaries $c_-(c_+)$ of the low-$c_-$ parts of the three regions of the scan
(small gaps of at most two points are filled, runs of a few points and isolated
outliers are discarded) are fitted for $c_+>0.2$ with
$q_\text{eff}(c_+)=(1-c_+^2)\sum_{k=0}^4 a_k\,c_+^{2k}$ and
$c_-=c_+[2q_\text{eff}-1+c_+^2]/[1-c_+^2+2q_\text{eff}c_+^2]$, which give the lines
$c_{-,\text{ex}}^{\text{ID}^+}$, $c_{-,0}^{\text{ID}^+}$ and $c_{-,0}^{\text{IH}}$.

In [ ]:
def true_runs(col):
    """(start, end) of the runs of True values of a boolean array (end inclusive)."""
    idx = np.flatnonzero(col)
    if len(idx) == 0:
        return []
    blocks = np.split(idx, np.flatnonzero(np.diff(idx) > 1) + 1)
    return [(b[0], b[-1]) for b in blocks]


def low_cm_envelope(mask, min_run, max_jump, max_residual):
    """Upper boundary (c_+, c_-) of the low-c_- connected part of mask[i (c_-), j (c_+)]."""
    c = SOUND_SPEEDS_FIG10
    cp_b, cm_b = [], []
    for j, cp in enumerate(c):
        col = mask[:, j] & (c < cp)
        runs = true_runs(col)
        for (a1, b1), (a2, b2) in zip(runs[:-1], runs[1:]):
            if a2 - b1 - 1 <= 2:  # fill small gaps
                col[b1 + 1 : a2] = True
        runs = [(a, b) for a, b in true_runs(col) if b - a + 1 >= min_run and c[a] < 0.35 * cp]
        if runs:
            cp_b.append(cp)
            cm_b.append(c[runs[0][1]])
    cp_b, cm_b = np.array(cp_b), np.array(cm_b)
    # isolated outliers with respect to the median of 11 neighbouring columns
    keep = np.ones(len(cp_b), dtype=bool)
    for k in range(len(cp_b)):
        med = np.median(cm_b[max(0, k - 5) : k + 6])
        if abs(cm_b[k] - med) > max_residual or (
            k > 0 and abs(cm_b[k] - cm_b[k - 1]) > max_jump and abs(cm_b[k] - med) > 0.5 * max_residual
        ):
            keep[k] = False
    return cp_b[keep], cm_b[keep]


def q_model(C, p):
    return (1.0 - C**2) * np.polynomial.polynomial.polyval(C**2, p)


def cm_from_q(C, q):
    return C * (2.0 * q - 1.0 + C**2) / (1.0 - C**2 + 2.0 * q * C**2)


def fit_envelope(C, m, degree=4):
    """Line c_-(c_+) fitted to the envelope points with c_+ > 0.2 (see above)."""
    q = (1.0 - C**2) * (C + m) / (2.0 * C * (1.0 - C * m))
    good = (C > 0.2) & (m > 0.0) & (m < C) & (q > 0.0) & (q < 1.0)
    C, m, q = C[good], m[good], q[good]
    weights = 1.0 + 3.0 * (C / C.max()) ** 6

    def residuals(p):
        m_pred, q_pred = cm_from_q(C, q_model(C, p)), q_model(C, p)
        bad = ~np.isfinite(m_pred) | (q_pred <= 0.0) | (q_pred >= 1.0) | (m_pred <= 0.0) | (m_pred >= C)
        return np.where(bad, 10.0, weights * (m_pred - m))

    p0 = np.polyfit(C**2, q / (1.0 - C**2), deg=degree)[::-1]
    p = least_squares(residuals, p0, max_nfev=50000, ftol=1e-13, xtol=1e-13, gtol=1e-13).x
    C_curve = np.linspace(SOUND_SPEEDS_FIG10[0], SOUND_SPEEDS_FIG10[-1], 2000)
    q_curve = q_model(C_curve, p)
    m_curve = cm_from_q(C_curve, q_curve)
    valid = (q_curve > 0.0) & (q_curve < 1.0) & (m_curve > 0.0) & (m_curve < C_curve)
    return C_curve[valid], m_curve[valid]


cminus_exchange_IDplus = fit_envelope(
    *low_cm_envelope(sound_speed_plane["exchange"], min_run=5, max_jump=0.08, max_residual=0.055)
)
cminus_zero_IDplus = fit_envelope(
    *low_cm_envelope(sound_speed_plane["IDplus_positive"], min_run=4, max_jump=0.06, max_residual=0.04)
)
cminus_zero_IH = fit_envelope(
    *low_cm_envelope(sound_speed_plane["IH_positive"], min_run=4, max_jump=0.06, max_residual=0.04)
)

Plot (`phase_plane_model_classes_panel.pdf`). `draw_model_classes` draws the left panel;
with `letter=True` it draws the left panel of Figure 2 of the letter instead.

In [ ]:
CS_CONFORMAL = 1.0 / np.sqrt(3.0)


def add_faded_ellipse(ax, center, width, height, angle, color, alpha_max, n_layers, zorder=7):
    """Ellipse fading out towards its edge."""
    for i in range(n_layers, 0, -1):
        s = i / n_layers
        ax.add_patch(Ellipse(
            xy=center, width=width * s, height=height * s, angle=angle, facecolor=color,
            edgecolor="none", alpha=alpha_max * (1.0 - s) ** 1.8, zorder=zorder,
        ))


def add_faded_wedge(ax, center, radius, theta1, theta2, color, alpha_max, n_layers, zorder=7):
    """Circular sector fading out towards its edge."""
    for i in range(n_layers, 0, -1):
        s = i / n_layers
        ax.add_patch(Wedge(
            center=center, r=radius * s, theta1=theta1, theta2=theta2, facecolor=color,
            edgecolor="none", alpha=alpha_max * (1.0 - s) ** 1.8, zorder=zorder,
        ))


def add_conformal_arrow(ax, y_tip, text, color, ls="-", lw=2.35, zorder=8):
    """Vertical arrow from the conformal point with a framed label."""
    ax.add_patch(FancyArrowPatch(
        (CS_CONFORMAL, CS_CONFORMAL), (CS_CONFORMAL, y_tip), arrowstyle="-|>", mutation_scale=24,
        lw=lw, linestyle=ls, color=color, shrinkA=0.0, shrinkB=0.0, zorder=zorder,
    ))
    ax.text(
        CS_CONFORMAL + 0.024, y_tip, text, fontsize=11.0, ha="left", va="center", color=color,
        zorder=zorder + 1,
        bbox=dict(boxstyle="round,pad=0.32", facecolor="white", edgecolor=color, linewidth=1.35, alpha=1.0),
    )


def draw_model_classes(ax, legend_fontsize, letter=False):
    """(c_+, c_-) plane with the fold-onset lines, the regions of the scan above and
    schematic patches of physically motivated equations of state.

    letter=True: without c_{-,ex}^{ID+} and c_{-,0}^{IH}, labels without '+', two
    patches for pure SU(3) Yang-Mills instead of one.
    """
    suffix = "" if letter else "^+"
    C = np.linspace(0.0, 1.0, 4000)
    cm_MH, cm_ID = cminus_crit_MHminus(C), pc.inverse_detonation_plus_fold_critical_cm(C)
    C_DH = np.linspace(1.0e-4, 1.0 - 1.0e-6, 3000)
    cm_DH = cminus_crit_DHplus(C_DH)
    valid_DH = (cm_DH > 0.0) & (cm_DH < C_DH)

    # light grey where at least one sound speed exceeds the conformal value
    c_bg = np.linspace(0.0, 1.0, 400)
    CP_bg, CM_bg = np.meshgrid(c_bg, c_bg)
    ax.pcolormesh(
        c_bg, c_bg, ((CP_bg > CS_CONFORMAL) | (CM_bg > CS_CONFORMAL)).astype(int),
        cmap=ListedColormap([(1.0, 1.0, 1.0, 0.0), (0.93, 0.93, 0.93, 1.0)]),
        vmin=0, vmax=1, shading="auto", zorder=0, rasterized=True,
    )
    if not letter:
        C_ex, cm_ex = cminus_exchange_IDplus
        ax.fill_between(C_ex, 0.0, cm_ex, color="#f6b26b", alpha=0.62, zorder=1)
    C_zero, cm_zero = cminus_zero_IDplus
    ax.fill_between(C_zero, 0.0, cm_zero, color="#e06666", alpha=0.62, zorder=2)
    diag = np.linspace(0.0, 1.0, 500)
    ax.plot(diag, diag, color="dodgerblue", ls=":", lw=1.4, label=r"$c_+=c_-$", zorder=3)
    ax.axvline(CS_CONFORMAL, color="0.80", ls="--", lw=1.3, zorder=3)
    ax.axhline(CS_CONFORMAL, color="0.80", ls="--", lw=1.3, zorder=3)
    if not letter:
        ax.plot(*cminus_exchange_IDplus, color="darkorange", lw=2.8, zorder=5,
                label=r"$c_{-,\mathrm{ex}}^{\mathrm{ID}^+}$")
    ax.plot(C_zero, cm_zero, color="firebrick", lw=2.8, zorder=6,
            label=rf"$c_{{-,0}}^{{\mathrm{{ID{suffix}}}}}$")
    if not letter:
        ax.plot(*cminus_zero_IH, color="darkred", ls="--", lw=2.2, zorder=7,
                label=r"$c_{-,0}^{\mathrm{IH}}$")
    ax.plot(C_DH[valid_DH], cm_DH[valid_DH], color="black", ls=":", lw=2.3, zorder=8.5,
            label=rf"$c_{{-,\mathrm{{crit}}}}^{{\mathrm{{DH}}{suffix}}}(c_+)$")
    ax.plot(C[(cm_MH >= 0.0) & (cm_MH <= C + 1e-10)], cm_MH[(cm_MH >= 0.0) & (cm_MH <= C + 1e-10)],
            color="black", ls="--", lw=2.0, zorder=8.5,
            label=r"$c_{-,\mathrm{crit}}^{\mathrm{MH}^-}(c_+)$")
    ax.plot(C[(cm_ID >= 0.0) & (cm_ID <= C + 1e-10)], cm_ID[(cm_ID >= 0.0) & (cm_ID <= C + 1e-10)],
            color="black", ls="-", lw=2.0, zorder=8.5,
            label=rf"$c_{{-,\mathrm{{crit}}}}^{{\mathrm{{ID}}{suffix}}}(c_+)$")

    # schematic equations of state
    add_faded_wedge(ax, (CS_CONFORMAL, CS_CONFORMAL), 0.25 * CS_CONFORMAL, 180.0, 270.0,
                    "deepskyblue", alpha_max=0.16, n_layers=80)
    ax.text(CS_CONFORMAL - 0.25, CS_CONFORMAL - 0.055, "EW-like", color="deepskyblue", fontsize=12, zorder=8)
    ns_center = (0.5 * (CS_CONFORMAL + 0.85), 0.5 * (CS_CONFORMAL + 0.85))
    add_faded_ellipse(ax, ns_center, (0.85 - CS_CONFORMAL) * 1.75, 0.13, 45.0, "#1f9e89",
                      alpha_max=0.14, n_layers=80)
    ax.text(ns_center[0] + 0.08, ns_center[1] - 0.08, "QCD-like\nneutron stars", color="#1f9e89",
            fontsize=11, ha="center", va="center", zorder=8)
    if not letter:
        add_faded_ellipse(ax, (0.4, 0.4), 0.28, 0.15, 45.0, "tab:green", alpha_max=0.12, n_layers=80)
        ax.text(0.4 - 0.05, 0.4 - 0.125, "QCD-like\n(pure YM)", color="tab:green", fontsize=12, zorder=8)
    add_faded_ellipse(ax, (0.13, 0.13), 0.16, 0.10, 45.0, "tab:olive", alpha_max=0.14, n_layers=80)
    ax.text(0.13 - 0.12, 0.13 + 0.065, "Walecka\nliquid-gas", color="tab:olive", fontsize=11, zorder=8)
    add_faded_ellipse(ax, (1.0, 1.0), 0.34, 0.18, 45.0, "mediumorchid", alpha_max=0.16, n_layers=90)
    ax.text(0.76, 0.90, "stiff charged\ncondensates", color="mediumorchid", fontsize=11,
            ha="center", va="center", zorder=8)

    ax.plot(CS_CONFORMAL, CS_CONFORMAL, "o", ms=8, color="k", mfc="white", mew=1.5, zorder=9,
            label=r"$c_+=c_-=1/\sqrt{3}$")
    ax.plot(C_PLUS_PP, C_MINUS_PP, "*", ms=14, color="k", zorder=9)
    if letter:
        # pure SU(3) Yang-Mills (lattice equation of state of Giusti et al., arXiv:2501.10284):
        # c_s ~ 0.11-0.27 in the deconfined phase within ~2% above T_c, c_s ~ 0.21-0.28 in
        # the confined phase for 0.8 <= T/T_c <= 1; drawn 35% larger than these ranges
        c_deconfined, c_confined = (0.11, 0.27), (0.21, 0.28)
        for c_p, c_m in ((c_deconfined, c_confined), (c_confined, c_deconfined)):
            add_faded_ellipse(ax, (np.mean(c_p), np.mean(c_m)), 1.35 * (c_p[1] - c_p[0]),
                              1.35 * (c_m[1] - c_m[0]), 0.0, "tab:green", alpha_max=0.16, n_layers=80)
        ax.text(0.18, 0.31, "pure $SU(3)$ YM\n(supercooled)", color="tab:green", fontsize=11,
                ha="center", va="center", zorder=8)
        ax.text(0.28, 0.2, "(superheated)", color="tab:green", fontsize=11, ha="left",
                va="center", zorder=8, rotation=-90)

    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.0)
    ax.set_xlabel(r"$c_+$")
    ax.set_ylabel(r"$c_-$")
    ticks = [0.0, CS_CONFORMAL, 1.0]
    ax.set_xticks(ticks)
    ax.set_xticklabels([r"$0$", r"$\frac{1}{\sqrt{3}}$", r"$1$"])
    ax.set_yticks(ticks)
    ax.set_yticklabels([r"$0$", r"$\frac{1}{\sqrt{3}}$", r"$1$"])
    ax.minorticks_off()
    ax.set_box_aspect(1)
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False, loc="upper left", fontsize=legend_fontsize)


# overlap regions of the phase planes: stripes at +45 degrees for the direct branches,
# MH+ and ID, at -45 degrees for the other inverse branches and MH-
PP_ORIENTATION = {sol: -1 if sol in (ST.InverseDetonationPlus, ST.InverseHybrid,
                                     ST.InverseDeflagration, ST.MixedHybridMinus) else +1
                  for sol in color_dict}
PP_XLIM, PP_YLIM = (0.0, 1.0), (-1.0, 1.0)
MIN_AREA_PP = 1.0e-10


def phase_plane_geometries(clip=True):
    """Branch regions of the phase plane in unit-square coordinates (clipped to it)."""
    geoms = {}
    for sol, res in phase_plane.items():
        geom = band_polygon(res["xi"], res["low"], res["high"], MIN_AREA_PP)
        if geom is not None:
            geom = to_unit_square(geom, PP_XLIM, PP_YLIM)
            geom = clean_geometry(geom.intersection(box(0.0, 0.0, 1.0, 1.0)) if clip else geom)
            if geom is not None:
                geoms[sol] = geom
    return geoms


with figure_style("profile", LARGE_FONT_RC):
    fig, (ax_model, ax_phase) = plt.subplots(
        1, 2, figsize=(2.0 * 5.4 + 0.9, 5.4 + 0.55), subplot_kw={"box_aspect": 1},
        gridspec_kw={"width_ratios": [1.0, 1.0], "wspace": 0.25},
    )

    draw_model_classes(ax_model, legend_fontsize=8.8)

    # right: phase plane with overlaps
    draw_overlap_cells(
        ax_phase, split_cells(phase_plane_geometries(), PHASE_PLANE_ORDER, MIN_AREA_PP),
        PHASE_PLANE_ORDER, PP_ORIENTATION, color_dict, PP_XLIM, PP_YLIM, width=0.015,
        solid_alpha=0.90, stripe_alpha=0.92, min_area=MIN_AREA_PP,
    )
    ax_phase.axhline(0.0, c="k", lw=1.0, zorder=30)
    ax_phase.axvline(C_MINUS_PP, c="k", ls="--", lw=1.0, zorder=30)
    ax_phase.axvline(C_PLUS_PP, c="k", ls=":", lw=1.0, zorder=30)
    for res in phase_plane.values():
        idx = np.flatnonzero(res["finite"])
        for block in np.split(idx, np.flatnonzero(np.diff(idx) > 1) + 1):
            if len(block) < 2:
                continue
            xb, lb, hb = res["xi"][block], res["low"][block], res["high"][block]
            for x, y in ((xb, lb), (xb, hb), ([xb[0], xb[0]], [lb[0], hb[0]]), ([xb[-1], xb[-1]], [lb[-1], hb[-1]])):
                ax_phase.plot(x, y, c="k", lw=1.0, zorder=40)
    ax_phase.set_xlim(*PP_XLIM)
    ax_phase.set_ylim(*PP_YLIM)
    ax_phase.set_box_aspect(1)
    ax_phase.set_title(rf"$c_+ = {C_PLUS_PP:g},\ c_- = {C_MINUS_PP:g}$")
    ax_phase.set_xlabel(r"$\xi_w$")
    ax_phase.set_ylabel(r"$\alpha_{N}$")
    ax_phase.set_xticks(
        [0.0, C_MINUS_PP, C_PLUS_PP, 1.0], labels=[r"$0$", r"$c_-$", r"$c_+$", r"$1$"]
    )
    ax_phase.grid(False)

    with quiet(TIGHT_LAYOUT_WARNING):
        plt.tight_layout()

    if save_figures:
        fig.savefig(
            save_prefix + "phase_plane_model_classes_panel.pdf",
            format="pdf", bbox_inches="tight", facecolor="white", pad_inches=0.04,
        )

    plt.show()

### Letter, Figure 2

**Left**: critical curves for the onset of folds on the $\mathrm{ID}$ (solid), $\mathrm{MH}^-$
(dashed), and $\mathrm{DH}$ (dotted) branches in the $(c_+,c_-)$ plane. Folds occur below the
corresponding curves. The red region marks the appearance of inverse solutions with
$\alpha_N>0$. Shaded patches and arrows indicate the schematic sound-speed ranges and parametric
trends of the microscopic examples discussed in the text. The star marks
$(c_+,c_-)=(0.9,0.16)$. **Right**: hydrodynamic domains in the $(\xi_w,\alpha_N)$ plane for this
benchmark, with colors as in Fig. 1 and hatching denoting regions where several solutions
coexist.

Left: the left panel of Figure 10 without $c_{-,\mathrm{ex}}^{\mathrm{ID}^+}$ and
$c_{-,0}^{\mathrm{IH}}$ (`draw_model_classes` with `letter=True`); right: the phase plane
of Figure 10 with the two sub-branches of the direct hybrids and of the inverse
detonations merged into one region each (`model_classes_phase_plane_noninjective_map.pdf`).
The left panel shows the following microscopic examples, in addition to those of
Figure 10:

- **pure $SU(3)$ Yang–Mills** (green): one patch for supercooled transitions
  (deconfined phase ahead of the wall, confined phase behind it) and one for
  superheated transitions (phases swapped). The sound speeds
  $c_s^2=(d\ln s/d\ln T)^{-1}$ follow from the lattice equation of state of
  Giusti et al. ([arXiv:2501.10284](https://arxiv.org/abs/2501.10284)):
  $c_s\simeq0.11$–$0.27$ in the deconfined phase within about 2% above $T_c$, and
  $c_s\simeq0.21$–$0.28$ in the confined phase for $0.8\le T/T_c\le1$;
- **thermal mass threshold** (dashed orange arrow): a thermal plasma behind the
  wall dominated by a heavy, Boltzmann-suppressed species; reaching the onset of
  the folds, $c_+/c_-\simeq2.5$ at $c_+=1/\sqrt3$, requires $m/T\simeq18$;
- **degenerate fermions** (blue arrow): a degenerate Fermi gas with
  $c_-^2=k_F^2/[3(k_F^2+m^2)]$ reaches much lower sound speeds, e.g.
  $c_-\simeq0.16$ for nucleons at saturation density ($k_F\simeq263$ MeV).

In [ ]:
# merged branches of the right panel: (direct hybrids), (inverse detonations)
MERGE_INTO = {ST.HybridPlus: ST.Hybrid, ST.InverseDetonation: ST.InverseDetonationPlus}
MERGED_ORDER = list(dict.fromkeys(MERGE_INTO.get(sol, sol) for sol in PHASE_PLANE_ORDER))

with figure_style("profile", LARGE_FONT_RC):
    # left (c_+, c_-) panel and right phase plane with the same height (sizes in inches)
    panel, gap, margin_l, margin_b, margin_t, margin_r = 5.4, 1.2, 0.62, 0.62, 0.18, 0.22
    fig_w = 2 * panel + gap + margin_l + margin_r
    fig_h = panel + margin_b + margin_t
    fig = plt.figure(figsize=(fig_w, fig_h))
    ax_model = fig.add_axes([margin_l / fig_w, margin_b / fig_h, panel / fig_w, panel / fig_h])
    ax_phase = fig.add_axes(
        [(margin_l + panel + gap) / fig_w, margin_b / fig_h, panel / fig_w, panel / fig_h]
    )

    # left panel
    draw_model_classes(ax_model, legend_fontsize=11.8, letter=True)
    add_conformal_arrow(ax_model, 0.1, r"degenerate fermions", color="#1A5490", lw=2.5)
    # Boltzmann-suppressed heavy species: c_+/c_- = 2.5 (onset of folds)
    add_conformal_arrow(ax_model, CS_CONFORMAL / 2.5, "thermal mass threshold", color="#C05621",
                        ls="--", lw=2.4, zorder=11)
    ax_model.plot(CS_CONFORMAL, CS_CONFORMAL, "o", ms=8, color="k", mfc="white", mew=1.5, zorder=25)

    # right panel: merged regions, closing the sampling seams between merged branches
    pieces = {}
    for sol, geom in phase_plane_geometries(clip=False).items():
        pieces.setdefault(MERGE_INTO.get(sol, sol), []).append(geom)
    geoms = {}
    for key, gs in pieces.items():
        geom = gs[0] if len(gs) == 1 else unary_union([g.buffer(2.0e-4) for g in gs]).buffer(-2.0e-4)
        geoms[key] = clean_geometry(clean_geometry(geom).intersection(box(0.0, 0.0, 1.0, 1.0)))
    draw_overlap_cells(
        ax_phase, split_cells(geoms, MERGED_ORDER, MIN_AREA_PP), MERGED_ORDER, PP_ORIENTATION,
        merged_color_dict, PP_XLIM, PP_YLIM, width=0.015, solid_alpha=0.90, stripe_alpha=0.92,
        min_area=MIN_AREA_PP,
    )
    for key in MERGED_ORDER:
        for poly in polygon_parts(geoms[key], MIN_AREA_PP):
            for ring in [poly.exterior, *poly.interiors]:
                xy = np.asarray(ring.coords)
                ax_phase.plot(xy[:, 0], PP_YLIM[0] + xy[:, 1] * (PP_YLIM[1] - PP_YLIM[0]),
                              c="k", lw=1.0, zorder=40)
    ax_phase.axhline(0.0, c="k", lw=1.0, zorder=30)
    ax_phase.axvline(C_MINUS_PP, c="k", ls="--", lw=1.0, zorder=30)
    ax_phase.axvline(C_PLUS_PP, c="k", ls=":", lw=1.0, zorder=30)
    ax_phase.set_xlim(*PP_XLIM)
    ax_phase.set_ylim(*PP_YLIM)
    ax_phase.grid(False)
    ax_phase.set_xticks([0.0, C_MINUS_PP, C_PLUS_PP, 1.0])
    ax_phase.set_xticklabels([r"$0$", r"$c_{\vphantom{+}-}$", r"$c_+$", r"$1$"], ha="center", va="top")
    ax_phase.tick_params(axis="x", which="major", pad=8)
    ax_phase.set_xlabel(r"$\xi_w$")
    ax_phase.set_ylabel(r"$\alpha_N$")

    if save_figures:
        fig.savefig(
            save_prefix + "model_classes_phase_plane_noninjective_map.pdf",
            format="pdf", bbox_inches="tight", facecolor="white", pad_inches=0.04,
        )

    plt.show()

## Figure 11

Thermodynamic splitting of the hydrodynamically degenerate solutions for $c_+=0.9$ and
$c_-=0.16$. The colour scale shows $\mathcal{D}_\mathrm{LTE}$, defined in eq. (5.6), only in the
regions where at least two physically distinct hydrodynamic solutions coexist at the same
$(\xi_w,\alpha_N)$. A vanishing value would signal two distinct solutions requiring the same
$\Psi_N$ in LTE. No such degeneracy is found in the interior of the overlap regions within the
numerical resolution of the scan, showing that the thermodynamic completion lifts the
hydrodynamic non-injectivity.

For every $\xi_w$ of the grid and every branch, the projection $\alpha_+\mapsto\alpha_N$
(`pc.compute_alpha_N`) is sampled over the local $\alpha_+$ interval (`pc.alpha_min`,
`pc.alpha_max` with `return_alpha_N=False`) and split into monotone pieces at its interior
extrema, which are refined on the native map. Every piece whose image contains $\alpha_N$
carries exactly one root, so all physical roots are counted, including the two roots of a
folded branch. Where there are at least three, the roots are computed (`pc.find_alpha_plus`
on univalued branches, bracketed `brentq` on the pieces of folded branches); roots at a
common boundary of two branches describing the same state $(v_+,v_-)$ are merged.
$\Psi_{N,\min}$ [eq. (3.19) of the paper] follows from $v_\pm$
(`pc.compute_fluid_velocities`) and $r_w=w_+/w_N=(\alpha_N-\alpha_0)/(\alpha_+-\alpha_0)$,
and $\mathcal{D}_\text{LTE}=\min_{i<j}|\ln(\Psi_{N,\min}^{(i)}/\Psi_{N,\min}^{(j)})|$
[eq. (5.6)] is evaluated only at the points with exactly three distinct states.
The profiles are integrated as $v(\xi)$ up to the shock fronts
(`ProfileSolveMode.D_V_D_XI_FOR_SHOCKS`), which removes spurious oscillations of
$\alpha_N(\alpha_+)$ for weak deflagrations.

In [ ]:
XI_W_FIG11 = np.linspace(1e-4, 1.0 - 1e-4, 201)
ALPHA_N_FIG11 = np.linspace(-1.0, 1.0, 301)
STEP_FIG11 = 1e-5
SOLVE_MODE_FIG11 = pc.ProfileSolveMode.D_V_D_XI_FOR_SHOCKS

N_ALPHA_PLUS = 81  # samples of alpha_+ per branch, geometric from the lower endpoint
ALPHA_N_NOISE = 2e-4  # turning points of alpha_N(alpha_+) below this are numerical noise
ALPHA_PLUS_EPS = 1e-9  # relative inset from the endpoints of the alpha_+ interval
SAME_STATE_TOL = 1e-6  # roots with |dv_+|, |dv_-| below this are the same state

# alpha_0 = alpha_N at r_w = 0 [eq. (3.11)], i.e. alpha_theta(c_+^2, c_-^2)
ALPHA_0_LTE = pc.compute_alpha_plus(C_PLUS_PP**2, C_MINUS_PP**2, C_MINUS_PP)
NU_PLUS_LTE = pc.get_nu_from_cs(C_PLUS_PP)
NU_MINUS_LTE = pc.get_nu_from_cs(C_MINUS_PP)


def alpha_N_LTE(sol, alpha_plus, xi_w):
    """Native projection alpha_+ -> alpha_N on branch sol."""
    return pc.compute_alpha_N(
        sol, alpha_plus, xi_w, C_PLUS_PP, C_MINUS_PP, STEP_FIG11, SOLVE_MODE_FIG11
    )


def monotone_pieces(sol, xi_w):
    """Monotone pieces (alpha_+^lo, alpha_+^hi, alpha_N^lo, alpha_N^hi) of the projection
    of branch sol at fixed xi_w. The interior extrema are located on the sampled map and
    refined on the native map; all end values are native."""
    a0 = pc.alpha_min(sol, xi_w, C_PLUS_PP, C_MINUS_PP, False, STEP_FIG11, SOLVE_MODE_FIG11)
    a1 = pc.alpha_max(sol, xi_w, C_PLUS_PP, C_MINUS_PP, False, STEP_FIG11, SOLVE_MODE_FIG11)
    a0 += ALPHA_PLUS_EPS * max(1.0, abs(a0))
    a1 -= ALPHA_PLUS_EPS * max(1.0, abs(a1))
    ap = a0 + (a1 - a0) * np.concatenate([[0.0], np.geomspace(1e-6, 1.0, N_ALPHA_PLUS - 1)])
    aN = np.array([alpha_N_LTE(sol, a, xi_w) for a in ap])
    # turning points (with hysteresis ALPHA_N_NOISE): +1 maximum, -1 minimum
    turns, direction, i_ext = [], 0, 0
    for i in range(1, len(aN)):
        if direction == 0:
            if abs(aN[i] - aN[0]) > ALPHA_N_NOISE:
                direction, i_ext = (1 if aN[i] > aN[0] else -1), i
        elif direction * (aN[i] - aN[i_ext]) > 0:
            i_ext = i
        elif direction * (aN[i_ext] - aN[i]) > ALPHA_N_NOISE:
            turns.append((i_ext, direction))
            direction, i_ext = -direction, i
    ends = [(a0, aN[0])]
    for i, kind in turns:
        ext = minimize_scalar(
            lambda a: -kind * alpha_N_LTE(sol, a, xi_w),
            bounds=(ap[i - 1], ap[i + 1]),
            method="bounded",
            options={"xatol": 1e-12 * max(1.0, abs(ap[i]))},
        )
        ends.append((ext.x, -kind * ext.fun))
    ends.append((a1, aN[-1]))
    return [(x0, x1, f0, f1) for (x0, f0), (x1, f1) in zip(ends[:-1], ends[1:])]


def root_on_piece(sol, piece, univalued, alpha_N, xi_w):
    """alpha_+ of the unique root on a monotone piece: pc.find_alpha_plus on univalued
    branches (away from the noisy end values), brentq on the native map otherwise."""
    x0, x1, f0, f1 = piece
    if univalued and min(f0, f1) + ALPHA_N_NOISE < alpha_N < max(f0, f1) - ALPHA_N_NOISE:
        return pc.find_alpha_plus(
            sol, alpha_N, xi_w, C_PLUS_PP, C_MINUS_PP, STEP_FIG11, SOLVE_MODE_FIG11
        )
    return brentq(lambda a: alpha_N_LTE(sol, a, xi_w) - alpha_N, x0, x1, xtol=1e-14)


def physical_state(sol, alpha_plus, xi_w):
    """v_+, v_- and ln Psi_N,min [eq. (3.21)] of one root. With r_w = w_+/w_N and the
    LTE ratio Psi_+ = (v_+/v_-) (gamma_-/gamma_+)^(nu_- - 2) at the wall (as in
    pc.compute_enthalpy_ratio_LTE), Psi_N,min = Psi_+ r_w^(1 - nu_-/nu_+)."""
    v_plus, v_minus = pc.compute_fluid_velocities(xi_w, alpha_plus, sol, C_PLUS_PP, C_MINUS_PP)
    r_w = (alpha_N_LTE(sol, alpha_plus, xi_w) - ALPHA_0_LTE) / (alpha_plus - ALPHA_0_LTE)
    log_psi = (
        np.log(v_plus / v_minus)
        + 0.5 * (NU_MINUS_LTE - 2.0) * np.log((1.0 - v_plus**2) / (1.0 - v_minus**2))
        + (1.0 - NU_MINUS_LTE / NU_PLUS_LTE) * np.log(r_w)
    )
    return v_plus, v_minus, log_psi


def scan_three_state_LTE():
    n_states = np.zeros((ALPHA_N_FIG11.size, XI_W_FIG11.size), dtype=int)
    D_LTE = np.full(n_states.shape, np.nan)
    for ix, xi_w in enumerate(XI_W_FIG11):
        pieces = {
            sol: monotone_pieces(sol, xi_w)
            for sol in ST.__members__.values()
            if branch_exists(sol, xi_w, C_PLUS_PP, C_MINUS_PP)
        }
        for iy, alpha_N in enumerate(ALPHA_N_FIG11):
            # one root on every monotone piece whose image contains alpha_N
            roots = [
                (sol, piece, len(sol_pieces) == 1)
                for sol, sol_pieces in pieces.items()
                for piece in sol_pieces
                if min(piece[2], piece[3]) < alpha_N < max(piece[2], piece[3])
            ]
            n_states[iy, ix] = len(roots)
            if len(roots) < 3:
                continue
            states = []
            for sol, piece, univalued in roots:
                state = physical_state(sol, root_on_piece(sol, piece, univalued, alpha_N, xi_w), xi_w)
                # the same physical state reached from two branches at a common boundary
                if not any(
                    abs(state[0] - s[0]) < SAME_STATE_TOL and abs(state[1] - s[1]) < SAME_STATE_TOL
                    for s in states
                ):
                    states.append(state)
            n_states[iy, ix] = len(states)
            if len(states) == 3:
                D_LTE[iy, ix] = np.min(np.diff(np.sort([s[2] for s in states])))
    return {"n_states": n_states, "D_LTE": D_LTE}


fig11_scan = cached_scan("fig11_three_state_LTE", scan_three_state_LTE)

Plot (`LTE_splitting.pdf`), with the branch boundaries of Figure 10.

In [ ]:
with figure_style("profile"):
    # D_LTE only where exactly three distinct states coexist (NaN elsewhere) and D_LTE > 0
    D_shown = np.ma.masked_where(~(fig11_scan["D_LTE"] > 0.0), fig11_scan["D_LTE"])

    fig, ax = plt.subplots(figsize=(7, 7 / 1.658))
    pcm = ax.pcolormesh(
        XI_W_FIG11, ALPHA_N_FIG11, D_shown, shading="auto", cmap="viridis_r",
        norm=LogNorm(vmin=D_shown.min(), vmax=D_shown.max()), rasterized=True, zorder=2,
    )
    for res in phase_plane.values():
        good = res["finite"]
        ax.plot(res["xi"][good], res["low"][good], c="k", lw=0.55, zorder=10)
        ax.plot(res["xi"][good], res["high"][good], c="k", lw=0.55, zorder=10)
    ax.axhline(0.0, c="k", lw=0.6, zorder=11)
    ax.axvline(C_MINUS_PP, c="k", ls="--", lw=0.7, zorder=11)
    ax.axvline(C_PLUS_PP, c="k", ls=":", lw=0.7, zorder=11)

    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(-1.0, 1.0)
    ax.set_xticks([0.0, C_MINUS_PP, C_PLUS_PP, 1.0], labels=[r"$0$", r"$c_-$", r"$c_+$", r"$1$"])
    ax.set_xlabel(r"$\xi_w$")
    ax.set_ylabel(r"$\alpha_N$")
    ax.set_title(rf"$c_+={C_PLUS_PP:g},\quad c_-={C_MINUS_PP:g}$")
    ax.grid(False)

    cbar = fig.colorbar(pcm, ax=ax, pad=0.02)
    cbar.set_label(
        r"$\mathcal{D}_{\rm LTE}=\min_{i<j}\left|\ln\left("
        r"\Psi_{N,\min}^{(i)}/\Psi_{N,\min}^{(j)}\right)\right|$",
        fontsize=14,
    )

    plt.tight_layout()

    if save_figures:
        fig.savefig(save_prefix + "LTE_splitting.pdf", bbox_inches="tight", pad_inches=0.03)

    plt.show()

## Figure 12

Kinetic-energy fraction $K=\rho_\mathrm{kin}/\rho_\mathrm{tot}$ for $(c_+,c_-)=(0.75,0.4)$, marked
by grey vertical dashed lines. The left and right panels show $K$ for direct and inverse
solutions, with colour indicating $|\alpha_N|$. The black dashed curves show $K$ along the
ordinary CJ boundaries $\alpha_\mathrm{CJ}^\mathrm{dir}(\xi_w)$ and
$\alpha_\mathrm{CJ}^\mathrm{inv}(\xi_w)$ defined in eq. (3.21). The orange dashed curves extend
the same sonic conditions, $v_-=c_-$ and $v_+=c_+$, onto the corresponding mixed-hybrid branches,
as discussed in section 3.5; open circles mark the endpoints of the displayed mixed-hybrid
segments. In the inverse panel, the horizontal red dashed line marks $K_\mathrm{inv}^\mathrm{max}$
of eq. (6.7), equal to $(1+c_+c_-)/2$ for this benchmark. Grey-shaded regions are kinematically
excluded.

Kinetic-energy fraction $K=\rho_\text{kin}/\rho_\text{tot}$ on a grid in
$(\xi_w,\alpha_N)$ for $(c_+,c_-)=(0.75,0.40)$ (`get_efficiency` with
`EfficiencyNormalization.TotalEnergyDensity`; the perturbed volume extends to
$\xi_f=\max(\xi_w,c_+,\xi_\text{sh})$). The profiles are integrated as $v(\xi)$ up to
the shock fronts with step size $3\times10^{-6}$.

In [ ]:
C_PLUS_FIG12, C_MINUS_FIG12 = 0.75, 0.40
STEP_FIG12 = 3e-6
SOLVE_MODE_FIG12 = pc.ProfileSolveMode.D_V_D_XI_FOR_SHOCKS
ALPHA_N_FIG12 = np.array([
    0.005, 0.01, 0.02, 0.03, 0.05, 0.075, 0.1, 0.2, 0.3, 0.4,
    0.5, 0.6, 0.7, 0.75, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3,
    1.4, 1.5, 1.75, 2.0, 3.0, 4.0, 5.0, 10.0,
])
XI_W_FIG12 = np.arange(0.01, 1.0, 0.001)


def kinetic_fraction(xi_w, alpha_N):
    """K of the profile at (xi_w, alpha_N), NaN if no profile can be built there."""
    try:
        profile = hydrodex.FluidProfile(
            xi_w, alpha_N, C_PLUS_FIG12, C_MINUS_FIG12, STEP_FIG12, SOLVE_MODE_FIG12
        )
    except (ValueError, RuntimeError):  # no solution, or a degenerate one (e.g. xi_w = c_+)
        return np.nan
    return profile.get_efficiency(pc.EfficiencyNormalization.TotalEnergyDensity)


fig12_K = cached_scan(
    "fig12_efficiency_scan",
    lambda: {
        float(alpha_N): np.array([kinetic_fraction(xi_w, alpha_N) for xi_w in XI_W_FIG12])
        for alpha_N in np.concatenate([ALPHA_N_FIG12, -ALPHA_N_FIG12])
    },
)

Upper bounds and CJ curves of $K(\xi_w)$, evaluated on the phase-plane boundaries
$\alpha_{N,\max}$ / $\alpha_{N,\min}$ (`pc.alpha_max`, `pc.alpha_min`), shifted by
$\delta=\max(10^{-4},10^{-3}|\alpha_N|)$:

- upper bound for direct transitions: maximum over the direct branches and
  $\text{MH}^+$ of $K$ at $\alpha_{N,\max}-\delta$;
- upper bound for inverse transitions: $K$ at $\alpha_{N,\min}-\delta$ (or, where no
  profile exists there, $\alpha_{N,\min}+\delta$) for the lowest $\alpha_{N,\min}$ of the
  inverse branches;
- CJ curves: $K$ at $\alpha_{N,\min}+\delta$ of $\text{DH}^+$ (direct) and at
  $\alpha_{N,\max}-\delta$ of the inverse hybrids (inverse), continued onto the mixed
  hybrids with $\alpha_{N,\min}$ of $\text{MH}^-$ where it is positive and
  $\alpha_{N,\max}$ of $\text{MH}^+$ where it is negative. Where no profile exists at
  the first point, the point at $2\delta$ is used.

In [ ]:
XI_W_BOUND = np.linspace(1e-4, 1.0 - 1e-4, 300)


def shift(alpha_N):
    return max(1e-4, 1e-3 * max(abs(alpha_N), 1e-6))


def K_or_nan(xi_w, alpha_N):
    """Positive K at (xi_w, alpha_N), otherwise NaN."""
    K = kinetic_fraction(xi_w, alpha_N)
    return K if K > 0.0 else np.nan


def first_finite_K(xi_w, alphas):
    for alpha_N in alphas:
        K = K_or_nan(xi_w, alpha_N)
        if np.isfinite(K):
            return K
    return np.nan


def in_range(sol, xi_w):
    return pc.xi_min(sol, C_PLUS_FIG12, C_MINUS_FIG12) + 1e-4 <= xi_w <= pc.xi_max(sol, C_PLUS_FIG12, C_MINUS_FIG12) - 1e-4


def alpha_N_edge(sol, xi_w, upper):
    return (pc.alpha_max if upper else pc.alpha_min)(sol, xi_w, C_PLUS_FIG12, C_MINUS_FIG12)


def upper_bound_direct():
    K = np.full(XI_W_BOUND.size, np.nan)
    direct = [ST.Detonation, ST.Deflagration, ST.Hybrid, ST.HybridPlus, ST.MixedHybridPlus]
    for i, xi_w in enumerate(XI_W_BOUND):
        values = [
            K_or_nan(xi_w, a - shift(a))
            for sol in direct if in_range(sol, xi_w)
            for a in [alpha_N_edge(sol, xi_w, True)] if 0.0 < a < 100.0
        ]
        values = [k for k in values if np.isfinite(k)]
        if values:
            K[i] = max(values)
    return K


def upper_bound_inverse():
    K = np.full(XI_W_BOUND.size, np.nan)
    inverse = [ST.InverseDetonation, ST.InverseDetonationPlus, ST.InverseHybrid, ST.InverseDeflagration]
    for i, xi_w in enumerate(XI_W_BOUND):
        alphas = [alpha_N_edge(sol, xi_w, False) for sol in inverse if in_range(sol, xi_w)]
        alphas = [a for a in alphas if -20.0 < a < 0.0]
        if alphas:
            a = min(alphas)
            K[i] = first_finite_K(xi_w, [b for b in (a - shift(a), a + shift(a)) if b < 0.0])
    return K


def CJ_curve(sol, upper):
    """K inside the alpha_N,min (upper=False, alpha_N > 0) or alpha_N,max (upper=True,
    alpha_N < 0) boundary of an ordinary hybrid branch."""
    K = np.full(XI_W_BOUND.size, np.nan)
    sign = -1 if upper else +1
    for i, xi_w in enumerate(XI_W_BOUND):
        if in_range(sol, xi_w):
            a = alpha_N_edge(sol, xi_w, upper)
            if sign * a > 0.0:
                K[i] = first_finite_K(xi_w, [a + sign * shift(a), a + 2 * sign * shift(a)])
    return K


def CJ_curve_mixed(sol, upper):
    """As CJ_curve, on a mixed-hybrid branch from (MH-) or up to (MH+) the wall
    velocity where its boundary crosses alpha_N = 0."""
    K = np.full(XI_W_BOUND.size, np.nan)
    sign = -1 if upper else +1
    lo = pc.xi_min(sol, C_PLUS_FIG12, C_MINUS_FIG12) + 1e-4
    hi = pc.xi_max(sol, C_PLUS_FIG12, C_MINUS_FIG12) - 1e-4
    xi_zero = next(xi for xi in np.linspace(lo, hi, 400) if alpha_N_edge(sol, xi, upper) >= 0.0)
    for i, xi_w in enumerate(XI_W_BOUND):
        if (xi_zero - 1e-4 <= xi_w <= hi) if sign > 0 else (lo <= xi_w <= xi_zero + 1e-4):
            a = alpha_N_edge(sol, xi_w, upper)
            if sign * (a + sign * shift(a)) > 0.0:
                K[i] = first_finite_K(xi_w, [a + sign * shift(a), a + 2 * sign * shift(a)])
    return K


fig12_bounds = cached_scan(
    "fig12_bounds",
    lambda: {
        "upper_direct": upper_bound_direct(),
        "upper_inverse": upper_bound_inverse(),
        "CJ_direct": CJ_curve(ST.HybridPlus, upper=False),
        "CJ_inverse": CJ_curve(ST.InverseHybrid, upper=True),
        "CJ_mixed_direct": CJ_curve_mixed(ST.MixedHybridMinus, upper=False),
        "CJ_mixed_inverse": CJ_curve_mixed(ST.MixedHybridPlus, upper=True),
    },
)

Plot (`efficiency_factor.pdf`). Isolated points of a curve are drawn as markers.

In [ ]:
def plot_positive(ax, x, y, **plot_kwargs):
    """Plot the positive values of y, with markers for isolated points."""
    valid = np.isfinite(y) & (y > 0)
    ax.plot(x, np.where(valid, y, np.nan), **plot_kwargs)
    isolated = valid & ~np.r_[False, valid[:-1]] & ~np.r_[valid[1:], False]
    if isolated.any():
        marker_kwargs = {**plot_kwargs, "linestyle": "none", "marker": "o", "markersize": 4.5}
        marker_kwargs.pop("ls", None)
        ax.plot(x[isolated], y[isolated], **marker_kwargs)


with figure_style("profile"):
    cmap = mpl.colormaps["viridis"]
    norm = LogNorm(vmin=ALPHA_N_FIG12.min(), vmax=ALPHA_N_FIG12.max())
    K_inverse_max = (1 + C_PLUS_FIG12 * C_MINUS_FIG12) / 2
    cj_style = dict(linewidth=1.9, linestyle="--", zorder=5)
    tick_style = dict(which="major", labelsize=28, length=9, width=1.6)

    # the inverse upper bound is extended to xi_w = 0
    i0 = np.flatnonzero(fig12_bounds["upper_inverse"] > 0)[0]
    bound_direct = (XI_W_BOUND, fig12_bounds["upper_direct"])
    bound_inverse = (
        np.r_[0.0, XI_W_BOUND[i0:]],
        np.r_[fig12_bounds["upper_inverse"][i0], fig12_bounds["upper_inverse"][i0:]],
    )

    def plot_panel(ax, sign, bound, cj, cj_mixed):
        x, K_bound = bound
        valid = np.isfinite(K_bound) & (K_bound > 0)
        ax.fill_between(x, K_bound, 1.0, where=valid, color="0.85", edgecolor="none", zorder=0)
        if sign < 0:
            ax.axhspan(K_inverse_max, 1.0, facecolor="0.75", edgecolor="none", zorder=0.5)
            ax.axhline(K_inverse_max, color="red", linestyle="--", linewidth=1.9, zorder=4)
        for cs, label, color in ((C_PLUS_FIG12, "c_+", "k"), (C_MINUS_FIG12, "c_-", "0.45")):
            ax.axvline(cs, color=color, linestyle="--", linewidth=1.5, alpha=0.5,
                       label=rf"${label} = {cs:.3g}$")
        for alpha_N in ALPHA_N_FIG12:
            plot_positive(ax, XI_W_FIG12, fig12_K[float(sign * alpha_N)],
                          color=cmap(norm(alpha_N)), linewidth=2)
        plot_positive(ax, XI_W_BOUND, cj, color="k", **cj_style)
        plot_positive(ax, XI_W_BOUND, cj_mixed, color="orange", **cj_style)
        # endpoint of the mixed-hybrid segment
        valid = np.flatnonzero(np.isfinite(cj_mixed) & (cj_mixed > 0))
        if valid.size:
            i = valid[-1] if sign < 0 else valid[0]
            ax.scatter(XI_W_BOUND[i], cj_mixed[i], s=70, facecolors="white",
                       edgecolors="k", linewidths=1.2, zorder=6)
        ax.set_yscale("log")
        ax.grid(False)

    fig = plt.figure(figsize=(16, 6), facecolor="white")
    gs = fig.add_gridspec(1, 2, wspace=0.18, left=0.11, right=0.90, top=0.85, bottom=0.17)
    ax_direct = fig.add_subplot(gs[0, 0])
    gs_right = gs[0, 1].subgridspec(1, 2, width_ratios=[1, 0.035], wspace=0.02)
    ax_inverse = fig.add_subplot(gs_right[0, 0])
    cax = fig.add_subplot(gs_right[0, 1])

    panels = (
        (ax_direct, +1, bound_direct, "CJ_direct", "CJ_mixed_direct",
         r"Direct Transitions ($\alpha_N > 0$)", (0.5, 0.6), (4e-3, 1.2e-2)),
        (ax_inverse, -1, bound_inverse, "CJ_inverse", "CJ_mixed_inverse",
         r"Inverse Transitions ($\alpha_N < 0$)", (0.4, 0.55), (6e-3, 1.1e-2)),
    )
    for ax, sign, bound, cj, cj_mixed, title, zoom_x, zoom_y in panels:
        ax.set_facecolor("white")
        plot_panel(ax, sign, bound, fig12_bounds[cj], fig12_bounds[cj_mixed])
        ax.set(xlim=(0, 1), ylim=(1e-4, 1))
        ax.set_xlabel(r"$\xi_w$", fontsize=24)
        ax.set_title(title, fontsize=24)
        ax.tick_params(**tick_style)

        inset = ax.inset_axes([0.04, 0.55, 0.42, 0.42])
        plot_panel(inset, sign, bound, fig12_bounds[cj], fig12_bounds[cj_mixed])
        inset.set(xlim=zoom_x, ylim=zoom_y, xticks=[], yticks=[])
        inset.minorticks_off()
        frame, *_ = mark_inset(ax, inset, loc1=3, loc2=1, fc="none", ec="0.35", lw=1.0, linestyle="--", zorder=9)
        frame.set(linestyle="-", linewidth=1.4, zorder=10)

    ax_direct.set_ylabel(r"$K=\rho_{\rm kin}/\rho_{\rm N, tot}$", fontsize=24)
    cbar = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), cax=cax)
    cbar.solids.set_edgecolor("face")
    cbar.set_label(r"$|\alpha_N|$", fontsize=24, labelpad=8)
    cbar.ax.tick_params(**tick_style)

    if save_figures:
        fig.savefig(save_prefix + "efficiency_factor.pdf", facecolor="white", bbox_inches="tight", pad_inches=0.25)

    plt.show()

## Figure 13

Peak position (left) and peak amplitude (right) as functions of the sound speed $c_s=c_-$, for
fixed $c_+=1/\sqrt{3}$ and $\mathcal{H}_*R_*=0.1$. Solid lines denote direct solutions with
$\alpha_N=+0.1$, while dashed lines denote inverse solutions with $\alpha_N=-0.1$. The colour
indicates the wall velocity and the markers identify the hydrodynamic branch. The grey regions
mark $c_-<c_{-,\mathrm{crit}}^{\mathrm{ID}^+}$ where different solution branches can overlap in
the $(\xi_w,\alpha_N)$ plane and they are not displayed.

Figures 13 and 14 use the sound-shell model (`hydrodex.SoundShellSpectrum`) with
exponential nucleation and a constant-EoS expansion history. `gw_spectrum` computes
the shape function, the kinetic spectrum and the gravitational-wave spectrum of a
profile; without an explicit source duration it uses the nonlinear time,
$\mathcal{H}_*\Delta\tau=\mathcal{H}_*R_*/\sqrt{\Omega_K}$. The grids of the
computation are passed as a dictionary (`chi`, `qR`, `kR` and the numbers of points of
the internal momentum integrations `n_qR`, `n_p`, `n_z`).

In [ ]:
def gw_spectrum(profile, H_R, cs, omega, grids, H_delta_tau=None):
    """Shape function, kinetic and gravitational-wave spectrum of a FluidProfile."""
    spec = hydrodex.SoundShellSpectrum(
        profile=profile,
        H_R=H_R,
        nhist=ssm.NucleationHistory.EXPONENTIAL,
        exhist=ssm.ExpansionHistory.CONSTANT_EOS,
        cs=cs,
        omega=omega,
        Omega_star=1.0,
    )
    spec.set_correlator_integration_settings(grids["n_p"], grids["n_z"], ssm.MomentumIntegration.FULL)
    spec.calculate_A_squared(grids["chi"])
    spec.calculate_kinetic_spectrum(grids["qR"])
    result = {
        "chi": np.asarray(spec.get_chi()),
        "A2": np.asarray(spec.get_shape_function()),
        "qR": np.asarray(spec.get_qR()),
        "kinetic_normalized": np.asarray(spec.get_normalized_kinetic_spectrum()),
        "Omega_K": spec.get_kinetic_energy_density_parameter(),
    }
    if H_delta_tau is None:
        H_delta_tau = H_R / np.sqrt(result["Omega_K"])
    spec.set_source_duration(H_delta_tau)
    spec.calculate_gw_spectrum(grids["kR"], grids["n_qR"])
    result.update(
        H_delta_tau=H_delta_tau,
        kR=np.asarray(spec.get_kR()),
        Omega_GW=np.asarray(spec.get_gravitational_wave_spectrum()),
    )
    return result


def positive_xy(x, y):
    """Points with finite, positive x and y."""
    good = np.isfinite(x) & np.isfinite(y) & (x > 0.0) & (y > 0.0)
    return x[good], y[good]


def sampled_peak(x, y):
    """(x, y) of the largest positive sample."""
    x, y = positive_xy(x, y)
    i = int(np.argmax(y))
    return x[i], y[i]

Self-consistent gravitational-wave scan in $(\xi_w,\alpha_N,c_-)$ at $c_+=1/\sqrt{3}$
and $\mathcal{H}_*R_*=0.1$. For every point, `FluidProfile` determines the solution type
and the profile, and the spectrum is computed with $c_s=c_-$ and $\omega=c_-^2$; points
without a valid profile are skipped. This is the most expensive scan of the notebook.

In [ ]:
C_PLUS_FIG13 = 1 / np.sqrt(3)
H_R_FIG13 = 0.1
STEP_FIG13 = 5.0e-5
ALPHA_N_VALUES = [0.10, -0.10]
XI_W_VALUES = [0.07, 0.12, 0.22, 0.32, 0.42, 0.52, 0.62, 0.72, 0.82]
CM_CASES = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
GW_GRIDS_FIG13 = dict(
    chi=np.logspace(-3.0, 4.7, 10000),
    qR=np.logspace(-3.0, 2.2, 250),
    kR=np.logspace(-3.0, 2.0, 180),
    n_qR=700,
    n_p=120,
    n_z=96,
)


def gw_scan_point(alpha_N, xi_w, cm):
    """Solution type and GW spectrum at one point of the scan (None without solution)."""
    try:
        profile = hydrodex.FluidProfile(xi_w=xi_w, alpha_N=alpha_N, cp=C_PLUS_FIG13, cm=cm, step_size=STEP_FIG13)
        spectrum = gw_spectrum(profile, H_R_FIG13, cs=cm, omega=cm**2, grids=GW_GRIDS_FIG13)
    except (ValueError, RuntimeError):
        return None
    return {"solution_type": profile.get_solution_type(), "kR": spectrum["kR"], "Omega_GW": spectrum["Omega_GW"]}


fig13_scan = cached_scan(
    "fig13_gw_scan",
    lambda: {
        (alpha_N, xi_w, cm): gw_scan_point(alpha_N, xi_w, cm)
        for alpha_N in ALPHA_N_VALUES for xi_w in XI_W_VALUES for cm in CM_CASES
    },
)

Plot (`unequal_css_peak_tracking_panel.pdf`). The peak of each spectrum is refined with
a quadratic fit of $\log\Omega_\text{GW}$ in $\log kR_*$ through the five points around
the sampled maximum. Points with $c_-<c_{-,\text{crit}}^{\text{ID}^+}(c_+)$ are not shown.

In [ ]:
def refined_peak(k, spectrum, half_window=2):
    """Peak of a sampled spectrum from a quadratic fit of log(spectrum) in log(k) around
    the sampled maximum (the sampled maximum at the ends of the grid, for a non-concave
    fit or for a vertex outside the fitted points)."""
    k, spectrum = positive_xy(k, spectrum)
    i = int(np.argmax(spectrum))
    if i == 0 or i == len(k) - 1:
        return k[i], spectrum[i]
    x = np.log(k[max(0, i - half_window) : i + half_window + 1])
    y = np.log(spectrum[max(0, i - half_window) : i + half_window + 1])
    a, b, c = np.polyfit(x, y, deg=2)
    x_peak = -b / (2.0 * a)
    if a >= 0.0 or not x.min() <= x_peak <= x.max():
        return k[i], spectrum[i]
    return np.exp(x_peak), np.exp(a * x_peak**2 + b * x_peak + c)


# marker, kind (line style) and legend order of the solution types
MARKERS_FIG13 = {
    ST.Detonation: "o", ST.HybridPlus: "^", ST.Hybrid: "v", ST.Deflagration: "s",
    ST.InverseDetonationPlus: "D", ST.InverseDetonation: "P", ST.InverseHybrid: "X",
    ST.InverseDeflagration: "<", ST.MixedHybridPlus: ">", ST.MixedHybridMinus: "h",
}
INVERSE_LIKE = (ST.InverseDetonationPlus, ST.InverseDetonation, ST.InverseHybrid,
                ST.InverseDeflagration, ST.MixedHybridMinus)

FIG13_RC = {"axes.labelsize": 17, "axes.titlesize": 14, "xtick.labelsize": 14, "ytick.labelsize": 14, "legend.fontsize": 12}

with figure_style("profile", FIG13_RC):
    cminus_crit = pc.inverse_detonation_plus_fold_critical_cm(C_PLUS_FIG13)
    peaks = {key: refined_peak(res["kR"], res["Omega_GW"]) for key, res in fig13_scan.items() if res is not None}
    shown = sorted(key for key in peaks if key[2] >= cminus_crit - 1e-12)
    present = [sol for sol in color_dict if any(fig13_scan[key]["solution_type"] == sol for key in shown)]
    xi_values = sorted({xi_w for _, xi_w, _ in shown})
    xi_norm = Normalize(vmin=min(xi_values), vmax=max(xi_values))
    xi_cmap = plt.colormaps["viridis"]

    fig = plt.figure(figsize=(14.2, 5.7), constrained_layout=True)
    grid_spec = fig.add_gridspec(nrows=1, ncols=3, width_ratios=[1.0, 1.0, 0.035], wspace=0.0)
    ax_position = fig.add_subplot(grid_spec[0, 0])
    ax_amplitude = fig.add_subplot(grid_spec[0, 1], sharex=ax_position)
    cax = fig.add_subplot(grid_spec[0, 2])
    axes = [ax_position, ax_amplitude]

    for ax in axes:
        ax.set_axisbelow(True)
        ax.set_xlim(0.0, 1.0)
        ax.axvspan(0.0, cminus_crit, facecolor="0.88", edgecolor="none", alpha=1.0,
                   antialiased=False, clip_on=True, zorder=0.5)
        ax.axvline(cminus_crit, color="0.25", ls=":", lw=1.55, zorder=3)
        ax.grid(True, which="both", alpha=0.22, zorder=0)

    for alpha_N, xi_w in sorted({key[:2] for key in shown}):
        keys = [key for key in shown if key[:2] == (alpha_N, xi_w)]
        cms = np.array([key[2] for key in keys])
        k_peak, Omega_peak = np.array([peaks[key] for key in keys]).T
        sols = [fig13_scan[key]["solution_type"] for key in keys]
        inverse = np.array([sol in INVERSE_LIKE for sol in sols])
        color = xi_cmap(xi_norm(xi_w))
        # curves: solid for direct-like, dashed for inverse-like solutions
        starts = [0] + [i for i in range(1, len(keys)) if inverse[i] != inverse[i - 1]] + [len(keys)]
        for i0, i1 in zip(starts[:-1], starts[1:]):
            if i1 - i0 >= 2:
                for ax, y in ((ax_position, k_peak), (ax_amplitude, Omega_peak)):
                    ax.plot(cms[i0:i1], y[i0:i1], color=color, ls="--" if inverse[i0] else "-", lw=2.25,
                            solid_capstyle="round", dash_capstyle="round", zorder=5)
        # markers: one shape for each solution type
        for sol in present:
            mask = np.array([s == sol for s in sols])
            if mask.any():
                for ax, y in ((ax_position, k_peak), (ax_amplitude, Omega_peak)):
                    ax.scatter(cms[mask], y[mask], s=43, marker=MARKERS_FIG13[sol], facecolor=color,
                               edgecolor="black", linewidth=0.95, zorder=8)

    for ax in axes:
        ax.tick_params(axis="both", which="major", labelsize=18)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color("black")
            spine.set_linewidth(1.0)
            spine.set_zorder(50)
        ax.set_xlabel(r"$c_-$", fontsize=20)
        ax.set_yscale("log")
    ax_position.set_ylabel(r"$(kR_*)_{\rm peak}$", fontsize=20)
    ax_amplitude.set_ylabel(r"$\Omega_{\rm GW,*}^{\rm peak}$", fontsize=20)
    ax_position.set_title("Peak position", fontsize=18)
    ax_amplitude.set_title("Peak amplitude", fontsize=18)
    ax_position.set_ylim(3.0e-1, 2.0e2)
    ax_amplitude.set_ylim(1.0e-11, 1.0e-3)

    colorbar = fig.colorbar(ScalarMappable(norm=xi_norm, cmap=xi_cmap), cax=cax)
    colorbar.set_label(r"$\xi_w$", fontsize=16, labelpad=9)
    colorbar.ax.tick_params(labelsize=18)
    colorbar.ax.yaxis.set_ticks_position("right")
    colorbar.ax.yaxis.set_label_position("right")
    colorbar.outline.set_edgecolor("black")
    colorbar.outline.set_linewidth(1.0)
    colorbar.set_ticks(xi_values)

    solution_legend = ax_position.legend(
        handles=[
            Line2D([], [], color="black", lw=1.7, ls="--" if sol in INVERSE_LIKE else "-",
                   marker=MARKERS_FIG13[sol], markersize=6.3, markerfacecolor="white",
                   markeredgecolor="black", markeredgewidth=0.95, label=short_label_dict[sol])
            for sol in present
        ],
        loc="upper left", bbox_to_anchor=(0.015, 0.985), borderaxespad=0.0, ncols=1,
        columnspacing=0.9, handlelength=2.4, handletextpad=0.55, fontsize=12,
        frameon=True, framealpha=0.96, facecolor="white", edgecolor="0.55",
    )
    solution_legend.set_zorder(100)
    ax_position.add_artist(solution_legend)
    critical_legend = ax_amplitude.legend(
        handles=[
            Patch(facecolor="0.88", edgecolor="black", linewidth=0.6,
                  label=r"$c_-<c_{-,\mathrm{crit}}^{ID^+}$"),
            Line2D([], [], color="none", linestyle="none", marker=None, label=r"$c_+=1/\sqrt{3}$"),
        ],
        loc="lower left", bbox_to_anchor=(0.69, 0.025), borderaxespad=0.0, fontsize=14,
        frameon=True, framealpha=0.96, facecolor="white", edgecolor="0.55",
    )
    critical_legend.set_zorder(100)

    if save_figures:
        fig.savefig(
            save_prefix + "unequal_css_peak_tracking_panel.pdf",
            format="pdf", bbox_inches="tight", pad_inches=0.03, facecolor="white",
        )

    plt.show()

## Figure 14

Mixed-hybrid spectra and comparison with neighbouring ordinary branches for $c_+=0.75$,
$c_-=0.40$, and $\mathcal{H}_*R_*=0.05$. The upper panels show scans along the two mixed-hybrid
branches at fixed wall velocity. The lower rows compare the branches to direct and inverse
hybrids. From left to right, the velocity profiles, normalized kinetic spectra, and
gravitational-wave spectra are shown. Despite the oppositely directed fluid motion on the two
sides of the wall, the mixed-hybrid kinetic and gravitational-wave spectra closely resemble
ordinary hybrids with different velocity profiles. Changing the background from $\omega=c_-^2$ to
$\omega=1/3$ at fixed $c_s$ and source duration has a comparatively modest effect, whereas
changing the acoustic lifetime significantly modifies the amplitude and removes the low-$k$
shoulder present for $\Delta\tau_\mathrm{nl}$.

Gravitational-wave spectra along the two mixed-hybrid branches, at $\xi_w=0.45$
($\text{MH}^-$) and $\xi_w=0.65$ ($\text{MH}^+$), and at four representative points
(MH$^-$, MH$^+$ and the direct and inverse hybrids DH and IH) for the comparison of the
lower rows: velocity profile, kinetic spectrum and gravitational-wave spectra for the
baseline ($\omega=c_-^2$, $\Delta\tau=\Delta\tau_\text{nl}$), a radiation background
($\omega=1/3$, same $\Delta\tau$) and a fixed duration ($\omega=c_-^2$,
$\mathcal{H}_*\Delta\tau=0.1$). All for $(c_+,c_-)=(0.75,0.40)$,
$\mathcal{H}_*R_*=0.05$ and $c_s=c_-$, with the step size $10^{-5}$ for the profiles.

In [ ]:
C_PLUS_FIG14, C_MINUS_FIG14 = 0.75, 0.40
H_R_FIG14 = 0.05

# scans along the mixed-hybrid branches: wall velocity, values of alpha_N, colours
MH_SCANS = {
    "MH-": (0.45, [-0.001, -0.005, -0.030, -0.050], "Greens"),
    "MH+": (0.65, [+0.001, +0.020, +0.050, +0.080], "Purples"),
}
GW_GRIDS_MH_SCAN = dict(
    chi=np.logspace(-1.0, 4.7, 1000),
    qR=np.logspace(-3.0, 5.2, 950),
    kR=np.logspace(-3.0, 2.0, 180),
    n_qR=900,
    n_p=260,
    n_z=120,
)

# representative points (xi_w, alpha_N, colour) and the grids used for them
POINTS_FIG14 = {
    "MH-": (0.45, -0.050, "#006d2c"),
    "MH+": (0.72, +0.065, "#54278f"),
    "DH": (0.50, +0.080, "#c0392b"),
    "IH": (0.74, -0.060, "#1f4e79"),
}
GW_GRIDS_MH_POINTS = dict(
    chi=np.logspace(-1, 4.5, 10000),
    qR=np.logspace(-3.0, 4.0, 100),
    kR=np.logspace(-4.0, 2.0, 180),
    n_qR=500,
    n_p=120,
    n_z=64,
)
GW_GRIDS_HYBRID_POINTS = dict(
    chi=np.logspace(-1.0, 4.0, 10050),
    qR=np.logspace(-2.5, 4.0, 100),
    kR=np.logspace(-4.0, 2.0, 80),
    n_qR=700,
    n_p=120,
    n_z=64,
)


def fig14_profile(xi_w, alpha_N):
    return hydrodex.FluidProfile(xi_w=xi_w, alpha_N=alpha_N, cp=C_PLUS_FIG14, cm=C_MINUS_FIG14, step_size=1.0e-5)


def mixed_hybrid_scans():
    return {
        name: [
            gw_spectrum(fig14_profile(xi_w, alpha_N), H_R_FIG14, C_MINUS_FIG14, C_MINUS_FIG14**2, GW_GRIDS_MH_SCAN)
            for alpha_N in alpha_N_values
        ]
        for name, (xi_w, alpha_N_values, _) in MH_SCANS.items()
    }


def representative_point(name):
    """Profile and GW spectra of the three cases at one representative point."""
    xi_w, alpha_N, _ = POINTS_FIG14[name]
    grids = GW_GRIDS_MH_POINTS if name.startswith("MH") else GW_GRIDS_HYBRID_POINTS
    profile = fig14_profile(xi_w, alpha_N)
    baseline = gw_spectrum(profile, H_R_FIG14, C_MINUS_FIG14, C_MINUS_FIG14**2, grids)
    return {
        "xi": np.asarray(profile.get_xi_coordinate()),
        "v": np.asarray(profile.get_fluid_velocity()),
        "baseline": baseline,
        "radiation": gw_spectrum(profile, H_R_FIG14, C_MINUS_FIG14, 1.0 / 3.0, grids, baseline["H_delta_tau"]),
        "fixed_duration": gw_spectrum(profile, H_R_FIG14, C_MINUS_FIG14, C_MINUS_FIG14**2, grids, 0.10),
    }


fig14_scans = cached_scan("fig14_mixed_hybrid_scans", mixed_hybrid_scans)
fig14_points = cached_scan(
    "fig14_representative_points", lambda: {name: representative_point(name) for name in POINTS_FIG14}
)

Plot (`combined_mixed_hybrid_diagnostics.pdf`).

In [ ]:
BRANCH_LABELS = {"MH-": r"${\rm MH}^{-}$", "MH+": r"${\rm MH}^{+}$", "DH": r"${\rm DH}$", "IH": r"${\rm IH}$"}
FRAME_KW = dict(frameon=True, framealpha=0.92, facecolor="white", edgecolor="0.45")


def format_axis(ax, log_grid):
    ax.tick_params(axis="both", which="major", labelsize=16)
    ax.grid(True, which="both" if log_grid else "major", alpha=0.23)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(0.90)


def plot_comparison_row(axes_row, names, profile_legend_fs, info_xy, info_fs):
    """Velocity profiles, kinetic spectra and GW spectra of two representative points."""
    ax_profile, ax_kinetic, ax_gw = axes_row
    pair_title = BRANCH_LABELS[names[0]] + " vs " + BRANCH_LABELS[names[1]]

    for name in names:
        xi_w, _, color = POINTS_FIG14[name]
        ax_profile.plot(fig14_points[name]["xi"], fig14_points[name]["v"], color=color, lw=2.2,
                        solid_capstyle="round", label=BRANCH_LABELS[name], zorder=4)
        ax_profile.axvline(xi_w, color=color, ls=":", lw=1.1, alpha=0.70, zorder=2)
    ax_profile.axhline(0.0, color="0.45", ls="--", lw=0.9, alpha=0.75, zorder=1)
    ax_profile.set_xlim(0.39, 0.76)
    ax_profile.set_ylim(-0.20, 0.20)
    ax_profile.set_xlabel(r"$\xi$", fontsize=18)
    ax_profile.set_ylabel(r"$v(\xi)$", fontsize=18)
    ax_profile.set_title(pair_title + ": velocity profiles", fontsize=18)
    ax_profile.legend(loc="upper right", fontsize=profile_legend_fs, **FRAME_KW)
    info_text = "\n\n".join(
        BRANCH_LABELS[name] + rf"$:\ \xi_w={POINTS_FIG14[name][0]:.2f},\ \alpha_N={POINTS_FIG14[name][1]:+.3f}$"
        for name in names
    )
    x, y, ha = info_xy
    ax_profile.text(
        x, y, info_text, transform=ax_profile.transAxes, ha=ha, va="bottom", fontsize=info_fs,
        linespacing=1.12, zorder=20,
        bbox={"boxstyle": "round,pad=0.30", "facecolor": "white", "edgecolor": "0.50", "linewidth": 0.75, "alpha": 0.88},
    )
    format_axis(ax_profile, log_grid=False)

    for name in names:
        baseline = fig14_points[name]["baseline"]
        ax_kinetic.loglog(*positive_xy(baseline["qR"], baseline["kinetic_normalized"]),
                          color=POINTS_FIG14[name][2], lw=2.2, solid_capstyle="round", label=BRANCH_LABELS[name])
    ax_kinetic.set_xlim(1.0e-3, 1.0e4)
    ax_kinetic.set_ylim(1.0e-7, 1.0e1)
    ax_kinetic.set_xlabel(r"$qR_*$", fontsize=18)
    ax_kinetic.set_ylabel(r"$E_{\rm kin}/R_*$", fontsize=18)
    ax_kinetic.set_title(pair_title + ": kinetic spectra", fontsize=18)
    ax_kinetic.legend(loc="best", fontsize=10, **FRAME_KW)
    format_axis(ax_kinetic, log_grid=True)

    # colour = point, line style = expansion / lifetime assumption
    gw_cases = [
        ("baseline", "-", r"$\omega=c_-^2,\ \Delta\tau_{\rm nl}$"),
        ("radiation", "--", r"$\omega=1/3,\ \Delta\tau_{\rm nl}$"),
        ("fixed_duration", ":", r"$\omega=c_-^2,\ {\rm fixed}\ \Delta\tau$"),
    ]
    for name in names:
        color = POINTS_FIG14[name][2]
        for case, ls, _ in gw_cases:
            spectrum = fig14_points[name][case]
            ax_gw.loglog(*positive_xy(spectrum["kR"], spectrum["Omega_GW"]), color=color, ls=ls, lw=2.0,
                         solid_capstyle="round", dash_capstyle="round")
            ax_gw.plot(*sampled_peak(spectrum["kR"], spectrum["Omega_GW"]), marker="o", ms=4.8, color=color,
                       markeredgecolor="black", markeredgewidth=0.35, zorder=6)
    ax_gw.set_xlim(1.0e-4, 1.0e2)
    ax_gw.set_ylim(1.0e-15, 1.0e-7)
    ax_gw.set_xlabel(r"$kR_*$", fontsize=18)
    ax_gw.set_ylabel(r"$\Omega_{\rm GW,*}/\bar{\Omega}_*^2$", fontsize=18, labelpad=5)
    ax_gw.set_title(pair_title + ": expansion/lifetime", fontsize=18)
    ax_gw.add_artist(ax_gw.legend(
        handles=[Line2D([], [], color=POINTS_FIG14[name][2], lw=2.3, label=BRANCH_LABELS[name]) for name in names],
        loc="upper right", fontsize=10, **FRAME_KW,
    ))
    ax_gw.legend(
        handles=[Line2D([], [], color="black", lw=2.0, ls=ls, label=label) for _, ls, label in gw_cases],
        loc="upper left", fontsize=12, title="Universe expansion", title_fontsize=10, **FRAME_KW,
    )
    format_axis(ax_gw, log_grid=True)


FIG14_RC = {"axes.labelsize": 18, "axes.titlesize": 16, "xtick.labelsize": 14, "ytick.labelsize": 14, "legend.fontsize": 11}

with figure_style("profile", FIG14_RC):
    fig = plt.figure(figsize=(18.0, 14.5), facecolor="white", layout="constrained")
    grid = fig.add_gridspec(nrows=3, ncols=6, height_ratios=[0.92, 1.00, 1.00], wspace=0.84, hspace=0.30)
    fig.set_constrained_layout_pads(w_pad=0.04, h_pad=0.04, wspace=0.075, hspace=0.04)
    ax_top_minus = fig.add_subplot(grid[0, 0:3])
    ax_top_plus = fig.add_subplot(grid[0, 3:6], sharex=ax_top_minus, sharey=ax_top_minus)
    axes_middle = [fig.add_subplot(grid[1, 2 * i : 2 * i + 2]) for i in range(3)]
    axes_bottom = [fig.add_subplot(grid[2, 2 * i : 2 * i + 2]) for i in range(3)]

    # first row: scans along the two mixed-hybrid branches
    for ax, name in ((ax_top_minus, "MH-"), (ax_top_plus, "MH+")):
        xi_w, alpha_N_values, cmap = MH_SCANS[name]
        colors = plt.colormaps[cmap](np.linspace(0.25, 0.90, len(alpha_N_values)))
        for spectrum, alpha_N, color in zip(fig14_scans[name], alpha_N_values, colors):
            ax.loglog(*positive_xy(spectrum["kR"], spectrum["Omega_GW"]), color=color, lw=2.1,
                      solid_capstyle="round", label=rf"$\alpha_N={alpha_N:+.3f}$")
            ax.plot(*sampled_peak(spectrum["kR"], spectrum["Omega_GW"]), marker="o", ms=4.8, color=color,
                    markeredgecolor="black", markeredgewidth=0.35, zorder=6)
        ax.set_xlim(1.0e-3, 1.0e2)
        ax.set_ylim(1.0e-15, 1.0e-7)
        ax.set_xlabel(r"$kR_*$", fontsize=18)
        ax.set_title(BRANCH_LABELS[name] + rf"$:\ \xi_w={xi_w:.2f}$", fontsize=18)
        ax.legend(loc="lower right", fontsize=12, **FRAME_KW)
        format_axis(ax, log_grid=True)
    ax_top_minus.set_ylabel(r"$\Omega_{\rm GW,*}/\bar{\Omega}_*^2$", fontsize=18)
    ax_top_minus.text(
        0.025, 0.965,
        rf"$c_+={C_PLUS_FIG14:.2f},\quad c_-={C_MINUS_FIG14:.2f}$" "\n"
        rf"$\mathcal{{H}}_*R_*={H_R_FIG14:.3f}$" "\n"
        r"$\Delta\tau_{\rm fin}=R_*/\sqrt{\Omega_K}$",
        transform=ax_top_minus.transAxes, ha="left", va="top", fontsize=17, linespacing=1.17, zorder=20,
        bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": "0.45", "linewidth": 0.8, "alpha": 0.90},
    )

    plot_comparison_row(axes_middle, ("MH-", "DH"), profile_legend_fs=13, info_xy=(0.975, 0.04, "right"), info_fs=13.5)
    plot_comparison_row(axes_bottom, ("MH+", "IH"), profile_legend_fs=13, info_xy=(0.03, 0.04, "left"), info_fs=12.5)

    # constrained layout gives up on this figure ("axes sizes collapsed to zero"), so the
    # figure is drawn with the positions of the grid
    with quiet("constrained_layout not applied"):
        if save_figures:
            fig.savefig(
                save_prefix + "combined_mixed_hybrid_diagnostics.pdf",
                format="pdf", bbox_inches="tight", pad_inches=0.04, facecolor="white",
            )
        plt.show()

## Figure 15

Representative kinematic exotic CSS profiles for $c_+=0.75$ and $c_-=0.40$. The red dashed line
marks the wall position $\xi_w$, the grey dotted lines mark $c_+$ and $c_-$, and the black dotted
lines mark the shock front positions. The boxes report the corresponding values of $\xi_w$,
$\alpha_+$, and $\alpha_N$. All configurations satisfy the local wall matching, but they do not
define physical hydrodynamic branches. In each case at least one shock front is traversed as a
rarefaction shock and therefore fails both the evolutionarity and entropy conditions.

The exotic candidates are constructed without `FluidProfile`, since they are not
hydrodynamic branches: from the wall-frame velocities $v_\pm$ at the wall, the CSS
self-similar equations for $\xi(v)$ and $\ln w(v)$ are integrated away from the wall on
each side, either to a shock front ($\xi\mu(\xi,v)=c_s^2$, connected to a fluid at rest
by the algebraic shock matching) or, for a rarefaction, to $v\to0$. The strength
$\alpha_N$ in the panels follows from $\alpha_N=\alpha_0+r_w(\alpha_+-\alpha_0)$ with
$r_w=w_+/w_N$ [eq. (3.11) of the paper] (`pc.rescale_alpha_by_enthalpy`).

In [ ]:
EXOTIC_RC = {
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.18,
    "grid.color": "0.7",
    "axes.edgecolor": "black",
    "axes.linewidth": 0.9,
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "text.usetex": True,
}
C_PLUS_FIG15, C_MINUS_FIG15 = 0.75, 0.40
EPS = 1e-6  # distance of the integration end points from |v| = 0 and |v| = 1

# title, colour (of the corresponding branch), xi_w, v_+, v_- and the kind of solution
# ahead of ("rest", "fan" or "front_shock") and behind ("rest", "fan" or "back_shock")
# the wall; in each case at least one shock is a rarefaction shock
EXOTIC_CASES = [
    ("direct detonation + trailing shock", ST.Detonation, 0.65, 0.65, 0.30, "rest", "back_shock"),
    ("inverse detonation + preceding shock", ST.InverseDetonationPlus, 0.50, 0.85, 0.50, "front_shock", "rest"),
    ("direct hybrid + two shocks", ST.HybridPlus, 0.65, 0.58, 0.30, "front_shock", "back_shock"),
    ("inverse hybrid + two shocks", ST.InverseHybrid, 0.50, 0.80, 0.90, "front_shock", "back_shock"),
    ("mixed hybrid + front shock", ST.MixedHybridPlus, 0.55, 0.85, 0.50, "front_shock", "fan"),
    ("mixed hybrid + trailing shock", ST.MixedHybridMinus, 0.65, 0.70, 0.30, "fan", "back_shock"),
]


def integrate_in_v(xi0, v0, w0, cs, v1):
    """Self-similar CSS equations with v as integration variable (dense output):
    2v/xi = gamma^2 (1 - xi v) [mu^2/c_s^2 - 1] dv/dxi,  dln w/dv = (1 + 1/c_s^2) gamma^2 mu."""
    def rhs(v, y):
        xi = y[0]
        g2, m = pc.gamma(v) ** 2, pc.mu(xi, v)
        return [g2 * (1.0 - xi * v) * (m * m / cs**2 - 1.0) * xi / (2.0 * v), pc.get_nu_from_cs(cs) * g2 * m]

    return solve_ivp(rhs, (v0, v1), [xi0, np.log(w0)], rtol=1e-8, atol=1e-10, max_step=2e-3, dense_output=True)


def shock_segment(xi_w, v_wall, w_wall, cs, side, n=700):
    """Profile from the wall to the first shock front on the given side ("front": xi > xi_w,
    "back": xi < xi_w), and the enthalpy of the fluid at rest beyond the shock."""
    if side == "front":
        v_end = EPS if v_wall > 0.0 else -1.0 + EPS
    else:
        v_end = 1.0 - EPS if v_wall > 0.0 else -EPS
    sol = integrate_in_v(xi_w, v_wall, w_wall, cs, v_end)
    # first root of xi mu(xi, v) - c_s^2 on the requested side
    v_grid = np.linspace(sol.t[0], sol.t[-1], 4000)
    xi_grid = sol.sol(v_grid)[0]
    F = xi_grid * pc.mu(xi_grid, v_grid) - cs**2
    ok = xi_grid > xi_w + 1e-5 if side == "front" else xi_grid < xi_w - 1e-5
    i = next(i for i in range(len(v_grid) - 1) if ok[i] and ok[i + 1] and F[i] * F[i + 1] <= 0.0)
    v_sh = brentq(lambda v: sol.sol(v)[0] * pc.mu(sol.sol(v)[0], v) - cs**2, v_grid[i], v_grid[i + 1],
                  xtol=1e-12, rtol=1e-10, maxiter=100)
    v = np.linspace(sol.t[0], v_sh, n)
    xi, ln_w = sol.sol(v)
    w = np.exp(ln_w)
    # enthalpy ratio across the shock (moving / rest) from the shock matching
    u_rest, u_moving = abs(xi[-1]), abs(pc.mu(xi[-1], v[-1]))
    w_rest = w[-1] / ((u_rest / (1.0 - u_rest**2)) / (u_moving / (1.0 - u_moving**2)))
    order = np.argsort(xi)
    return xi[order], v[order], w[order], xi[-1], w_rest


def fan_segment(xi_w, v_wall, w_wall, cs, n=700):
    """Rarefaction from the wall to v -> 0."""
    sol = integrate_in_v(xi_w, v_wall, w_wall, cs, EPS if v_wall > 0.0 else -EPS)
    v = np.linspace(sol.t[0], sol.t[-1], n)
    xi, ln_w = sol.sol(v)
    w = np.exp(ln_w)
    order = np.argsort(xi)
    return xi[order], v[order], w[order], xi[-1], w[-1]


def at_rest(x0, x1, w, n=80):
    """Fluid at rest for x0 < xi < x1."""
    x = np.linspace(x0, x1, n) if x1 > x0 else np.array([])
    return x, np.zeros_like(x), np.full_like(x, w)


def exotic_profile(xi_w, v_plus, v_minus, kind_plus, kind_minus):
    """xi, v, w, alpha_+, alpha_N and shock positions of an exotic candidate (w_+ = 1)."""
    u_plus, u_minus = pc.mu(xi_w, v_plus), pc.mu(xi_w, v_minus)  # bubble-frame velocities
    w_plus = 1.0
    w_minus = pc.gamma(v_plus) ** 2 * v_plus / (pc.gamma(v_minus) ** 2 * v_minus) * w_plus
    shocks = []
    # behind the wall, xi < xi_w
    if kind_minus == "rest":
        minus = at_rest(0.0, xi_w, w_minus)
    elif kind_minus == "fan":
        xi, v, w, xi_end, w_end = fan_segment(xi_w, u_minus, w_minus, C_MINUS_FIG15)
        rest = at_rest(0.0, max(0.0, min(xi[0], xi_end)), w_end)
        minus = [np.r_[r, s] for r, s in zip(rest, (xi, v, w))]
    else:
        xi, v, w, xi_sh, w_rest = shock_segment(xi_w, u_minus, w_minus, C_MINUS_FIG15, "back")
        minus = [np.r_[r, s] for r, s in zip(at_rest(0.0, xi_sh, w_rest), (xi, v, w))]
        shocks.append(xi_sh)
    # ahead of the wall, xi > xi_w
    if kind_plus == "rest":
        plus = at_rest(xi_w, 1.0, w_plus)
    elif kind_plus == "fan":
        xi, v, w, xi_end, w_end = fan_segment(xi_w, u_plus, w_plus, C_PLUS_FIG15)
        rest = at_rest(min(1.0, max(xi[-1], xi_end)), 1.0, w_end)
        plus = [np.r_[s, r] for r, s in zip(rest, (xi, v, w))]
    else:
        xi, v, w, xi_sh, w_rest = shock_segment(xi_w, u_plus, w_plus, C_PLUS_FIG15, "front")
        plus = [np.r_[s, r] for r, s in zip(at_rest(xi_sh, 1.0, w_rest), (xi, v, w))]
        shocks.append(xi_sh)
    # xi_w twice, so that the wall discontinuity is drawn as a vertical line
    xi = np.r_[minus[0], [xi_w, xi_w], plus[0]]
    v = np.r_[minus[1], [u_minus, u_plus], plus[1]]
    w = np.r_[minus[2], [w_minus, w_plus], plus[2]]
    inside = (xi >= -1e-8) & (xi <= 1.0 + 1e-8)
    xi, v, w = xi[inside], v[inside], w[inside]
    alpha_plus = pc.compute_alpha_plus(v_plus, v_minus, C_MINUS_FIG15)
    alpha_N = pc.rescale_alpha_by_enthalpy(alpha_plus, w_plus / w[-1], C_PLUS_FIG15, C_MINUS_FIG15)
    return {"xi": xi, "v": v, "alpha_plus": alpha_plus, "alpha_N": alpha_N, "shocks": shocks}


exotic_profiles = [exotic_profile(*case[2:]) for case in EXOTIC_CASES]

Plot (`exotic_profiles_panel.pdf`).

In [ ]:
def padded_interval(lo, hi, factor=1.2):
    """Interval widened away from zero by `factor`."""
    return lo * (factor if lo < 0 else 1.0 / factor), hi * (factor if hi > 0 else 1.0 / factor)


with figure_style("profile", EXOTIC_RC):
    # corner of the parameter box and manual x-limits of some panels
    INFO_CORNERS = ["upper left", "lower right", "upper right", "lower right", "lower right", "upper right"]
    CORNERS = {"upper left": (0.03, 0.97, "left", "top"), "upper right": (0.97, 0.97, "right", "top"),
               "lower right": (0.97, 0.08, "right", "bottom")}
    PANEL_XLIM = {0: (0.6, 0.7), 2: (0.55, 0.85), 5: (0.55, 0.85)}

    fig, axes = plt.subplots(2, 3, figsize=(15.0, 8.4), sharex=False, sharey=False)
    axes = axes.ravel()

    for i, (ax, case, profile, corner) in enumerate(zip(axes, EXOTIC_CASES, exotic_profiles, INFO_CORNERS)):
        title, sol, xi_w = case[:3]
        ax.plot(profile["xi"], profile["v"], color=color_dict[sol], lw=2.4, zorder=3)
        ax.axvline(xi_w, color="red", ls="--", lw=1.5, zorder=2)
        ax.axvline(C_PLUS_FIG15, color="0.45", ls=":", lw=1.2, zorder=1)
        ax.axvline(C_MINUS_FIG15, color="0.65", ls=":", lw=1.2, zorder=1)
        for xi_sh in profile["shocks"]:
            ax.axvline(xi_sh, color="black", ls=":", lw=1.4, zorder=2)

        # limits: the relevant xi positions and the velocity range (at least +-0.2), padded
        xi_marks = [C_PLUS_FIG15, C_MINUS_FIG15, xi_w] + profile["shocks"]
        xlim = padded_interval(min(xi_marks), max(xi_marks))
        ylim = padded_interval(min(profile["v"].min(), -0.2), max(profile["v"].max(), 0.2))
        ax.set_xlim(*PANEL_XLIM.get(i, (max(0.0, xlim[0]), min(1.0, xlim[1]))))
        ax.set_ylim(max(-1.0, ylim[0]), min(1.0, ylim[1]))

        x, y, ha, va = CORNERS[corner]
        ax.text(
            x, y,
            rf"$\xi_w = {xi_w:.2f}$" "\n" rf"$\alpha_+ = {profile['alpha_plus']:+.2f}$" "\n"
            rf"$\alpha_N = {profile['alpha_N']:+.2f}$",
            transform=ax.transAxes, ha=ha, va=va, fontsize=16, zorder=6, clip_on=False,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.45", alpha=0.92),
        )
        ax.set_title(title, fontsize=18, pad=10)
        ax.tick_params(axis="both", which="major", labelsize=18)
        if i % 3 == 0:
            ax.set_ylabel(r"$v(\xi)$", fontsize=18)
        if i // 3 == 1:
            ax.set_xlabel(r"$\xi$", fontsize=18)

    axes[0].legend(
        handles=[
            Line2D([], [], color="red", ls="--", lw=1.5, label=r"$\xi_w$"),
            Line2D([], [], color="0.45", ls=":", lw=1.2, label=r"$c_+$"),
            Line2D([], [], color="0.65", ls=":", lw=1.2, label=r"$c_-$"),
            Line2D([], [], color="black", ls=":", lw=1.4, label=r"$\xi_{\rm sh}$"),
        ],
        loc="upper right", bbox_to_anchor=(1.0, 1.0), bbox_transform=axes[0].transAxes,
        borderaxespad=0.4, frameon=False, fontsize=16,
    )
    fig.tight_layout(pad=0.8)

    if save_figures:
        fig.savefig(save_prefix + "exotic_profiles_panel.pdf", format="pdf", bbox_inches="tight", facecolor="white")

    plt.show()

## Figure 16

Numerical test of the monotonicity of $\lambda(c_+,\xi_w)$. We plot
$d\lambda/dx-\lambda(1-\lambda)/x$, with $x=\xi_w/c_+$, for representative values of $c_+$. The
quantity remains positive throughout $0<x<1$, implying $\partial_{\xi_w}\lambda>0$.

We compute $\lambda(c_+,\xi_w)$ on the sonic family $v_+=c_+$ from
$d\ln r_w/d\xi_w$, integrating the rarefaction together with
$d\xi(v)/d\xi_w$, and evaluate $d\lambda/dx-\lambda(1-\lambda)/x$ with
$x=\xi_w/c_+$.

In [ ]:
C_PLUS_LAMBDA = [0.2, 0.3, 0.40, 0.50, 0.60, 0.70, 0.75, 0.80, 0.90, 0.95, 0.99, 0.999, 0.9999, 0.99999]
X_LAMBDA = np.linspace(5.0e-5, 1.0 - 5.0e-5, 501)  # x = xi_w / c_+


def lambda_sonic(cp, xi_w):
    """lambda(c_+, xi_w) on the sonic family v_+ = c_+.

    Along the inverse rarefaction ahead of the wall (v = -exp(t)) we integrate xi(v),
    J = d xi(v) / d xi_w (J = 1 at the sonic wall) and I = int J / (1 - xi v)^2 dv up to
    v -> 0; then lambda = 2 xi_w (1 - xi_w^2) / (c_+ (c_+ - xi_w)) I.
    """
    def rhs(t, y):
        xi, J, _ = y
        v = -np.exp(t)
        g2, m = pc.gamma(v) ** 2, pc.mu(xi, v)
        sonic = m**2 / cp**2 - 1.0
        f = xi * g2 * (1.0 - xi * v) / (2.0 * v) * sonic  # d xi / d v
        df_dxi = g2 / (2.0 * v) * (
            (1.0 - 2.0 * xi * v) * sonic + xi * (1.0 - xi * v) * (2.0 * m / cp**2) * (1.0 - v**2) / (1.0 - xi * v) ** 2
        )
        return (v * f, v * df_dxi * J, v * J / (1.0 - xi * v) ** 2)

    sol = solve_ivp(
        rhs, (np.log(-pc.mu(xi_w, cp)), np.log(1.0e-13)), (xi_w, 1.0, 0.0),
        method="DOP853", rtol=2.0e-11, atol=2.0e-13,
    )
    return 2.0 * xi_w * (1.0 - xi_w**2) / (cp * (cp - xi_w)) * sol.y[2, -1]


def strong_monotonicity_gap(cp):
    """d lambda / dx - lambda (1 - lambda) / x along X_LAMBDA."""
    lam = np.array([lambda_sonic(cp, cp * x) for x in X_LAMBDA])
    return np.gradient(lam, X_LAMBDA, edge_order=2) - lam * (1.0 - lam) / X_LAMBDA


fig16_gap = cached_scan("fig16_lambda", lambda: {cp: strong_monotonicity_gap(cp) for cp in C_PLUS_LAMBDA})

Plot (`strong_monotonicity_test.pdf`).

In [ ]:
with figure_style("profile"):
    fig, ax = plt.subplots(figsize=(7.0, 5.0))
    for cp, gap in fig16_gap.items():
        ax.plot(X_LAMBDA, gap, lw=1.8, label=rf"$c_+={cp:g}$")
    ax.axhline(0.0, ls="--", c="k", lw=1.0)
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 0.4)
    ax.set_xlabel(r"$x=\xi_w/c_+$")
    ax.set_ylabel(r"$\displaystyle \frac{d\lambda}{dx}-\frac{\lambda(1-\lambda)}{x}$")
    ax.legend(frameon=False, fontsize=14, ncol=2)
    ax.grid(True, alpha=0.22)
    plt.tight_layout()

    if save_figures:
        fig.savefig(save_prefix + "strong_monotonicity_test.pdf", bbox_inches="tight", facecolor="white")

    plt.show()

## Figure 17

Projection of the $v_+\to c_+$ sonic endpoint,
$\alpha_N\big(\alpha_{\bar\theta}(c_+,\xi_w),\,r_w^<(c_+)\big)$, for $c_+=0.9$ at the two
thresholds $c_-=c_{-,0}^{\mathrm{ID}^+}\simeq0.192$ (left) and
$c_-=c_{-,0}^{\mathrm{IH}}\simeq0.193$ (right). This is the upper $\alpha_N$ bound of inverse
hybrids for all $\xi_w$, cf. eq. (4.23), and for $\xi_w<c_-$ (red) also the upper bound
$\alpha_{N,\max}^{\mathrm{ID}^+}$ of the inverse-detonation '(+)' branch wherever it is positive,
cf. eq. (4.17). Since $c_->\xi_r$, the minimum of $r_w^<(c_+)$, the projection is still increasing
at $\xi_w=c_-$. It reaches $\alpha_N=0$ at the upper end of the $\xi_w$ range of inverse
detonations, $\xi_w\to c_-$ (left), while its maximum, which sets $c_{-,0}^{\mathrm{IH}}$, lies at
$\xi_w^{\max}>c_-$, outside the $\xi_w$ range of inverse detonations but within the inverse-hybrid
range (right).

The projection is computed directly, without `FluidProfile`: $r_w^<(c_+;\xi_w)$ is
obtained from the inverse rarefaction ahead of the wall with $v_+=c_+$
(`ln_rw_inverse_rarefaction`; it does not depend on $c_-$) and tabulated once. The two
thresholds are computed in the cell: $c_{-,0}^{\mathrm{ID}^+}$, where the bound vanishes
at $\xi_w=c_-$, and $c_{-,0}^{\mathrm{IH}}$, where its maximum vanishes
(`zero_crossing_bound_maximum.pdf`).

In [ ]:
C_PLUS_FIG17 = 0.9

# r_w^< does not depend on c_-: tabulate ln r_w^< once and interpolate it
xi_tab = 1e-4 * C_PLUS_FIG17 + (C_PLUS_FIG17 - 1e-4 * C_PLUS_FIG17) * (
    0.5 - 0.5 * np.cos(np.linspace(0.0, np.pi, 1200))
)[:-1]
ln_rw_sonic = CubicSpline(xi_tab, [ln_rw_inverse_rarefaction(x, C_PLUS_FIG17, C_PLUS_FIG17) for x in xi_tab])


def alpha_N_bound(xi_w, cm):
    """Upper alpha_N bound along the sonic endpoint v_+ = c_+, v_- = xi_w."""
    return np.vectorize(pc.rescale_alpha_by_enthalpy)(
        pc.compute_alpha_plus(C_PLUS_FIG17, xi_w, cm), np.exp(ln_rw_sonic(xi_w)), C_PLUS_FIG17, cm
    )


def interior_max(cm):
    """Maximum of the bound over c_-^2/c_+ < xi_w < (c_- + c_+)/2 (without the trivial end
    xi_w -> c_+)."""
    xs = np.linspace(cm**2 / C_PLUS_FIG17, 0.5 * (cm + C_PLUS_FIG17), 4001)[1:]
    i = int(np.argmax(alpha_N_bound(xs, cm)))
    res = minimize_scalar(
        lambda x: -alpha_N_bound(x, cm), bounds=(xs[i - 1], xs[i + 1]), method="bounded",
        options={"xatol": 1e-12},
    )
    return res.x, -res.fun


xi_r = minimize_scalar(ln_rw_sonic, bounds=(1e-3, 0.99 * C_PLUS_FIG17), method="bounded", options={"xatol": 1e-10}).x
# c_{-,0}^{ID+} (bound at xi_w = c_- vanishes) and c_{-,0}^{IH} (maximum vanishes)
CM_ZERO_IDPLUS = brentq(lambda cm: alpha_N_bound(cm, cm), 0.17, 0.21, xtol=1e-12)
CM_ZERO_IH = brentq(lambda cm: interior_max(cm)[1], 0.17, 0.21, xtol=1e-12)

with figure_style("profile", {"text.latex.preamble": r"\usepackage{amsmath}"}):
    XLIM_FIG17, YLIM_FIG17 = (0.09, 0.3), (-0.2, 0.05)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))

    for ax, cm in zip(axes, (CM_ZERO_IDPLUS, CM_ZERO_IH)):
        xs = np.linspace(cm**2, C_PLUS_FIG17, 3000)[1:-1]
        y = alpha_N_bound(xs, cm)
        shared = xs <= cm  # also the bound of the inverse detonations
        xm, ym = interior_max(cm)
        ax.plot(xs[shared], y[shared], color="tab:red", lw=2)
        ax.plot(xs[~shared], y[~shared], color="tab:blue", lw=2)
        ax.axhline(0.0, color="k", lw=0.6)
        ax.axvline(cm, color="k", ls="--", lw=0.8)
        ax.axvline(xi_r, color="0.5", ls=":", lw=0.8)
        ax.plot(xm, ym, "o", color="tab:red" if xm <= cm else "tab:blue", ms=5, zorder=5)
        ax.set_xlabel(r"$\xi_w$")
        ticks = {0.1: "0.1", cm: "$c_-$", xi_r: r"$\xi_r$", xm: r"$\xi_w^{\max}$", 0.3: "0.3"}
        ticks = {x: label for x, label in ticks.items() if XLIM_FIG17[0] - 1e-9 <= x <= XLIM_FIG17[1] + 1e-9}
        ax.set_xticks(list(ticks))
        ax.set_xticklabels(list(ticks.values()))
        ax.set_xlim(*XLIM_FIG17)
        ax.set_ylim(*YLIM_FIG17)

    axes[0].set_ylabel(r"$\alpha_N$")
    axes[0].text(0.97, 0.04, rf"$c_+={C_PLUS_FIG17},\ c_-={CM_ZERO_IDPLUS:.3f}$",
                 transform=axes[0].transAxes, ha="right", va="bottom")
    axes[1].text(0.97, 0.96, rf"$c_+={C_PLUS_FIG17},\ c_-={CM_ZERO_IH:.3f}$",
                 transform=axes[1].transAxes, ha="right", va="top")
    axes[0].legend(
        handles=[
            Line2D([0], [0], color="tab:red", lw=2, marker="o", ms=5, label=r"$\alpha_N^{\mathrm{ID^+},\,\max}$"),
            Line2D([0], [0], color="tab:blue", lw=2, marker="o", ms=5, label=r"$\alpha_N^{\mathrm{IH},\,\max}$"),
        ],
        loc="upper left", frameon=True, fancybox=True, facecolor="white", edgecolor="black",
        framealpha=1.0, fontsize=mpl.rcParams["legend.fontsize"],
    )
    fig.tight_layout()

    if save_figures:
        fig.savefig(save_prefix + "zero_crossing_bound_maximum.pdf", bbox_inches="tight")

    plt.show()